# BUDT758T Final Project Yelp Review Usefulness Prediction


This notebook predicts whether a Yelp review is **highly useful** using review text/metadata, business information, user information, tip aggregates, and holiday/date features. The primary submitted model is **Model 1: Final Blended Model**, an ensemble built from LightGBM, XGBoost, and MLP variants.

**Important reproducibility note:** Model 1 was intentionally preserved as closely as possible. The cleanup focuses on notebook structure, path handling, comments, sanity checks, and comparison-model reliability. No retuning or redesign was performed on the final blended model.


## Executive Summary

The notebook is organized to make the final submission workflow easy to audit and rerun:

1. **Load data and configure paths** for Kaggle or local execution.
2. **Engineer features** from review, business, user, tip, and holiday data.
3. **Run Model 1**, the preserved final blended model used for submission.
4. **Run comparison models** including logistic regression, decision tree, random forest, and LightGBM.
5. **Export final predictions** and print sanity-check metrics.

The final model uses an FPR-constrained thresholding strategy: thresholds are evaluated using the ROC curve, and the selected threshold is constrained by **FPR ≤ 0.10**. This matches the project objective of maximizing true positive rate while controlling false positives.


## Model Inventory

| Notebook section | Model | Purpose | Notes |
|---|---|---|---|
| Model 1 | Final blended LightGBM/XGBoost/MLP model | Primary submission model | Preserved |
| Model 2 | Logistic Regression with cleaned business dataset | Simple baseline | Uses held-out validation for ROC/AUC reporting. |
| Model 5 | Logistic Regression without holiday features | Ablation baseline | Checks performance without external holiday variables. |
| Model 6 | Logistic Regression with holiday features | Ablation baseline | Tests whether holiday/date features improve logistic regression. |
| Model 3 | Random Forest | Nonlinear tree-based comparison | Uses separate feature engineering from the logistic/blend feature path. |
| Model 7 | LightGBM | Gradient boosting comparison |  Uses the FPR ≤ 0.10 evaluation rule. |

The model numbering follows the original working file where possible.


## How to Run This Notebook

Run the notebook from top to bottom.

For Kaggle, upload the following CSV files as a dataset and either let the notebook auto-detect them under `/kaggle/input` or set `DATA_DIR` manually in the path configuration cell:

- `review_train_x.csv`
- `review_train_y.csv`
- `review_test_x.csv`
- `business.csv` or `business_data_cleaned.csv`
- `user.csv` or `user_clean.csv`
- `tip.csv` or `tips_cleaned.csv`

Outputs are written to `OUTPUT_DIR`. On Kaggle this defaults to `/kaggle/working`; locally it defaults to the data folder.

**Runtime note:** report-only blocks such as full model comparison, cross-validation, learning curves, and tuning curves are controlled by `RUN_REPORT_BLOCKS`. They are disabled by default so the final submission path can run faster and more reliably.


## Source File Scan and Model Section Map

Before editing, the uploaded notebook was scanned and the main sections were identified. This map is included so the grader can understand how the messy working file was reorganized.

- **Model 1 / Final blended model:** first pipeline in the original file. This is the protected final submission block and includes feature engineering, model-zoo scoring, blend search, local refinement, and final submission export.
- **Model 2 / Logistic Regression with business dataset:** baseline logistic regression using cleaned business features.
- **Decision Tree baseline:** retained as an interpretable comparison model.
- **Model 5 / Logistic Regression without holiday features:** ablation model that removes holiday-related predictors.
- **Model 6 / Logistic Regression with holiday features:** logistic regression comparison with holiday/date predictors.
- **Model 3 / Random Forest:** tree ensemble comparison using a separate feature-engineering path.
- **Model 7 / LightGBM:** completed using the standalone LightGBM notebook as the source of truth.

Cleanup actions were limited to organization, comments, path handling, syntax fixes, safer validation reporting for comparison models, and sanity checks. The final blended model logic was not retuned or redesigned.


## 1. Project Setup and Imports

This section imports the packages used across the notebook. The imports are intentionally centralized so later cells can be run in order without missing dependencies.


In [ ]:
# Yelp Top-Useful Prediction
# Super Advanced 85+ Version
#
# Includes:
# 1. External data source: US holiday calendar
# 2. 10+ feature insight tables/charts
# 3. 6+ model comparison
# 4. Cross-validation
# 5. Learning curve
# 6. Tuning curve
# 7. Model zoo: many LGB/XGB/MLP variants
# 8. Advanced top-K random blend
# 9. Final submission under FPR <= 0.097

import os
import gc
import ast
import json
import re
import time
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, accuracy_score

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier

import lightgbm as lgb
import xgboost as xgb
import holidays

## 2. File Paths and Data Loading

This section defines the reproducibility settings, Kaggle-compatible paths, required input filenames, and model-control flags. The `csv_path()` helper only locates files; it does not modify row order or file contents.


In [ ]:

# 1. Settings and Kaggle-compatible file paths

RANDOM_STATE = 42
N_JOBS = -1

# Kaggle note:
# 1. Upload the CSV files as a Kaggle dataset.
# 2. Set DATA_DIR below to that dataset folder if auto-detection does not find it.
#    Example: DATA_DIR = "/kaggle/input/yelp-review"
from pathlib import Path

TRAIN_X_FILE = "review_train_x.csv"
TRAIN_Y_FILE = "review_train_y.csv"
TEST_X_FILE = "review_test_x.csv"

# Model 1 originally referenced business.csv, user.csv, and tip.csv.
# The cleaned Kaggle filenames are listed as fallbacks so the notebook can run on Kaggle.
BUSINESS_FILE = "business.csv"
USER_FILE = "user.csv"
TIP_FILE = "tip.csv"

BUSINESS_CLEAN_FILE = "business_data_cleaned.csv"
USER_CLEAN_FILE = "user_clean.csv"
TIP_CLEAN_FILE = "tips_cleaned.csv"

DATA_DIR = os.environ.get("DATA_DIR", "")
OUTPUT_DIR = os.environ.get("OUTPUT_DIR", "")

def csv_path(filename, fallbacks=()):
    """Return a CSV path without changing row order or file contents."""
    candidates = []
    search_names = [filename] + list(fallbacks)

    if DATA_DIR:
        for name in search_names:
            candidates.append(Path(DATA_DIR) / name)

    for name in search_names:
        candidates.append(Path.cwd() / name)

    kaggle_root = Path("/kaggle/input")
    if kaggle_root.exists():
        for name in search_names:
            candidates.extend(kaggle_root.glob(f"**/{name}"))

    for candidate in candidates:
        if candidate.exists():
            return str(candidate)

    raise FileNotFoundError(
        f"Could not find any of: {search_names}. "
        "Set DATA_DIR to the folder containing the project CSV files."
    )

# If DATA_DIR was not supplied, infer it from review_train_x.csv.
if not DATA_DIR:
    DATA_DIR = str(Path(csv_path(TRAIN_X_FILE)).parent)

if not OUTPUT_DIR:
    OUTPUT_DIR = "/kaggle/working" if Path("/kaggle/working").exists() else DATA_DIR

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

FPR_LIMIT = 0.10
FPR_LIMIT_FINAL = 0.097

# The result from last submission
CURRENT_BEST_TPR = 0.7356718721389521
CURRENT_BEST_FPR = 0.09698189570279951

# change user.useful to False because it is leakage
USE_USER_USEFUL_FEATURES = True

# To run the blocks required for the report: 6 models / CV / learning curve / tuning curve
RUN_REPORT_BLOCKS = False

# Optional model-zoo switches. These are intentionally False by default to keep
# the notebook runnable on Kaggle unless you explicitly enable the extra models.
RUN_EXTRA_LGB = False
RUN_EXTRA_XGB = False
RUN_EXTRA_MLP = False

# Defaults used by the advanced blend-search call. These match the function
# defaults in Model 1 and only prevent NameError; they do not change the
# underlying Model 1 blending logic.
TOP_K_FOR_BLEND = 12
BLEND_SEARCH_SAMPLE_SIZE = 200000
RANDOM_BLEND_TRIALS = 30000
RANDOM_BLEND_TOP_EVAL = 300

# Runtime caps for report-only cells
MODEL_COMPARE_SAMPLE_SIZE = 120000
CURVE_SAMPLE_SIZE = 150000
MLP_SEEDS = [42, 2024, 777]
USE_XGB_GPU = False

print("DATA_DIR:", DATA_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)


In [ ]:

# 2. Load Data
# Direct pd.read_csv calls are used; csv_path only resolves Kaggle/local paths.

train_x = pd.read_csv(csv_path(TRAIN_X_FILE))
train_y = pd.read_csv(csv_path(TRAIN_Y_FILE))
test_x = pd.read_csv(csv_path(TEST_X_FILE))

# Preserve the original Model 1 preferred filenames, with cleaned-file fallbacks for Kaggle.
business = pd.read_csv(csv_path(BUSINESS_FILE, [BUSINESS_CLEAN_FILE]))
user = pd.read_csv(csv_path(USER_FILE, [USER_CLEAN_FILE]))
tip = pd.read_csv(csv_path(TIP_FILE, [TIP_CLEAN_FILE]))

if train_y.shape[1] == 1:
    y = train_y.iloc[:, 0].astype(int).reset_index(drop=True)
else:
    y = train_y["top_useful"].astype(int).reset_index(drop=True)

# Find distribution and shapes
print("train_x:", train_x.shape)
print("train_y:", train_y.shape)
print("test_x:", test_x.shape)
print("business:", business.shape)
print("user:", user.shape)
print("tip:", tip.shape)

print("\nLabel distribution:")
print(y.value_counts())
print("Positive rate:", y.mean())

assert len(train_x) == len(y), "train_x and train_y row counts do not match."


## 10. Evaluation Utilities

These utility functions calculate ROC-based metrics and select thresholds under the project constraint. They are defined early because Model 1 uses them before the later comparison-model sections.


In [ ]:
# ============================================================
# 3. Evaluation Functions
# ============================================================

# Find the best TPR under a specific FPR limit
def best_tpr_under_fpr(y_true, y_score, fpr_limit=0.10):
    fpr, tpr, thresholds = roc_curve(y_true, y_score)
    valid_idx = np.where(fpr <= fpr_limit)[0]

    if len(valid_idx) == 0:
        return {
            "auc": roc_auc_score(y_true, y_score),
            "fpr": np.nan,
            "tpr": np.nan,
            "threshold": np.nan,
            "accuracy": np.nan,
            "tn": np.nan,
            "fp": np.nan,
            "fn": np.nan,
            "tp": np.nan
        }

    best_idx = valid_idx[np.argmax(tpr[valid_idx])]
    best_threshold = thresholds[best_idx]

    y_pred = (y_score >= best_threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    return {
        "auc": roc_auc_score(y_true, y_score),
        "fpr": fp / (fp + tn),
        "tpr": tp / (tp + fn),
        "threshold": best_threshold,
        "accuracy": accuracy_score(y_true, y_pred),
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp
    }


def print_metrics(name, metrics):
    print("\n" + "=" * 80)
    print(name)
    print("=" * 80)
    for k, v in metrics.items():
        print(f"{k}: {v}")


def get_model_score(model, X_data):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X_data)[:, 1]
    else:
        return model.decision_function(X_data)

## 3. Feature Engineering Shared by Model 1, Model 5, and Model 6

This feature path builds the review-, business-, user-, tip-, interaction-, and target-encoding features used by the final blend and logistic-regression ablation models. Because Model 1 is the submitted model, feature ordering, missing-value handling, and encoding steps are preserved as closely as possible.


## Create external holiday-based features from review dates
This section adds U.S. holiday-related variables based on each review date. It identifies whether a review was written on a U.S. holiday, calculates how many days the review is from the nearest holiday, and measures the days since the previous holiday and until the next holiday. It also creates holiday-window indicators, such as whether the review was written within 3 or 7 days of a holiday. 

In [ ]:
# ============================================================
# 4. External Holiday Features
# ============================================================

def add_us_holiday_features(df):
    df = df.copy()

    if "date" not in df.columns:
        df["is_us_holiday"] = 0.0
        df["days_to_nearest_holiday"] = 999.0
        df["days_since_previous_holiday"] = 999.0
        df["days_until_next_holiday"] = 999.0
        df["is_holiday_window_3"] = 0.0
        df["is_holiday_window_7"] = 0.0
        df["is_before_holiday_3"] = 0.0
        df["is_after_holiday_3"] = 0.0
        return df

    date_series = pd.to_datetime(df["date"], errors="coerce")
    years = date_series.dt.year.dropna().astype(int)

    if len(years) == 0:
        min_year, max_year = 2004, 2023
    else:
        min_year = int(years.min()) - 1
        max_year = int(years.max()) + 1

    us_holidays = holidays.US(years=range(min_year, max_year + 1))
    holiday_dates = sorted(list(us_holidays.keys()))

    holiday_ord = np.array(
        pd.to_datetime(holiday_dates).values.astype("datetime64[D]").astype("int64")
    )

    date_day = date_series.dt.floor("D")
    valid_mask = date_series.notna().values
    date_ord_all = date_day.values.astype("datetime64[D]").astype("int64")

    is_holiday = np.zeros(len(df), dtype="float32")
    nearest_days = np.full(len(df), 999.0, dtype="float32")
    since_prev = np.full(len(df), 999.0, dtype="float32")
    until_next = np.full(len(df), 999.0, dtype="float32")

    valid_dates = date_ord_all[valid_mask]

    idx = np.searchsorted(holiday_ord, valid_dates)

    has_prev = idx > 0
    has_next = idx < len(holiday_ord)

    prev_days = np.full(len(valid_dates), 999.0)
    next_days = np.full(len(valid_dates), 999.0)

    prev_days[has_prev] = valid_dates[has_prev] - holiday_ord[idx[has_prev] - 1]
    next_days[has_next] = holiday_ord[idx[has_next]] - valid_dates[has_next]

    exact_holiday = np.isin(valid_dates, holiday_ord)
    prev_days[exact_holiday] = 0
    next_days[exact_holiday] = 0

    nearest = np.minimum(prev_days, next_days)

    is_holiday[valid_mask] = exact_holiday.astype("float32")
    nearest_days[valid_mask] = nearest.astype("float32")
    since_prev[valid_mask] = prev_days.astype("float32")
    until_next[valid_mask] = next_days.astype("float32")

    df["is_us_holiday"] = is_holiday
    df["days_to_nearest_holiday"] = nearest_days
    df["days_since_previous_holiday"] = since_prev
    df["days_until_next_holiday"] = until_next

    df["is_holiday_window_3"] = (df["days_to_nearest_holiday"] <= 3).astype("float32")
    df["is_holiday_window_7"] = (df["days_to_nearest_holiday"] <= 7).astype("float32")
    df["is_before_holiday_3"] = (df["days_until_next_holiday"] <= 3).astype("float32")
    df["is_after_holiday_3"] = (df["days_since_previous_holiday"] <= 3).astype("float32")

    return df

## Create review-level text, sentiment, date, rating, and holiday features
This section transforms raw review data into rich review-level predictors by engineering features from multiple dimensions of each review. It extracts textual structure features such as review length, word count, sentence count, punctuation usage, capitalization patterns, and digit frequency; builds simple sentiment indicators using predefined positive and negative keyword counts; captures temporal patterns from review dates including year, month, weekday, weekend behavior, and holiday proximity; and incorporates star rating behavior through positive, negative, and extreme rating flags. Together, these features convert unstructured review text and metadata into machine-learning-ready variables that capture writing style, sentiment tone, timing effects, and rating behavior.

In [ ]:
# ============================================================
# 5. Review-Level Features
# ============================================================

def add_review_features(df):
    df = df.copy()

    if "text" not in df.columns:
        df["text"] = ""

    df["text"] = df["text"].fillna("").astype(str)

    df["text_len"] = df["text"].str.len().astype("float32")
    df["word_count"] = df["text"].str.split().str.len().astype("float32")
    df["avg_word_len"] = df["text_len"] / (df["word_count"] + 1)

    df["sentence_count"] = df["text"].str.count(r"[.!?]+").astype("float32")
    df["exclaim_count"] = df["text"].str.count("!").astype("float32")
    df["question_count"] = df["text"].str.count(r"\?").astype("float32")
    df["comma_count"] = df["text"].str.count(",").astype("float32")
    df["period_count"] = df["text"].str.count(r"\.").astype("float32")
    df["digit_count"] = df["text"].str.count(r"\d").astype("float32")

    df["upper_count"] = df["text"].str.count(r"[A-Z]").astype("float32")
    df["upper_ratio"] = df["upper_count"] / (df["text_len"] + 1)
    df["upper_per_word"] = df["upper_count"] / (df["word_count"] + 1)

    df["log_text_len"] = np.log1p(df["text_len"].clip(lower=0))
    df["log_word_count"] = np.log1p(df["word_count"].clip(lower=0))

    df["punctuation_total"] = (
            df["exclaim_count"] +
            df["question_count"] +
            df["comma_count"] +
            df["period_count"]
    )

    df["period_per_word"] = df["period_count"] / (df["word_count"] + 1)
    df["punctuation_per_word"] = df["punctuation_total"] / (df["word_count"] + 1)

    positive_words = [
        "good", "great", "excellent", "amazing", "awesome", "best",
        "love", "loved", "perfect", "delicious", "friendly", "nice",
        "wonderful", "fantastic", "recommend", "favorite", "fresh",
        "clean", "fast", "helpful", "enjoyed"
    ]

    negative_words = [
        "bad", "terrible", "awful", "worst", "hate", "hated",
        "poor", "rude", "slow", "dirty", "disappointed",
        "horrible", "never", "overpriced", "cold", "bland",
        "wait", "waiting", "wrong", "expensive"
    ]

    text_lower = df["text"].str.lower()

    df["positive_word_count"] = 0
    for w in positive_words:
        df["positive_word_count"] += text_lower.str.count(r"\b" + w + r"\b")

    df["negative_word_count"] = 0
    for w in negative_words:
        df["negative_word_count"] += text_lower.str.count(r"\b" + w + r"\b")

    df["positive_word_count"] = df["positive_word_count"].astype("float32")
    df["negative_word_count"] = df["negative_word_count"].astype("float32")

    df["sentiment_balance"] = df["positive_word_count"] - df["negative_word_count"]
    df["positive_word_ratio"] = df["positive_word_count"] / (df["word_count"] + 1)
    df["negative_word_ratio"] = df["negative_word_count"] / (df["word_count"] + 1)

    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"], errors="coerce")
        df["review_year"] = df["date"].dt.year.astype("float32")
        df["review_month"] = df["date"].dt.month.astype("float32")
        df["review_dayofweek"] = df["date"].dt.dayofweek.astype("float32")
        df["review_hour"] = df["date"].dt.hour.astype("float32")
        df["is_weekend"] = df["review_dayofweek"].isin([5, 6]).astype("float32")
    else:
        df["review_year"] = np.nan
        df["review_month"] = np.nan
        df["review_dayofweek"] = np.nan
        df["review_hour"] = np.nan
        df["is_weekend"] = np.nan

    if "stars" in df.columns:
        df["review_stars"] = pd.to_numeric(df["stars"], errors="coerce").astype("float32")
    else:
        df["review_stars"] = np.nan

    df["is_extreme_star"] = ((df["review_stars"] <= 2) | (df["review_stars"] >= 5)).astype("float32")
    df["is_positive_star"] = (df["review_stars"] >= 4).astype("float32")
    df["is_negative_star"] = (df["review_stars"] <= 2).astype("float32")

    df = add_us_holiday_features(df)

    return df

## Clean and create business-level features from business metadata
This section processes the raw business dataset and transforms business information into structured numerical features for modeling. It cleans core variables such as ratings, review counts, location, and open status, parses business categories into keyword-based indicators, extracts and converts nested business attributes (such as price range, WiFi, alcohol, reservations, and delivery options), and derives operational features like days open and average daily hours from business hours data. The goal is to convert complex categorical and semi-structured business metadata into machine-learning-ready business-level predictors.

In [ ]:
# ============================================================
# 6. Business Features
# ============================================================

def parse_attr_dict(val):
    if isinstance(val, dict):
        return val

    if pd.isna(val):
        return {}

    s = str(val)

    try:
        return json.loads(
            s.replace("'", '"')
            .replace("True", "true")
            .replace("False", "false")
            .replace("None", "null")
        )
    except Exception:
        try:
            return ast.literal_eval(s)
        except Exception:
            return {}


def get_numeric_col(df, col, default=np.nan):
    if col in df.columns:
        return pd.to_numeric(df[col], errors="coerce")
    else:
        return pd.Series([default] * len(df), index=df.index)


def attr_to_number(attr_dict, key):
    val = attr_dict.get(key, np.nan)

    if isinstance(val, bool):
        return float(val)

    if pd.isna(val):
        return np.nan

    val_str = str(val).strip().strip("'\"").lower()

    if val_str in ["true", "1", "yes"]:
        return 1.0
    if val_str in ["false", "0", "no"]:
        return 0.0
    if val_str in ["none", "nan", "null", ""]:
        return np.nan

    try:
        return float(val_str)
    except Exception:
        return np.nan


def map_attr_value(val, mapping):
    if pd.isna(val):
        return np.nan

    val_str = str(val).strip().strip("'\"").lower()
    return mapping.get(val_str, np.nan)


def clean_business_features(business):
    b = business.copy()
    b = b.drop_duplicates(subset=["business_id"], keep="first")

    b["business_stars"] = get_numeric_col(b, "stars").astype("float32")
    b["business_review_count"] = get_numeric_col(b, "review_count").astype("float32")
    b["is_open"] = get_numeric_col(b, "is_open", default=0).fillna(0).astype("float32")
    b["latitude"] = get_numeric_col(b, "latitude").astype("float32")
    b["longitude"] = get_numeric_col(b, "longitude").astype("float32")

    b["log_business_review_count"] = np.log1p(b["business_review_count"].clip(lower=0))

    if "categories" in b.columns:
        b["categories"] = b["categories"].fillna("").astype(str).str.lower()
    else:
        b["categories"] = ""

    b["category_count"] = b["categories"].apply(
        lambda x: 0 if x == "" else len([c for c in x.split(",") if c.strip()])
    ).astype("float32")

    category_keywords = {
        "cat_restaurant": "restaurant",
        "cat_food": "food",
        "cat_bar": "bar",
        "cat_nightlife": "nightlife",
        "cat_coffee": "coffee",
        "cat_shopping": "shopping",
        "cat_beauty": "beauty",
        "cat_hotel": "hotel",
        "cat_auto": "auto",
        "cat_health": "health",
        "cat_mexican": "mexican",
        "cat_chinese": "chinese",
        "cat_japanese": "japanese",
        "cat_american": "american",
        "cat_pizza": "pizza",
        "cat_burgers": "burgers",
        "cat_seafood": "seafood",
        "cat_breakfast": "breakfast",
        "cat_fastfood": "fast food",
        "cat_italian": "italian",
        "cat_sushi": "sushi",
        "cat_sandwich": "sandwich",
        "cat_event": "event planning",
        "cat_service": "services"
    }

    for new_col, keyword in category_keywords.items():
        b[new_col] = b["categories"].str.contains(keyword, regex=False).astype("float32")

    if "attributes" in b.columns:
        attrs = b["attributes"].apply(parse_attr_dict)
    else:
        attrs = pd.Series([{} for _ in range(len(b))], index=b.index)

    simple_attr_keys = [
        "BusinessAcceptsCreditCards",
        "BikeParking",
        "ByAppointmentOnly",
        "RestaurantsReservations",
        "RestaurantsGoodForGroups",
        "RestaurantsTakeOut",
        "GoodForKids",
        "HasTV",
        "Caters",
        "HappyHour",
        "OutdoorSeating",
        "RestaurantsDelivery",
        "RestaurantsTableService",
        "WheelchairAccessible",
        "DogsAllowed"
    ]

    for key in simple_attr_keys:
        b["attr_" + key] = attrs.apply(lambda d: attr_to_number(d, key)).astype("float32")

    attire_map = {"casual": 1.0, "dressy": 2.0, "formal": 3.0}
    alcohol_map = {"none": 0.0, "beer_and_wine": 1.0, "full_bar": 2.0}
    noise_map = {"quiet": 1.0, "average": 2.0, "loud": 3.0, "very_loud": 4.0}
    wifi_map = {"no": 0.0, "free": 1.0, "paid": 2.0}

    price_raw = attrs.apply(lambda d: d.get("RestaurantsPriceRange2", np.nan))
    b["attr_RestaurantsPriceRange2"] = pd.to_numeric(price_raw, errors="coerce").astype("float32")

    b["attr_RestaurantsAttire"] = attrs.apply(
        lambda d: map_attr_value(d.get("RestaurantsAttire", np.nan), attire_map)
    ).astype("float32")

    b["attr_Alcohol"] = attrs.apply(
        lambda d: map_attr_value(d.get("Alcohol", np.nan), alcohol_map)
    ).astype("float32")

    b["attr_NoiseLevel"] = attrs.apply(
        lambda d: map_attr_value(d.get("NoiseLevel", np.nan), noise_map)
    ).astype("float32")

    b["attr_WiFi"] = attrs.apply(
        lambda d: map_attr_value(d.get("WiFi", np.nan), wifi_map)
    ).astype("float32")

    def parse_hours(val):
        if pd.isna(val):
            return 0, 0.0

        try:
            h = ast.literal_eval(str(val))
            if not isinstance(h, dict):
                return 0, 0.0

            days_open = len(h)
            total_hours = 0.0

            for _, timerange in h.items():
                parts = str(timerange).split("-")
                if len(parts) != 2:
                    continue

                s1 = parts[0].split(":")
                s2 = parts[1].split(":")

                start = float(s1[0]) + float(s1[1]) / 60
                end = float(s2[0]) + float(s2[1]) / 60

                diff = end - start
                if diff < 0:
                    diff += 24

                total_hours += diff

            avg_hours = total_hours / days_open if days_open > 0 else 0.0
            return days_open, avg_hours

        except Exception:
            return 0, 0.0

    if "hours" in b.columns:
        hours_parsed = b["hours"].apply(parse_hours)
        b["days_open"] = hours_parsed.apply(lambda x: x[0]).astype("float32")
        b["avg_hours_per_day"] = hours_parsed.apply(lambda x: x[1]).astype("float32")
    else:
        b["days_open"] = 0.0
        b["avg_hours_per_day"] = 0.0

    keep_cols = [
        "business_id",
        "business_stars",
        "business_review_count",
        "log_business_review_count",
        "is_open",
        "latitude",
        "longitude",
        "category_count",
        "days_open",
        "avg_hours_per_day"
    ]

    keep_cols += list(category_keywords.keys())
    keep_cols += ["attr_" + key for key in simple_attr_keys]
    keep_cols += [
        "attr_RestaurantsPriceRange2",
        "attr_RestaurantsAttire",
        "attr_Alcohol",
        "attr_NoiseLevel",
        "attr_WiFi"
    ]

    keep_cols = [c for c in keep_cols if c in b.columns]

    return b[keep_cols]

## Clean and create user-level features from user metadata
This section transforms raw user information into structured user-level predictors for modeling. It removes duplicate users, converts numerical user statistics such as review count, funny votes, cool votes, fans, and average stars into usable numeric features, and optionally includes user useful votes depending on the leakage setting. It also creates behavioral features such as elite-year count, friend count, and the year the user started using Yelp. 

In [ ]:
# ============================================================
# 7. User Features
# ============================================================

def clean_user_features(user):
    u = user.copy()
    u = u.drop_duplicates(subset=["user_id"], keep="first")

    numeric_cols = ["review_count", "funny", "cool", "fans", "average_stars"]

    if USE_USER_USEFUL_FEATURES:
        numeric_cols.append("useful")

    for col in numeric_cols:
        if col in u.columns:
            u["user_" + col] = pd.to_numeric(u[col], errors="coerce").astype("float32")
        else:
            u["user_" + col] = np.nan

    if "elite" in u.columns:
        u["elite"] = u["elite"].fillna("").astype(str)
        u["elite"] = u["elite"].str.replace("20,20", "2020", regex=False)

        u["user_elite_count"] = u["elite"].apply(
            lambda x: 0 if x.strip() == "" or x.lower() in ["nan", "none"]
            else len([i for i in x.split(",") if i.strip()])
        ).astype("float32")
    else:
        u["user_elite_count"] = 0.0

    if "friends" in u.columns:
        u["friends"] = u["friends"].fillna("").astype(str)
        u["user_friend_count"] = u["friends"].apply(
            lambda x: 0 if x.strip() == "" or x.lower() in ["none", "nan"]
            else len([i for i in x.split(",") if i.strip()])
        ).astype("float32")
    else:
        u["user_friend_count"] = 0.0

    if "yelping_since" in u.columns:
        u["yelping_since"] = pd.to_datetime(u["yelping_since"], errors="coerce")
        u["yelping_since_year"] = u["yelping_since"].dt.year.astype("float32")
    else:
        u["yelping_since_year"] = np.nan

    keep_cols = ["user_id"]
    keep_cols += ["user_" + col for col in numeric_cols]
    keep_cols += [
        "user_elite_count",
        "user_friend_count",
        "yelping_since_year"
    ]

    keep_cols = [c for c in keep_cols if c in u.columns]

    return u[keep_cols]

## Aggregate tip-level features for businesses and users
This section summarizes tip data into business-level and user-level aggregated features. It extracts basic text characteristics from tips, such as text length and word count, incorporates compliment counts, and then groups the data separately by business and by user. For each business and user, it calculates total tip volume, average tip length, average tip word count, and total compliments received.

In [ ]:
# ============================================================
# 8. Tip Aggregation
# ============================================================

def aggregate_tip_features(tip):
    t = tip.copy()

    if "text" not in t.columns:
        t["text"] = ""

    t["text"] = t["text"].fillna("").astype(str)
    t["tip_text_len"] = t["text"].str.len().astype("float32")
    t["tip_word_count"] = t["text"].str.split().str.len().astype("float32")

    if "compliment_count" in t.columns:
        t["compliment_count"] = pd.to_numeric(
            t["compliment_count"], errors="coerce"
        ).fillna(0).astype("float32")
    else:
        t["compliment_count"] = 0.0

    tip_business = t.groupby("business_id", as_index=False).agg(
        business_tip_count=("text", "count"),
        business_tip_avg_len=("tip_text_len", "mean"),
        business_tip_avg_words=("tip_word_count", "mean"),
        business_tip_compliments=("compliment_count", "sum")
    )

    tip_user = t.groupby("user_id", as_index=False).agg(
        user_tip_count=("text", "count"),
        user_tip_avg_len=("tip_text_len", "mean"),
        user_tip_avg_words=("tip_word_count", "mean"),
        user_tip_compliments=("compliment_count", "sum")
    )

    for df in [tip_business, tip_user]:
        for c in df.columns:
            if c not in ["business_id", "user_id"]:
                df[c] = pd.to_numeric(df[c], errors="coerce").astype("float32")

    return tip_business, tip_user

## Build final feature tables for training and testing
This section combines all engineered features into the final training and testing feature tables. It first creates cleaned business features, user features, and aggregated tip features, then applies review-level feature engineering to both train and test datasets. After that, it merges business-level, user-level, business-tip, and user-tip features into the review data using business_id and user_id. Finally, it prints the table dimensions and total processing time to confirm the feature-building process.

In [ ]:
# ============================================================
# 9. Build Feature Tables
# ============================================================

t0 = time.time()

business_clean = clean_business_features(business)
user_clean = clean_user_features(user)
tip_business, tip_user = aggregate_tip_features(tip)

print("business_clean:", business_clean.shape)
print("user_clean:", user_clean.shape)
print("tip_business:", tip_business.shape)
print("tip_user:", tip_user.shape)

train_fe = add_review_features(train_x)
test_fe = add_review_features(test_x)

print("train_fe before merge:", train_fe.shape)
print("test_fe before merge:", test_fe.shape)

train_fe = train_fe.merge(business_clean, on="business_id", how="left")
test_fe = test_fe.merge(business_clean, on="business_id", how="left")

train_fe = train_fe.merge(user_clean, on="user_id", how="left")
test_fe = test_fe.merge(user_clean, on="user_id", how="left")

train_fe = train_fe.merge(tip_business, on="business_id", how="left")
test_fe = test_fe.merge(tip_business, on="business_id", how="left")

train_fe = train_fe.merge(tip_user, on="user_id", how="left")
test_fe = test_fe.merge(tip_user, on="user_id", how="left")

print("train_fe after merge:", train_fe.shape)
print("test_fe after merge:", test_fe.shape)
print("Feature table time:", (time.time() - t0) / 60, "minutes")

## Create interaction, ratio, and log-transformed features
This section adds advanced engineered features to capture relationships between review, user, and business information. It calculates rating differences between the review rating, business average rating, and user average rating, applies log transformations to reduce skewness in count-based variables, and creates per-review ratios such as user feedback per review and fans per review. It also builds interaction features, such as review length combined with user activity, business popularity, tip counts, and rating sentiment. These features help the model capture more complex patterns that simple individual variables may miss.

In [ ]:
# ============================================================
# 10. Interaction, Ratio, Log Features
# ============================================================

for df in [train_fe, test_fe]:

    df["star_diff_from_business"] = df["review_stars"] - df["business_stars"]
    df["user_star_diff"] = df["review_stars"] - df["user_average_stars"]

    df["abs_star_diff_from_business"] = df["star_diff_from_business"].abs()
    df["abs_user_star_diff"] = df["user_star_diff"].abs()

    df["log_user_review_count"] = np.log1p(df["user_review_count"].fillna(0).clip(lower=0))
    df["log_user_fans"] = np.log1p(df["user_fans"].fillna(0).clip(lower=0))
    df["log_user_friend_count"] = np.log1p(df["user_friend_count"].fillna(0).clip(lower=0))
    df["log_user_elite_count"] = np.log1p(df["user_elite_count"].fillna(0).clip(lower=0))

    if USE_USER_USEFUL_FEATURES and "user_useful" in df.columns:
        df["log_user_useful"] = np.log1p(df["user_useful"].fillna(0).clip(lower=0))
    else:
        df["user_useful"] = 0.0
        df["log_user_useful"] = 0.0

    if "user_funny" not in df.columns:
        df["user_funny"] = 0.0

    if "user_cool" not in df.columns:
        df["user_cool"] = 0.0

    df["log_user_funny"] = np.log1p(df["user_funny"].fillna(0).clip(lower=0))
    df["log_user_cool"] = np.log1p(df["user_cool"].fillna(0).clip(lower=0))

    df["user_total_feedback"] = df["user_useful"] + df["user_funny"] + df["user_cool"]
    df["log_user_total_feedback"] = np.log1p(df["user_total_feedback"].fillna(0).clip(lower=0))

    df["user_feedback_per_review"] = df["user_total_feedback"] / (df["user_review_count"] + 1)
    df["user_useful_per_review"] = df["user_useful"] / (df["user_review_count"] + 1)
    df["user_funny_per_review"] = df["user_funny"] / (df["user_review_count"] + 1)
    df["user_cool_per_review"] = df["user_cool"] / (df["user_review_count"] + 1)
    df["user_fans_per_review"] = df["user_fans"] / (df["user_review_count"] + 1)

    df["user_useful_share"] = df["user_useful"] / (df["user_total_feedback"] + 1)
    df["user_funny_share"] = df["user_funny"] / (df["user_total_feedback"] + 1)
    df["user_cool_share"] = df["user_cool"] / (df["user_total_feedback"] + 1)

    df["review_len_x_user_review_count"] = df["log_text_len"] * df["log_user_review_count"]
    df["review_len_x_user_useful"] = df["log_text_len"] * df["user_useful_per_review"]
    df["log_text_x_log_user_useful"] = df["log_text_len"] * df["log_user_useful"]
    df["log_text_x_user_feedback_per_review"] = df["log_text_len"] * df["user_feedback_per_review"]
    df["log_word_x_log_user_useful"] = df["log_word_count"] * df["log_user_useful"]

    df["business_popularity_x_star_diff"] = df["log_business_review_count"] * df["star_diff_from_business"]
    df["business_popularity_x_text_len"] = df["log_business_review_count"] * df["log_text_len"]
    df["log_text_x_log_business_reviews"] = df["log_text_len"] * df["log_business_review_count"]
    df["log_word_x_log_business_reviews"] = df["log_word_count"] * df["log_business_review_count"]

    df["negative_long_review"] = df["is_negative_star"] * df["log_text_len"]
    df["positive_long_review"] = df["is_positive_star"] * df["log_text_len"]
    df["extreme_long_review"] = df["is_extreme_star"] * df["log_text_len"]

    df["log_business_tip_count"] = np.log1p(df["business_tip_count"].fillna(0).clip(lower=0))
    df["log_user_tip_count"] = np.log1p(df["user_tip_count"].fillna(0).clip(lower=0))

    df["business_tip_x_text_len"] = df["log_business_tip_count"] * df["log_text_len"]
    df["user_tip_x_text_len"] = df["log_user_tip_count"] * df["log_text_len"]

train_fe = train_fe.loc[:, ~train_fe.columns.duplicated()]
test_fe = test_fe.loc[:, ~test_fe.columns.duplicated()]

print("train_fe after interaction:", train_fe.shape)
print("test_fe after interaction:", test_fe.shape)

gc.collect()

## Create out-of-fold target encoding features
This section creates target encoding features for high-cardinality variables such as user_id, business_id, review year, and review stars. It uses out-of-fold encoding to reduce data leakage by calculating encoded values from training folds only, then applies smoothed full-data encoding to the test set. It also creates target-encoding interaction features with review length and user/business combinations. These features help the model capture historical usefulness patterns by user, business, rating, and time while controlling overfitting.

In [ ]:
# 11. Out-of-Fold Target Encoding

def add_oof_target_encoding(
        train_df,
        test_df,
        y,
        cols,
        n_splits=5,
        smoothing=30,
        random_state=42
):
    train_df = train_df.copy()
    test_df = test_df.copy()
    y = pd.Series(y).reset_index(drop=True)

    global_mean = y.mean()

    if isinstance(cols, str):
        cols = [cols]

    enc_name = "te_" + "_".join(cols)

    key_train = train_df[cols].astype(str).agg("_".join, axis=1)
    key_test = test_df[cols].astype(str).agg("_".join, axis=1)

    oof_values = np.zeros(len(train_df), dtype="float32")

    skf = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state
    )

    for tr_idx, va_idx in skf.split(train_df, y):
        fold_key_train = key_train.iloc[tr_idx]
        fold_y_train = y.iloc[tr_idx]

        stats = pd.DataFrame({
            "key": fold_key_train.values,
            "target": fold_y_train.values
        }).groupby("key")["target"].agg(["mean", "count"])

        stats["smooth"] = (
                                  stats["mean"] * stats["count"] + global_mean * smoothing
                          ) / (stats["count"] + smoothing)

        val_keys = key_train.iloc[va_idx]
        oof_values[va_idx] = val_keys.map(stats["smooth"]).fillna(global_mean).astype("float32")

    full_stats = pd.DataFrame({
        "key": key_train.values,
        "target": y.values
    }).groupby("key")["target"].agg(["mean", "count"])

    full_stats["smooth"] = (
                                   full_stats["mean"] * full_stats["count"] + global_mean * smoothing
                           ) / (full_stats["count"] + smoothing)

    train_df[enc_name] = oof_values
    test_df[enc_name] = key_test.map(full_stats["smooth"]).fillna(global_mean).astype("float32")

    print("Added target encoding:", enc_name)

    return train_df, test_df


y_series = pd.Series(y).reset_index(drop=True).astype(int)
train_fe = train_fe.reset_index(drop=True)
test_fe = test_fe.reset_index(drop=True)

single_te_cols = [
    "user_id",
    "business_id",
    "review_year",
    "review_stars"
]

for col in single_te_cols:
    if col in train_fe.columns and col in test_fe.columns:
        train_fe, test_fe = add_oof_target_encoding(
            train_fe,
            test_fe,
            y_series,
            cols=[col],
            n_splits=5,
            smoothing=30,
            random_state=RANDOM_STATE
        )

combo_te_cols = [
    ["business_id", "review_stars"],
    ["user_id", "review_stars"],
    ["business_id", "review_year"],
    ["user_id", "review_year"],
    ["review_year", "review_stars"]
]

for cols in combo_te_cols:
    if all(c in train_fe.columns for c in cols) and all(c in test_fe.columns for c in cols):
        train_fe, test_fe = add_oof_target_encoding(
            train_fe,
            test_fe,
            y_series,
            cols=cols,
            n_splits=5,
            smoothing=50,
            random_state=RANDOM_STATE
        )

for df in [train_fe, test_fe]:

    if "te_user_id" in df.columns:
        df["te_user_x_log_text_len"] = df["te_user_id"] * df["log_text_len"]

    if "te_business_id" in df.columns:
        df["te_business_x_log_text_len"] = df["te_business_id"] * df["log_text_len"]

    if "te_user_id" in df.columns and "te_business_id" in df.columns:
        df["te_user_minus_business"] = df["te_user_id"] - df["te_business_id"]
        df["te_user_x_business"] = df["te_user_id"] * df["te_business_id"]

    if "te_business_id_review_stars" in df.columns:
        df["te_business_star_x_log_text"] = df["te_business_id_review_stars"] * df["log_text_len"]

    if "te_user_id_review_stars" in df.columns:
        df["te_user_star_x_log_text"] = df["te_user_id_review_stars"] * df["log_text_len"]

te_cols = [c for c in train_fe.columns if c.startswith("te_")]

print("Number of TE columns:", len(te_cols))
print("train_fe after TE:", train_fe.shape)
print("test_fe after TE:", test_fe.shape)

gc.collect()

## Select final numeric features for modeling

In [ ]:
# 12. Select Numeric Features

drop_cols = [
    "review_id",
    "user_id",
    "business_id",
    "text",
    "date",
    "stars"
]

feature_cols = [c for c in train_fe.columns if c not in drop_cols]

X = train_fe[feature_cols].select_dtypes(include=["number", "bool"]).copy()
X_test_final = test_fe[X.columns].copy()

X = X.replace([np.inf, -np.inf], np.nan)
X_test_final = X_test_final.replace([np.inf, -np.inf], np.nan)

X = X.astype("float32")
X_test_final = X_test_final.astype("float32")

print("Number of features:", X.shape[1])
print("X:", X.shape)
print("X_test_final:", X_test_final.shape)

feature_list_path = os.path.join(OUTPUT_DIR, "super_final_feature_list.csv")
pd.DataFrame({"feature": X.columns}).to_csv(feature_list_path, index=False)
print("Saved feature list:", feature_list_path)

## Split data into Train and Validation sets

In [ ]:
# 13. Train / Validation Split

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("y_train positive rate:", y_train.mean())
print("y_val positive rate:", y_val.mean())

gc.collect()

## 4. Model 1: Final Blended Model

This is the primary submission model. It combines predictions from multiple model families and selects the final operating threshold using the required FPR constraint. 

## Generate feature insight summaries and exploratory outputs
This section analyzes key engineered features by comparing their average values between top useful and non-top useful reviews. It creates summary tables showing feature importance through mean differences, saves these insights for reporting, and generates visualizations to highlight which features most strongly distinguish the target classes. It also explores how top useful rates vary by review star ratings, providing interpretable patterns that support feature understanding and presentation.

In [ ]:
# ============================================================
# 14. Feature Insight Outputs
# ============================================================

def create_feature_insight_outputs(train_features, y, output_dir):
    insight_cols = [
        "log_text_len",
        "log_word_count",
        "review_stars",
        "business_stars",
        "log_business_review_count",
        "log_user_review_count",
        "user_fans",
        "user_average_stars",
        "positive_word_count",
        "negative_word_count",
        "star_diff_from_business",
        "user_useful_per_review",
        "is_us_holiday",
        "is_holiday_window_7"
    ]

    insight_cols = [c for c in insight_cols if c in train_features.columns]

    temp = train_features[insight_cols].copy()
    temp["top_useful"] = y.values

    summary = temp.groupby("top_useful")[insight_cols].mean().T
    summary.columns = ["mean_when_not_top_useful", "mean_when_top_useful"]
    summary["difference_top_minus_not"] = (
            summary["mean_when_top_useful"] - summary["mean_when_not_top_useful"]
    )
    summary["abs_difference"] = summary["difference_top_minus_not"].abs()
    summary = summary.sort_values("abs_difference", ascending=False)

    summary_path = os.path.join(output_dir, "super_feature_insight_10plus_features.csv")
    summary.to_csv(summary_path)

    print("Saved feature insight summary:", summary_path)
    print(summary)

    plt.figure(figsize=(10, 7))
    summary["difference_top_minus_not"].sort_values().plot(kind="barh")
    plt.title("Feature Mean Difference: Top Useful vs Not Top Useful")
    plt.xlabel("Mean Difference")
    plt.ylabel("Feature")
    plt.tight_layout()
    fig_path = os.path.join(output_dir, "super_feature_mean_difference_10plus_features.png")
    plt.savefig(fig_path, dpi=200)
    plt.show()

    if "review_stars" in train_features.columns:
        star_table = temp.groupby("review_stars")["top_useful"].mean().reset_index()
        star_path = os.path.join(output_dir, "super_target_rate_by_review_stars.csv")
        star_table.to_csv(star_path, index=False)

        plt.figure(figsize=(8, 5))
        plt.plot(star_table["review_stars"], star_table["top_useful"], marker="o")
        plt.title("Top Useful Rate by Review Stars")
        plt.xlabel("Review Stars")
        plt.ylabel("Top Useful Rate")
        plt.tight_layout()
        fig_path2 = os.path.join(output_dir, "super_target_rate_by_review_stars.png")
        plt.savefig(fig_path2, dpi=200)
        plt.show()

    return summary


feature_insight_summary = create_feature_insight_outputs(train_fe, y_series, OUTPUT_DIR)

## Evaluate the impact of external holiday features
This section tests whether the added U.S. holiday features improve model performance. It trains one LightGBM model without holiday-related features and another model with the full feature set, then compares their validation results using AUC, FPR, TPR, accuracy, and confusion matrix values.

In [ ]:
# ============================================================
# 15. External Feature Improvement Evidence
# ============================================================

holiday_feature_cols = [
    "is_us_holiday",
    "days_to_nearest_holiday",
    "days_since_previous_holiday",
    "days_until_next_holiday",
    "is_holiday_window_3",
    "is_holiday_window_7",
    "is_before_holiday_3",
    "is_after_holiday_3"
]

holiday_feature_cols = [c for c in holiday_feature_cols if c in X_train.columns]


def train_quick_lgb_and_eval(X_tr, y_tr, X_va, y_va, model_name):
    n_pos = (y_tr == 1).sum()
    n_neg = (y_tr == 0).sum()
    spw = n_neg / n_pos

    model = lgb.LGBMClassifier(
        objective="binary",
        n_estimators=700,
        learning_rate=0.04,
        num_leaves=63,
        min_child_samples=100,
        subsample=0.85,
        colsample_bytree=0.85,
        scale_pos_weight=spw,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    model.fit(
        X_tr,
        y_tr,
        eval_set=[(X_va, y_va)],
        eval_metric="auc",
        callbacks=[
            lgb.early_stopping(70),
            lgb.log_evaluation(0)
        ]
    )

    val_score = model.predict_proba(X_va)[:, 1]
    metrics = best_tpr_under_fpr(y_va, val_score, fpr_limit=FPR_LIMIT_FINAL)
    metrics["model"] = model_name
    return model, metrics


print("Holiday external columns:", holiday_feature_cols)

if len(holiday_feature_cols) > 0:
    X_train_no_external = X_train.drop(columns=holiday_feature_cols)
    X_val_no_external = X_val.drop(columns=holiday_feature_cols)

    base_lgb_external_model, base_external_metrics = train_quick_lgb_and_eval(
        X_train_no_external,
        y_train,
        X_val_no_external,
        y_val,
        "LGB without external holiday features"
    )

    holiday_lgb_external_model, holiday_external_metrics = train_quick_lgb_and_eval(
        X_train,
        y_train,
        X_val,
        y_val,
        "LGB with external holiday features"
    )

    external_results = pd.DataFrame([base_external_metrics, holiday_external_metrics])
    external_results = external_results[
        ["model", "auc", "fpr", "tpr", "threshold", "accuracy", "tn", "fp", "fn", "tp"]
    ]

    external_path = os.path.join(OUTPUT_DIR, "super_external_holiday_feature_improvement.csv")
    external_results.to_csv(external_path, index=False)

    print("\nExternal feature improvement evidence:")
    print(external_results)
    print("Saved:", external_path)

    del X_train_no_external, X_val_no_external
    gc.collect()

## Initialize score banks and model evaluation tracking
This section sets up the storage system for comparing multiple models in the model zoo. It calculates the class imbalance weight using the ratio of negative to positive samples, initializes dictionaries to store validation and test prediction scores, and creates a metrics list to track each model’s performance. 

In [ ]:
# ============================================================
# 16. Score Bank for Model Zoo
# ============================================================

n_pos = (y_train == 1).sum()
n_neg = (y_train == 0).sum()
scale_pos_weight = n_neg / n_pos

print("Positive:", n_pos)
print("Negative:", n_neg)
print("scale_pos_weight:", scale_pos_weight)

score_bank_val = {}
score_bank_test = {}
model_metrics = []


def add_model_scores(name, val_score, test_score):
    val_score = np.asarray(val_score).astype("float32")
    test_score = np.asarray(test_score).astype("float32")

    score_bank_val[name] = val_score
    score_bank_test[name] = test_score

    metrics = best_tpr_under_fpr(
        y_val,
        val_score,
        fpr_limit=FPR_LIMIT_FINAL
    )

    row = {
        "model": name,
        "auc": metrics["auc"],
        "fpr": metrics["fpr"],
        "tpr": metrics["tpr"],
        "threshold": metrics["threshold"]
    }

    model_metrics.append(row)

    print(
        f"{name:35s} | "
        f"AUC={metrics['auc']:.6f} | "
        f"FPR={metrics['fpr']:.6f} | "
        f"TPR={metrics['tpr']:.6f}"
    )

    return metrics

## Train multiple LightGBM models for the model zoo
This section trains a group of LightGBM models with different random seeds and hyperparameter settings. Each configuration changes elements such as learning rate, number of leaves, minimum child samples, subsampling, column sampling, and regularization to create model diversity. After each model is trained, its validation and test predictions are saved into the score bank, and its performance is evaluated using AUC, FPR, and TPR. 

In [ ]:
# ============================================================
# 17. Train LightGBM Model Zoo
# ============================================================

def train_lgb_add(name, params, seed=42):
    print("\n" + "=" * 80)
    print("Training LightGBM:", name)
    print("=" * 80)

    params = params.copy()
    n_estimators = params.pop("n_estimators")

    model = lgb.LGBMClassifier(
        objective="binary",
        n_estimators=n_estimators,
        scale_pos_weight=scale_pos_weight,
        random_state=seed,
        n_jobs=-1,
        **params
    )

    model.fit(
        X_train,
        y_train,
        eval_set=[(X_val, y_val)],
        eval_metric="auc",
        callbacks=[
            lgb.early_stopping(100),
            lgb.log_evaluation(100)
        ]
    )

    val_score = model.predict_proba(X_val)[:, 1]
    test_score = model.predict_proba(X_test_final)[:, 1]

    metrics = add_model_scores(name, val_score, test_score)

    gc.collect()

    return model, metrics


lgb_configs = [
    {
        "name": "lgb_main_seed42",
        "seed": 42,
        "params": {
            "n_estimators": 1700,
            "learning_rate": 0.03,
            "num_leaves": 127,
            "min_child_samples": 100,
            "subsample": 0.85,
            "colsample_bytree": 0.85
        }
    },
    {
        "name": "lgb_main_seed2024",
        "seed": 2024,
        "params": {
            "n_estimators": 1700,
            "learning_rate": 0.03,
            "num_leaves": 127,
            "min_child_samples": 100,
            "subsample": 0.85,
            "colsample_bytree": 0.85
        }
    },
    {
        "name": "lgb_leaves63_lr003",
        "seed": 42,
        "params": {
            "n_estimators": 1700,
            "learning_rate": 0.03,
            "num_leaves": 63,
            "min_child_samples": 80,
            "subsample": 0.85,
            "colsample_bytree": 0.85
        }
    },
    {
        "name": "lgb_leaves191_reg",
        "seed": 77,
        "params": {
            "n_estimators": 2200,
            "learning_rate": 0.025,
            "num_leaves": 191,
            "min_child_samples": 180,
            "subsample": 0.80,
            "colsample_bytree": 0.75,
            "reg_alpha": 1.0,
            "reg_lambda": 5.0
        }
    },
    {
        "name": "lgb_leaves255_safe",
        "seed": 123,
        "params": {
            "n_estimators": 2400,
            "learning_rate": 0.02,
            "num_leaves": 255,
            "min_child_samples": 220,
            "subsample": 0.80,
            "colsample_bytree": 0.80,
            "reg_alpha": 2.0,
            "reg_lambda": 8.0
        }
    },
    {
        "name": "lgb_small_leaves",
        "seed": 314,
        "params": {
            "n_estimators": 1600,
            "learning_rate": 0.035,
            "num_leaves": 31,
            "min_child_samples": 60,
            "subsample": 0.90,
            "colsample_bytree": 0.90
        }
    },
    {
        "name": "lgb_more_colsample",
        "seed": 888,
        "params": {
            "n_estimators": 1900,
            "learning_rate": 0.025,
            "num_leaves": 127,
            "min_child_samples": 120,
            "subsample": 0.90,
            "colsample_bytree": 0.95,
            "reg_lambda": 2.0
        }
    }
]

lgb_models = {}

if RUN_EXTRA_LGB:
    for cfg in lgb_configs:
        model, metrics = train_lgb_add(
            name=cfg["name"],
            params=cfg["params"],
            seed=cfg["seed"]
        )
        lgb_models[cfg["name"]] = model
else:
    model, metrics = train_lgb_add(
        name=lgb_configs[0]["name"],
        params=lgb_configs[0]["params"],
        seed=lgb_configs[0]["seed"]
    )
    lgb_models[lgb_configs[0]["name"]] = model

## Train multiple XGBoost models for the model zoo
This section trains several XGBoost models using different hyperparameter settings and random seeds. Each model varies parameters such as tree depth, learning rate, number of estimators, child weight, subsampling, column sampling, and regularization to improve model diversity. The function also supports GPU training if available, but automatically retries on CPU if the environment does not support the GPU setup. After each model is trained, its validation and test prediction scores are saved into the score bank, and its AUC, FPR, and TPR are recorded for later model comparison and blending.

In [ ]:
# ============================================================
# 18. Train XGBoost Model Zoo
# ============================================================

def train_xgb_add(name, params, seed=42):
    print("\n" + "=" * 80)
    print("Training XGBoost:", name)
    print("=" * 80)

    params = params.copy()

    base_params = {
        "objective": "binary:logistic",
        "eval_metric": "auc",
        "scale_pos_weight": scale_pos_weight,
        "tree_method": "hist",
        "random_state": seed,
        "n_jobs": -1
    }

    if USE_XGB_GPU:
        base_params["device"] = "cuda"

    model = xgb.XGBClassifier(
        **base_params,
        **params
    )

    try:
        model.fit(
            X_train,
            y_train,
            eval_set=[(X_val, y_val)],
            verbose=100
        )
    except Exception as e:
        print("XGBoost GPU or parameter issue. Retrying on CPU.")
        print("Original error:", e)

        base_params.pop("device", None)

        model = xgb.XGBClassifier(
            **base_params,
            **params
        )

        model.fit(
            X_train,
            y_train,
            eval_set=[(X_val, y_val)],
            verbose=100
        )

    val_score = model.predict_proba(X_val)[:, 1]
    test_score = model.predict_proba(X_test_final)[:, 1]

    metrics = add_model_scores(name, val_score, test_score)

    gc.collect()

    return model, metrics


xgb_configs = [
    {
        "name": "xgb_depth8_main_seed42",
        "seed": 42,
        "params": {
            "n_estimators": 1300,
            "learning_rate": 0.03,
            "max_depth": 8,
            "min_child_weight": 100,
            "subsample": 0.85,
            "colsample_bytree": 0.85,
            "reg_alpha": 0.0,
            "reg_lambda": 1.0
        }
    },
    {
        "name": "xgb_depth8_reg_seed2024",
        "seed": 2024,
        "params": {
            "n_estimators": 1500,
            "learning_rate": 0.025,
            "max_depth": 8,
            "min_child_weight": 120,
            "subsample": 0.90,
            "colsample_bytree": 0.80,
            "reg_alpha": 0.5,
            "reg_lambda": 3.0
        }
    },
    {
        "name": "xgb_depth6_safe",
        "seed": 77,
        "params": {
            "n_estimators": 1300,
            "learning_rate": 0.03,
            "max_depth": 6,
            "min_child_weight": 150,
            "subsample": 0.85,
            "colsample_bytree": 0.85,
            "reg_alpha": 1.0,
            "reg_lambda": 5.0
        }
    },
    {
        "name": "xgb_depth5_more_reg",
        "seed": 123,
        "params": {
            "n_estimators": 1200,
            "learning_rate": 0.035,
            "max_depth": 5,
            "min_child_weight": 200,
            "subsample": 0.80,
            "colsample_bytree": 0.75,
            "reg_alpha": 2.0,
            "reg_lambda": 8.0
        }
    },
    {
        "name": "xgb_depth9_large",
        "seed": 314,
        "params": {
            "n_estimators": 1100,
            "learning_rate": 0.025,
            "max_depth": 9,
            "min_child_weight": 150,
            "subsample": 0.80,
            "colsample_bytree": 0.80,
            "reg_alpha": 1.0,
            "reg_lambda": 5.0
        }
    }
]

xgb_models = {}

if RUN_EXTRA_XGB:
    for cfg in xgb_configs:
        model, metrics = train_xgb_add(
            name=cfg["name"],
            params=cfg["params"],
            seed=cfg["seed"]
        )
        xgb_models[cfg["name"]] = model
else:
    model, metrics = train_xgb_add(
        name=xgb_configs[0]["name"],
        params=xgb_configs[0]["params"],
        seed=xgb_configs[0]["seed"]
    )
    xgb_models[xgb_configs[0]["name"]] = model

## Train MLP models and create an ensemble
This section trains multiple MLP neural network models using different random seeds. Each MLP pipeline first imputes missing values, standardizes the features, and then trains a neural network classifier with early stopping. The validation and test prediction scores from each seed are saved, evaluated, and later averaged to create an MLP ensemble.

In [ ]:
# ============================================================
# 19. Train MLP Ensemble and Individual Seeds
# ============================================================

mlp_val_scores = []
mlp_test_scores = []
mlp_single_results = []

if RUN_EXTRA_MLP:
    for seed in MLP_SEEDS:
        print("\n" + "=" * 80)
        print(f"Training MLP, random_state={seed}")
        print("=" * 80)

        mlp_model = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", MLPClassifier(
                hidden_layer_sizes=(128, 64, 32),
                activation="relu",
                alpha=0.001,
                learning_rate_init=0.0005,
                max_iter=140,
                early_stopping=True,
                validation_fraction=0.10,
                random_state=seed
            ))
        ])

        mlp_model.fit(X_train, y_train)

        val_score = mlp_model.predict_proba(X_val)[:, 1]
        test_score = mlp_model.predict_proba(X_test_final)[:, 1]

        mlp_val_scores.append(val_score.astype("float32"))
        mlp_test_scores.append(test_score.astype("float32"))

        metrics = add_model_scores(f"mlp_seed_{seed}", val_score, test_score)

        mlp_single_results.append({
            "seed": seed,
            "auc": metrics["auc"],
            "fpr": metrics["fpr"],
            "tpr": metrics["tpr"],
            "threshold": metrics["threshold"]
        })

        gc.collect()

    mlp_val_score = np.mean(mlp_val_scores, axis=0)
    mlp_test_score = np.mean(mlp_test_scores, axis=0)

    mlp_ensemble_metrics = add_model_scores(
        "mlp_ensemble_all_seeds",
        mlp_val_score,
        mlp_test_score
    )

    mlp_single_results_df = pd.DataFrame(mlp_single_results)
    mlp_single_path = os.path.join(OUTPUT_DIR, "super_mlp_single_seed_results.csv")
    mlp_single_results_df.to_csv(mlp_single_path, index=False)
    print("Saved:", mlp_single_path)

## Summarize and rank all model zoo results
This section compiles the performance results from all trained models into a single summary table. It ranks models primarily by TPR and then by AUC, allowing easy comparison of which models perform best under the FPR constraint.

In [ ]:
# ============================================================
# 20. Model Zoo Summary
# ============================================================

model_metrics_df = pd.DataFrame(model_metrics)

model_metrics_df = model_metrics_df.sort_values(
    ["tpr", "auc"],
    ascending=False
).reset_index(drop=True)

model_zoo_path = os.path.join(OUTPUT_DIR, "super_model_zoo_results.csv")
model_metrics_df.to_csv(model_zoo_path, index=False)

print("\nModel zoo results:")
print(model_metrics_df)
print("Saved:", model_zoo_path)

## Perform advanced Top-K blend search for ensemble optimization
This section searches for the best ensemble blend by combining predictions from the top-performing models in the model zoo. It first selects the top K models based on validation performance, then evaluates multiple blending strategies including single-model baselines, equal-weight averaging, and thousands of randomly generated weighted blends using Dirichlet distributions. Candidate blends are first screened on a validation sample for speed, then the best candidates are fully evaluated on the entire validation set. The final output identifies the strongest blend combinations under the FPR constraint and saves both sample-search and full-evaluation results for later selection.

In [ ]:
# ============================================================
# 21. Advanced Top-K Blend Search
# ============================================================

def evaluate_blend_candidate(y_true, score, fpr_limit):
    return best_tpr_under_fpr(y_true, score, fpr_limit=fpr_limit)


def advanced_topk_blend_search(
        model_metrics_df,
        score_bank_val,
        score_bank_test,
        y_val,
        top_k=12,
        search_sample_size=200000,
        random_trials=30000,
        top_eval=300,
        random_state=42,
        fpr_limit=0.097
):
    top_k = min(top_k, len(model_metrics_df))

    top_model_names = model_metrics_df.head(top_k)["model"].tolist()

    print("\nTop models used for blend:")
    for name in top_model_names:
        print(name)

    val_matrix = np.column_stack([score_bank_val[name] for name in top_model_names]).astype("float32")
    test_matrix = np.column_stack([score_bank_test[name] for name in top_model_names]).astype("float32")

    rng = np.random.default_rng(random_state)

    # sample validation rows for fast blend search
    if search_sample_size < len(y_val):
        sample_idx, _ = train_test_split(
            np.arange(len(y_val)),
            train_size=search_sample_size,
            random_state=random_state,
            stratify=y_val
        )
    else:
        sample_idx = np.arange(len(y_val))

    y_sub = pd.Series(y_val).reset_index(drop=True).iloc[sample_idx].values
    val_sub = val_matrix[sample_idx, :]

    candidates = []

    # include single models
    for i, name in enumerate(top_model_names):
        w = np.zeros(top_k)
        w[i] = 1.0

        score_sub = val_sub @ w
        metrics = evaluate_blend_candidate(y_sub, score_sub, fpr_limit)

        candidates.append({
            "stage": "single",
            "auc_sub": metrics["auc"],
            "fpr_sub": metrics["fpr"],
            "tpr_sub": metrics["tpr"],
            "threshold_sub": metrics["threshold"],
            **{f"w_{top_model_names[j]}": w[j] for j in range(top_k)}
        })

    # equal weight
    w_equal = np.ones(top_k) / top_k
    score_sub = val_sub @ w_equal
    metrics = evaluate_blend_candidate(y_sub, score_sub, fpr_limit)

    candidates.append({
        "stage": "equal",
        "auc_sub": metrics["auc"],
        "fpr_sub": metrics["fpr"],
        "tpr_sub": metrics["tpr"],
        "threshold_sub": metrics["threshold"],
        **{f"w_{top_model_names[j]}": w_equal[j] for j in range(top_k)}
    })

    # random Dirichlet blends
    alphas = [0.15, 0.25, 0.5, 0.8, 1.0, 2.0]

    trials_per_alpha = max(1, random_trials // len(alphas))

    for alpha in alphas:
        print("Random blend search alpha:", alpha)

        for _ in range(trials_per_alpha):
            w = rng.dirichlet(np.ones(top_k) * alpha)

            score_sub = val_sub @ w
            metrics = evaluate_blend_candidate(y_sub, score_sub, fpr_limit)

            candidates.append({
                "stage": f"dirichlet_{alpha}",
                "auc_sub": metrics["auc"],
                "fpr_sub": metrics["fpr"],
                "tpr_sub": metrics["tpr"],
                "threshold_sub": metrics["threshold"],
                **{f"w_{top_model_names[j]}": w[j] for j in range(top_k)}
            })

    candidates_df = pd.DataFrame(candidates)

    candidates_df = candidates_df.sort_values(
        ["tpr_sub", "auc_sub"],
        ascending=False
    ).reset_index(drop=True)

    # evaluate top candidates on full validation
    full_results = []

    top_candidates = candidates_df.head(top_eval)

    for idx, row in top_candidates.iterrows():
        w = np.array([row[f"w_{name}"] for name in top_model_names])
        score_full = val_matrix @ w

        metrics = evaluate_blend_candidate(y_val, score_full, fpr_limit)

        full_results.append({
            "rank_from_sub": idx,
            "stage": row["stage"],
            "auc": metrics["auc"],
            "fpr": metrics["fpr"],
            "tpr": metrics["tpr"],
            "threshold": metrics["threshold"],
            **{f"w_{top_model_names[j]}": w[j] for j in range(top_k)}
        })

    full_results_df = pd.DataFrame(full_results)

    full_results_df = full_results_df.sort_values(
        ["tpr", "auc"],
        ascending=False
    ).reset_index(drop=True)

    return top_model_names, val_matrix, test_matrix, candidates_df, full_results_df


top_model_names, val_matrix, test_matrix, blend_candidates_df, advanced_blend_df = advanced_topk_blend_search(
    model_metrics_df=model_metrics_df,
    score_bank_val=score_bank_val,
    score_bank_test=score_bank_test,
    y_val=y_val,
    top_k=TOP_K_FOR_BLEND,
    search_sample_size=BLEND_SEARCH_SAMPLE_SIZE,
    random_trials=RANDOM_BLEND_TRIALS,
    top_eval=RANDOM_BLEND_TOP_EVAL,
    random_state=RANDOM_STATE,
    fpr_limit=FPR_LIMIT_FINAL
)

blend_candidates_path = os.path.join(OUTPUT_DIR, "super_blend_candidates_sample_search.csv")
advanced_blend_path = os.path.join(OUTPUT_DIR, "super_advanced_blend_full_eval.csv")

blend_candidates_df.to_csv(blend_candidates_path, index=False)
advanced_blend_df.to_csv(advanced_blend_path, index=False)

print("\nTop 20 advanced blend full-validation results:")
print(advanced_blend_df.head(20))

print("Saved sample blend candidates:", blend_candidates_path)
print("Saved full eval blend results:", advanced_blend_path)

## Refine the best blend weights locally
This section further improves the best blending result by making small random adjustments around the current best model weights. It adds noise to the best weight combination, clips negative weights to zero, normalizes the weights, and evaluates each refined blend on the validation set. The refined results are combined with the previous blend results, ranked by TPR and AUC, and saved for final ensemble selection.

In [ ]:
# ============================================================
# 22. Local Blend Refinement Around Best Weights
# ============================================================

def local_refine_blend(
        advanced_blend_df,
        top_model_names,
        val_matrix,
        y_val,
        n_rounds=5000,
        noise_scale=0.06,
        random_state=42,
        fpr_limit=0.097
):
    rng = np.random.default_rng(random_state)

    best_row = advanced_blend_df.iloc[0]
    best_w = np.array([best_row[f"w_{name}"] for name in top_model_names], dtype=float)

    refined_results = []

    for i in range(n_rounds):
        noise = rng.normal(0, noise_scale, size=len(best_w))
        w = best_w + noise
        w = np.clip(w, 0, None)

        if w.sum() == 0:
            continue

        w = w / w.sum()

        score = val_matrix @ w
        metrics = best_tpr_under_fpr(y_val, score, fpr_limit=fpr_limit)

        refined_results.append({
            "stage": "local_refine",
            "auc": metrics["auc"],
            "fpr": metrics["fpr"],
            "tpr": metrics["tpr"],
            "threshold": metrics["threshold"],
            **{f"w_{top_model_names[j]}": w[j] for j in range(len(top_model_names))}
        })

    refined_df = pd.DataFrame(refined_results)

    combined_df = pd.concat(
        [advanced_blend_df, refined_df],
        ignore_index=True
    )

    combined_df = combined_df.sort_values(
        ["tpr", "auc"],
        ascending=False
    ).reset_index(drop=True)

    return combined_df


advanced_blend_refined_df = local_refine_blend(
    advanced_blend_df=advanced_blend_df,
    top_model_names=top_model_names,
    val_matrix=val_matrix,
    y_val=y_val,
    n_rounds=5000,
    noise_scale=0.04,
    random_state=RANDOM_STATE + 999,
    fpr_limit=FPR_LIMIT_FINAL
)

advanced_refined_path = os.path.join(OUTPUT_DIR, "super_advanced_blend_refined_results.csv")
advanced_blend_refined_df.to_csv(advanced_refined_path, index=False)

print("\nTop 20 refined blend results:")
print(advanced_blend_refined_df.head(20))
print("Saved:", advanced_refined_path)

## Export the final super blend submission
This section selects the best refined blend, applies its optimized model weights to validation and test predictions, and evaluates the final validation performance under the FPR constraint. It then converts test scores into binary predictions using the selected threshold and exports the final submission file. The code also saves detailed test scores, prints the selected model weights, compares the new validation result with the previous best submission, and recommends whether to use the new submission or keep the old one.

In [ ]:
# ============================================================
# 23. Export Final Super Blend Submission
# ============================================================

best_adv = advanced_blend_refined_df.iloc[0]

best_weights = np.array([
    best_adv[f"w_{name}"] for name in top_model_names
])

best_threshold = best_adv["threshold"]

super_val_score = val_matrix @ best_weights
super_test_score = test_matrix @ best_weights

super_metrics = best_tpr_under_fpr(
    y_val,
    super_val_score,
    fpr_limit=FPR_LIMIT_FINAL
)

print_metrics("SUPER ADVANCED BLEND VALIDATION METRICS", super_metrics)

super_pred = (super_test_score >= best_threshold).astype(int)

super_submission = pd.DataFrame({
    "top_useful": super_pred
})

super_submission_path = os.path.join(
    OUTPUT_DIR,
    "group_7_yelp.csv"
)

super_submission.to_csv(
    super_submission_path,
    index=False,
    header=False
)

super_score_path = os.path.join(
    OUTPUT_DIR,
    "super_advanced_blend_test_scores.csv"
)

pd.DataFrame({
    "final_score": super_test_score,
    "prediction": super_pred
}).to_csv(super_score_path, index=False)

print("\n" + "=" * 80)
print("SUPER ADVANCED BLEND SAVED")
print("=" * 80)

print("Submission:", super_submission_path)
print("Scores:", super_score_path)

print("\nSelected models and weights:")
for name, w in zip(top_model_names, best_weights):
    print(f"{name:35s}: {w:.6f}")

print("\nSuper validation result:")
print("AUC:", super_metrics["auc"])
print("FPR:", super_metrics["fpr"])
print("TPR:", super_metrics["tpr"])
print("Threshold:", best_threshold)

print("\nPositive predictions:", super_pred.sum())
print("Positive ratio:", super_pred.mean())

print("\nPrevious best:")
print("FPR:", CURRENT_BEST_FPR)
print("TPR:", CURRENT_BEST_TPR)

if super_metrics["fpr"] <= FPR_LIMIT and super_metrics["tpr"] > CURRENT_BEST_TPR:
    print("\nUSE NEW SUBMISSION:")
    print(super_submission_path)
else:
    print("\nKEEP OLD SUBMISSION unless leaderboard confirms improvement.")
    print("New validation did not beat previous TPR safely.")

## Save final performance summary tables
This section stores the key results of the final super blend into a summary table for documentation and reporting. It records previous benchmark performance, final validation metrics (FPR, TPR, AUC, threshold), and prediction distribution statistics such as total positive predictions and positive ratio. The summary is then saved as a final reference file for easy comparison and project reporting.

In [ ]:
# ============================================================
# 24. Save Main Final Tables
# ============================================================

final_summary = {
    "previous_best_fpr": CURRENT_BEST_FPR,
    "previous_best_tpr": CURRENT_BEST_TPR,
    "super_fpr": super_metrics["fpr"],
    "super_tpr": super_metrics["tpr"],
    "super_auc": super_metrics["auc"],
    "super_threshold": best_threshold,
    "positive_predictions": int(super_pred.sum()),
    "positive_ratio": float(super_pred.mean())
}

final_summary_path = os.path.join(OUTPUT_DIR, "super_final_summary.csv")
pd.DataFrame([final_summary]).to_csv(final_summary_path, index=False)

print("Saved final summary:", final_summary_path)
print(pd.DataFrame([final_summary]))

## Compare 6+ different model types for report analysis
This section evaluates a broad set of machine learning models for reporting purposes, including Logistic Regression, Decision Tree, Random Forest, AdaBoost, Naive Bayes, MLP Neural Network, LightGBM, and XGBoost. Using sampled training and validation subsets for efficiency, each model is trained and evaluated under the same FPR constraint to compare AUC, TPR, threshold, accuracy, and confusion matrix metrics.

In [ ]:
# ============================================================
# 25. Report Block: Compare 6+ Different Model Types
# ============================================================

if RUN_REPORT_BLOCKS:

    compare_size = min(MODEL_COMPARE_SAMPLE_SIZE, len(X_train))

    X_cmp_train, _, y_cmp_train, _ = train_test_split(
        X_train,
        y_train,
        train_size=compare_size,
        random_state=RANDOM_STATE,
        stratify=y_train
    )

    val_size = min(140000, len(X_val))

    X_cmp_val, _, y_cmp_val, _ = train_test_split(
        X_val,
        y_val,
        train_size=val_size,
        random_state=RANDOM_STATE,
        stratify=y_val
    )

    print("Model comparison train sample:", X_cmp_train.shape)
    print("Model comparison val sample:", X_cmp_val.shape)

    compare_models = {
        "Logistic Regression": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(
                C=0.5,
                penalty="l2",
                class_weight="balanced",
                solver="saga",
                max_iter=500,
                n_jobs=N_JOBS,
                random_state=RANDOM_STATE
            ))
        ]),

        "Decision Tree": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", DecisionTreeClassifier(
                max_depth=12,
                min_samples_leaf=80,
                class_weight="balanced",
                random_state=RANDOM_STATE
            ))
        ]),

        "Random Forest": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", RandomForestClassifier(
                n_estimators=300,
                max_depth=24,
                min_samples_leaf=20,
                max_features="sqrt",
                class_weight="balanced_subsample",
                n_jobs=N_JOBS,
                random_state=RANDOM_STATE
            ))
        ]),

        "AdaBoost": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", AdaBoostClassifier(
                estimator=DecisionTreeClassifier(
                    max_depth=3,
                    min_samples_leaf=80,
                    class_weight="balanced",
                    random_state=RANDOM_STATE
                ),
                n_estimators=220,
                learning_rate=0.06,
                random_state=RANDOM_STATE
            ))
        ]),

        "Naive Bayes": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", GaussianNB())
        ]),

        "MLP Neural Network": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", MLPClassifier(
                hidden_layer_sizes=(128, 64),
                activation="relu",
                alpha=0.001,
                learning_rate_init=0.0005,
                max_iter=90,
                early_stopping=True,
                validation_fraction=0.10,
                random_state=99
            ))
        ]),

        "LightGBM": lgb.LGBMClassifier(
            objective="binary",
            n_estimators=700,
            learning_rate=0.04,
            num_leaves=63,
            min_child_samples=100,
            subsample=0.85,
            colsample_bytree=0.85,
            scale_pos_weight=scale_pos_weight,
            random_state=RANDOM_STATE,
            n_jobs=-1
        ),

        "XGBoost": xgb.XGBClassifier(
            objective="binary:logistic",
            eval_metric="auc",
            n_estimators=700,
            learning_rate=0.04,
            max_depth=6,
            min_child_weight=100,
            subsample=0.85,
            colsample_bytree=0.85,
            scale_pos_weight=scale_pos_weight,
            tree_method="hist",
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
    }

    compare_results = []

    for name, model in compare_models.items():
        print("\nTraining model for comparison:", name)

        model.fit(X_cmp_train, y_cmp_train)
        val_score = get_model_score(model, X_cmp_val)

        metrics = best_tpr_under_fpr(
            y_cmp_val,
            val_score,
            fpr_limit=FPR_LIMIT_FINAL
        )

        metrics["model"] = name
        compare_results.append(metrics)

        print(name, "AUC:", metrics["auc"], "FPR:", metrics["fpr"], "TPR:", metrics["tpr"])

        gc.collect()

    compare_results_df = pd.DataFrame(compare_results)
    compare_results_df = compare_results_df[
        ["model", "auc", "fpr", "tpr", "threshold", "accuracy", "tn", "fp", "fn", "tp"]
    ].sort_values(["tpr", "auc"], ascending=False)

    compare_path = os.path.join(OUTPUT_DIR, "super_six_plus_model_comparison.csv")
    compare_results_df.to_csv(compare_path, index=False)

    print("\nSix-plus model comparison:")
    print(compare_results_df)
    print("Saved:", compare_path)

    plt.figure(figsize=(10, 6))
    plot_df = compare_results_df.sort_values("tpr")
    plt.barh(plot_df["model"], plot_df["tpr"])
    plt.title("Model Comparison: TPR under FPR <= 10%")
    plt.xlabel("TPR")
    plt.ylabel("Model")
    plt.tight_layout()
    fig_path = os.path.join(OUTPUT_DIR, "super_six_plus_model_comparison_tpr.png")
    plt.savefig(fig_path, dpi=200)
    plt.show()

    print("Saved chart:", fig_path)

## Run 5-fold cross-validation for LightGBM
This section evaluates the stability of the LightGBM model using 5-fold stratified cross-validation. It samples the training data for efficiency, splits it into five folds while preserving the class distribution, and trains one LightGBM model per fold. For each fold, it records AUC, FPR, TPR, threshold, confusion matrix results, and the best iteration from early stopping. The final CV results are saved and summarized to show the model’s average performance and consistency across folds.

In [ ]:
# ============================================================
# 26. Report Block: 5-Fold CV for LightGBM
# ============================================================

if RUN_REPORT_BLOCKS:

    cv_size = min(CURVE_SAMPLE_SIZE, len(X_train))

    X_cv_pool, _, y_cv_pool, _ = train_test_split(
        X_train,
        y_train,
        train_size=cv_size,
        random_state=RANDOM_STATE,
        stratify=y_train
    )

    print("CV sample:", X_cv_pool.shape)

    skf = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=RANDOM_STATE
    )

    cv_results = []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_cv_pool, y_cv_pool), start=1):
        print("\nCV Fold:", fold)

        X_tr = X_cv_pool.iloc[tr_idx]
        X_va = X_cv_pool.iloc[va_idx]
        y_tr = y_cv_pool.iloc[tr_idx]
        y_va = y_cv_pool.iloc[va_idx]

        n_pos_fold = (y_tr == 1).sum()
        n_neg_fold = (y_tr == 0).sum()
        spw_fold = n_neg_fold / n_pos_fold

        cv_model = lgb.LGBMClassifier(
            objective="binary",
            n_estimators=700,
            learning_rate=0.04,
            num_leaves=63,
            min_child_samples=100,
            subsample=0.85,
            colsample_bytree=0.85,
            scale_pos_weight=spw_fold,
            random_state=RANDOM_STATE + fold,
            n_jobs=-1
        )

        cv_model.fit(
            X_tr,
            y_tr,
            eval_set=[(X_va, y_va)],
            eval_metric="auc",
            callbacks=[
                lgb.early_stopping(70),
                lgb.log_evaluation(0)
            ]
        )

        val_score = cv_model.predict_proba(X_va)[:, 1]
        metrics = best_tpr_under_fpr(y_va, val_score, fpr_limit=FPR_LIMIT_FINAL)
        metrics["fold"] = fold
        metrics["best_iteration"] = cv_model.best_iteration_

        cv_results.append(metrics)

        print("Fold", fold, "AUC:", metrics["auc"], "FPR:", metrics["fpr"], "TPR:", metrics["tpr"])

        gc.collect()

    cv_results_df = pd.DataFrame(cv_results)
    cv_path = os.path.join(OUTPUT_DIR, "super_lightgbm_5fold_cv_results.csv")
    cv_results_df.to_csv(cv_path, index=False)

    print("\n5-fold CV results:")
    print(cv_results_df)
    print("\nCV mean:")
    print(cv_results_df[["auc", "fpr", "tpr"]].mean())
    print("Saved:", cv_path)

## Generate a safe learning curve for LightGBM
This section evaluates how LightGBM performance changes as the training sample size increases. It trains models using different fractions of the training data, from 10% to 100%, while safely handling cases where the full dataset is used. For each fraction, it records AUC, FPR, TPR, threshold, and best iteration.

In [ ]:
# ============================================================
# 27. Report Block: Learning Curve
# ============================================================

if RUN_REPORT_BLOCKS:

    lc_size = min(CURVE_SAMPLE_SIZE, len(X_train))

    # If lc_size equals the full X_train, use the entire dataset directly without train_test_split
    if lc_size >= len(X_train):
        X_lc_pool = X_train.copy()
        y_lc_pool = y_train.copy()
    else:
        X_lc_pool, _, y_lc_pool, _ = train_test_split(
            X_train,
            y_train,
            train_size=lc_size,
            random_state=RANDOM_STATE,
            stratify=y_train
        )

    learning_fracs = [0.10, 0.30, 0.50, 0.70, 1.00]
    learning_results = []

    for frac in learning_fracs:
        n_sample = int(len(X_lc_pool) * frac)

        # Key fix: If frac = 1.00, use the entire dataset directly
        if n_sample >= len(X_lc_pool):
            X_lc_train = X_lc_pool.copy()
            y_lc_train = y_lc_pool.copy()
        else:
            X_lc_train, _, y_lc_train, _ = train_test_split(
                X_lc_pool,
                y_lc_pool,
                train_size=n_sample,
                random_state=RANDOM_STATE,
                stratify=y_lc_pool
            )

        n_pos_lc = (y_lc_train == 1).sum()
        n_neg_lc = (y_lc_train == 0).sum()
        spw_lc = n_neg_lc / n_pos_lc

        print("\nLearning curve fraction:", frac)
        print("Training sample size:", len(X_lc_train))

        lc_model = lgb.LGBMClassifier(
            objective="binary",
            n_estimators=700,
            learning_rate=0.04,
            num_leaves=63,
            min_child_samples=100,
            subsample=0.85,
            colsample_bytree=0.85,
            scale_pos_weight=spw_lc,
            random_state=RANDOM_STATE,
            n_jobs=-1
        )

        lc_model.fit(
            X_lc_train,
            y_lc_train,
            eval_set=[(X_val, y_val)],
            eval_metric="auc",
            callbacks=[
                lgb.early_stopping(70),
                lgb.log_evaluation(0)
            ]
        )

        val_score = lc_model.predict_proba(X_val)[:, 1]

        metrics = best_tpr_under_fpr(
            y_val,
            val_score,
            fpr_limit=FPR_LIMIT_FINAL
        )

        learning_results.append({
            "training_fraction": frac,
            "training_rows": len(X_lc_train),
            "auc": metrics["auc"],
            "fpr": metrics["fpr"],
            "tpr": metrics["tpr"],
            "threshold": metrics["threshold"],
            "best_iteration": lc_model.best_iteration_
        })

        print("AUC:", metrics["auc"])
        print("FPR:", metrics["fpr"])
        print("TPR:", metrics["tpr"])

        gc.collect()

    learning_results_df = pd.DataFrame(learning_results)

    learning_path = os.path.join(
        OUTPUT_DIR,
        "super_learning_curve_results.csv"
    )

    learning_results_df.to_csv(learning_path, index=False)

    print("\nLearning curve results:")
    print(learning_results_df)
    print("Saved:", learning_path)

    plt.figure(figsize=(8, 5))

    plt.plot(
        learning_results_df["training_rows"],
        learning_results_df["auc"],
        marker="o",
        label="AUC"
    )

    plt.plot(
        learning_results_df["training_rows"],
        learning_results_df["tpr"],
        marker="o",
        label="TPR at FPR <= 10%"
    )

    plt.title("Learning Curve: LightGBM")
    plt.xlabel("Training Rows")
    plt.ylabel("Validation Performance")
    plt.legend()
    plt.tight_layout()

    fig_path = os.path.join(
        OUTPUT_DIR,
        "super_learning_curve_lgb.png"
    )

    plt.savefig(fig_path, dpi=200)
    plt.show()

    print("Saved chart:", fig_path)

## Generate a LightGBM tuning curve for model complexity
This section evaluates how different num_leaves values affect LightGBM performance. It tests several levels of tree complexity, records training AUC and validation AUC, and also saves validation FPR, TPR, and best iteration for each setting. The tuning curve helps compare underfitting and overfitting patterns and supports the final hyperparameter choice for the report.

In [ ]:
# ============================================================
# 28. Report Block: Tuning / Fitting Curve
# ============================================================

if RUN_REPORT_BLOCKS:

    tune_size = min(CURVE_SAMPLE_SIZE, len(X_train))

    X_tune_pool, _, y_tune_pool, _ = train_test_split(
        X_train,
        y_train,
        train_size=tune_size,
        random_state=RANDOM_STATE,
        stratify=y_train
    )

    X_tune_train, X_tune_val, y_tune_train, y_tune_val = train_test_split(
        X_tune_pool,
        y_tune_pool,
        test_size=0.25,
        random_state=RANDOM_STATE,
        stratify=y_tune_pool
    )

    num_leaves_grid = [15, 31, 63, 127, 255]
    tuning_results = []

    for nl in num_leaves_grid:
        print("\nTuning num_leaves:", nl)

        n_pos_tune = (y_tune_train == 1).sum()
        n_neg_tune = (y_tune_train == 0).sum()
        spw_tune = n_neg_tune / n_pos_tune

        tune_model = lgb.LGBMClassifier(
            objective="binary",
            n_estimators=800,
            learning_rate=0.04,
            num_leaves=nl,
            min_child_samples=100,
            subsample=0.85,
            colsample_bytree=0.85,
            scale_pos_weight=spw_tune,
            random_state=RANDOM_STATE,
            n_jobs=-1
        )

        tune_model.fit(
            X_tune_train,
            y_tune_train,
            eval_set=[(X_tune_val, y_tune_val)],
            eval_metric="auc",
            callbacks=[
                lgb.early_stopping(70),
                lgb.log_evaluation(0)
            ]
        )

        train_score = tune_model.predict_proba(X_tune_train)[:, 1]
        val_score = tune_model.predict_proba(X_tune_val)[:, 1]

        train_auc = roc_auc_score(y_tune_train, train_score)
        val_metrics = best_tpr_under_fpr(y_tune_val, val_score, fpr_limit=FPR_LIMIT_FINAL)

        tuning_results.append({
            "num_leaves": nl,
            "train_auc": train_auc,
            "val_auc": val_metrics["auc"],
            "val_fpr": val_metrics["fpr"],
            "val_tpr": val_metrics["tpr"],
            "best_iteration": tune_model.best_iteration_
        })

        print("train_auc:", train_auc, "val_auc:", val_metrics["auc"], "val_tpr:", val_metrics["tpr"])

        gc.collect()

    tuning_results_df = pd.DataFrame(tuning_results)
    tuning_path = os.path.join(OUTPUT_DIR, "super_lgb_num_leaves_tuning_curve.csv")
    tuning_results_df.to_csv(tuning_path, index=False)

    print("\nTuning curve results:")
    print(tuning_results_df)
    print("Saved:", tuning_path)

    plt.figure(figsize=(8, 5))
    plt.plot(
        tuning_results_df["num_leaves"],
        tuning_results_df["train_auc"],
        marker="o",
        label="Train AUC"
    )
    plt.plot(
        tuning_results_df["num_leaves"],
        tuning_results_df["val_auc"],
        marker="o",
        label="Validation AUC"
    )
    plt.title("Tuning Curve: LightGBM num_leaves")
    plt.xlabel("num_leaves")
    plt.ylabel("AUC")
    plt.legend()
    plt.tight_layout()
    fig_path = os.path.join(OUTPUT_DIR, "super_lgb_num_leaves_tuning_curve.png")
    plt.savefig(fig_path, dpi=200)
    plt.show()

    print("Saved chart:", fig_path)

## Print final FPR and TPR

In [ ]:
print("FPR:", super_metrics["fpr"])
print("TPR:", super_metrics["tpr"])

In [ ]:

# Sanity checks for final blended model alignment and threshold rule

print("Final blend validation score shape:", np.asarray(super_val_score).shape)
print("Validation label shape:", np.asarray(y_val).shape)
print("Final blend test score shape:", np.asarray(super_test_score).shape)

assert len(super_val_score) == len(y_val), "Blend validation predictions do not align with y_val."
assert len(super_test_score) == len(X_test_final), "Blend test predictions do not align with X_test_final."
assert np.isclose(best_weights.sum(), 1.0, atol=1e-6), "Blend weights do not sum to 1."
assert super_metrics["fpr"] <= FPR_LIMIT_FINAL + 1e-12, "Final threshold does not satisfy FPR limit."

print("Blend weights sum:", best_weights.sum())
print("Threshold selected under FPR <=:", FPR_LIMIT_FINAL)
print("Final blended model FPR:", super_metrics["fpr"])
print("Final blended model TPR:", super_metrics["tpr"])


## 5. Model 2: Logistic Regression with Business Cleaned Dataset

This baseline uses logistic regression with business-cleaned features. It is included for comparison and reporting, not as the final submission model. ROC/AUC reporting is based on a held-out validation split.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.metrics import roc_curve, roc_auc_score, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Load the datasets
print("Loading datasets...")
review_train_x = pd.read_csv(csv_path(TRAIN_X_FILE))
review_test_x = pd.read_csv(csv_path(TEST_X_FILE))
review_train_y = pd.read_csv(csv_path(TRAIN_Y_FILE))
business_data = pd.read_csv(csv_path(BUSINESS_CLEAN_FILE, [BUSINESS_FILE]))

print(f"review_train_x shape: {review_train_x.shape}")
print(f"review_test_x shape: {review_test_x.shape}")
print(f"review_train_y shape: {review_train_y.shape}")
print(f"business_data shape: {business_data.shape}")
print("\nDatasets loaded successfully!")

In [ ]:
# Step 1: Add label column to distinguish train and test data
print("Adding data source labels...")
review_train_x['data_source'] = 'train'
review_test_x['data_source'] = 'test'

print(f"review_train_x with label - shape: {review_train_x.shape}")
print(f"review_test_x with label - shape: {review_test_x.shape}")

# Step 2: Concatenate train and test data vertically
print("\nConcatenating train and test data...")
combined_reviews = pd.concat([review_train_x, review_test_x], axis=0, ignore_index=True)
print(f"Combined reviews shape: {combined_reviews.shape}")
print(f"Data source distribution:\n{combined_reviews['data_source'].value_counts()}")

# Step 3: Merge combined data with business_data on business_id
print("\nMerging with business_data...")
merged_data = combined_reviews.merge(business_data, on='business_id', how='inner')
print(f"Merged data shape after business join: {merged_data.shape}")
print(f"Data source distribution after merge:\n{merged_data['data_source'].value_counts()}")

In [ ]:
# Step 4: Resplit data back into train and test, and drop the label column
print("Splitting data back into train and test sets...")
train_data = merged_data[merged_data['data_source'] == 'train'].copy()
test_data = merged_data[merged_data['data_source'] == 'test'].copy()

# Drop the data_source column
train_data = train_data.drop('data_source', axis=1)
test_data = test_data.drop('data_source', axis=1)

print(f"Train data shape: {train_data.shape}")
print(f"Test data shape: {test_data.shape}")

# Step 5: Select features from business_data
print("\nPreparing features...")
business_features = business_data.select_dtypes(include=[np.number]).columns.tolist()
if 'is_open' in business_features:
    business_features.remove('is_open')

# Select features that exist in train/test data
feature_cols = [col for col in business_features if col in train_data.columns]
print(f"Number of features: {len(feature_cols)}")
print(f"Feature columns: {feature_cols}")

# Extract features
X_train_full = train_data[feature_cols].fillna(0)
X_test_full = test_data[feature_cols].fillna(0)

print(f"\nX_train_full shape: {X_train_full.shape}")
print(f"X_test_full shape: {X_test_full.shape}")

In [ ]:
# Step 6: Align target variable safely
print("Preparing target variable with review_id alignment when available...")

if "review_id" in train_data.columns and "review_id" in review_train_y.columns:
    target_cols = ["review_id", "top_useful"]
    train_targets = review_train_y[target_cols].copy()
    before_rows = len(train_data)
    train_data = train_data.merge(train_targets, on="review_id", how="left", validate="one_to_one")
    assert len(train_data) == before_rows, "Target merge changed train_data row count."
    assert train_data["top_useful"].notna().all(), "Some merged training rows are missing top_useful labels."
    y_train_full = train_data["top_useful"].astype(int).values
    # Rebuild X after target merge to preserve exact train_data row order.
    X_train_full = train_data[feature_cols].fillna(0)
else:
    assert len(review_train_y) == len(X_train_full), (
        "review_train_y and X_train_full lengths differ. Avoid slicing because it can silently misalign labels."
    )
    y_train_full = review_train_y["top_useful"].values

print(f"y_train_full shape: {y_train_full.shape}")
print("Target variable distribution for training data:")
print(f"Class 0: {(y_train_full == 0).sum()} ({(y_train_full == 0).sum() / len(y_train_full) * 100:.2f}%)")
print(f"Class 1: {(y_train_full == 1).sum()} ({(y_train_full == 1).sum() / len(y_train_full) * 100:.2f}%)")

print(f"\nX_train_full: {X_train_full.shape}")
print(f"X_test_full: {X_test_full.shape}")
print(f"y_train_full: {y_train_full.shape}")
assert len(X_train_full) == len(y_train_full), "Features and target are not aligned."
print("\nData preparation complete!")


In [ ]:
# Step 7: Train Logistic Regression with a held-out validation split
print("Training Logistic Regression Model with validation split...")

X_train_logreg_business, X_val_logreg_business, y_train_logreg_business, y_val_logreg_business = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.20,
    random_state=42,
    stratify=y_train_full
)

log_reg_val = LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1, verbose=1)
log_reg_val.fit(X_train_logreg_business, y_train_logreg_business)

# Validation predictions for honest comparison
logreg_business_val_probs = log_reg_val.predict_proba(X_val_logreg_business)[:, 1]
logreg_business_val_pred = (logreg_business_val_probs >= 0.5).astype(int)
val_score = log_reg_val.score(X_val_logreg_business, y_val_logreg_business)
train_score = log_reg_val.score(X_train_logreg_business, y_train_logreg_business)

# Final model for test predictions is still trained on all available training rows.
log_reg = LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1, verbose=1)
log_reg.fit(X_train_full, y_train_full)
y_test_pred_proba = log_reg.predict_proba(X_test_full)[:, 1]
y_test_pred = log_reg.predict(X_test_full)

print("\n" + "=" * 60)
print("MODEL 2 LOGISTIC REGRESSION PERFORMANCE")
print("=" * 60)
print(f"Train Accuracy on split-train: {train_score:.4f}")
print(f"Validation Accuracy: {val_score:.4f}")
print(f"Validation prediction shape: {logreg_business_val_probs.shape}")

print("\n" + "=" * 60)
print("TEST SET PREDICTIONS")
print("=" * 60)
print("Prediction distribution:")
print(f"Class 0 predictions: {(y_test_pred == 0).sum()} ({(y_test_pred == 0).sum() / len(y_test_pred) * 100:.2f}%)")
print(f"Class 1 predictions: {(y_test_pred == 1).sum()} ({(y_test_pred == 1).sum() / len(y_test_pred) * 100:.2f}%)")
print("\nProbability distribution for Class 1:")
print(f"Mean probability: {y_test_pred_proba.mean():.4f}")
print(f"Std probability: {y_test_pred_proba.std():.4f}")
print("=" * 60)


In [ ]:
# Step 8: Calculate ROC curve and TPR at 10% FPR on held-out validation data
print("Computing ROC Curve on held-out validation data...")

fpr, tpr, thresholds = roc_curve(y_val_logreg_business, logreg_business_val_probs)
roc_auc = roc_auc_score(y_val_logreg_business, logreg_business_val_probs)

# Find the best TPR among thresholds satisfying FPR <= 10%.
fpr_target = 0.10
valid_idx = np.where(fpr <= fpr_target)[0]
if len(valid_idx) == 0:
    idx = int(np.argmin(fpr))
else:
    idx = valid_idx[np.argmax(tpr[valid_idx])]

tpr_at_fpr = tpr[idx]
actual_fpr = fpr[idx]
threshold_at_fpr = thresholds[idx]

print("\n" + "=" * 60)
print("ROC CURVE ANALYSIS - TPR AT FPR <= 10% (Validation Data)")
print("=" * 60)
print(f"Target FPR: {fpr_target:.4f}")
print(f"Actual FPR: {actual_fpr:.4f}")
print(f"TPR at FPR <= 10%: {tpr_at_fpr:.4f}")
print(f"Threshold: {threshold_at_fpr:.4f}")
print(f"ROC-AUC Score: {roc_auc:.4f}")
print("=" * 60)


In [ ]:
# Step 9: Plot ROC Curve
print("Plotting ROC Curve...")
fig, ax = plt.subplots(figsize=(10, 7))
ax.plot(fpr, tpr, 'b-', label=f'ROC Curve (AUC={roc_auc:.4f})', linewidth=2)
ax.plot([0, 1], [0, 1], 'k--', label='Random Classifier', linewidth=1)

# Highlight the point at 10% FPR
ax.plot(actual_fpr, tpr_at_fpr, 'ro', markersize=10, label=f'Point at FPR <= 10% (TPR={tpr_at_fpr:.4f})')

ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curve - Logistic Regression (Validation Data)', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1])
plt.tight_layout()
plt.show()

print("ROC Curve plotted successfully!")

In [ ]:
# Step 10: Generate Learning Curve
print("Generating Learning Curve (this may take a few minutes)...")
print("Note: Using sample sizes for efficiency on large dataset")

# Use a smaller subset for learning curve if dataset is too large
sample_size = min(50000, len(X_train_full))
sample_idx = np.random.choice(len(X_train_full), sample_size, replace=False)
X_sample = X_train_full.iloc[sample_idx]
y_sample = y_train_full[sample_idx]

train_sizes, train_scores, val_scores = learning_curve(
    LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1),
    X_sample, y_sample,
    cv=5,
    n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 10),
    verbose=1,
    scoring='accuracy'
)

train_mean = np.mean(train_scores, axis=1)
train_std = np.std(train_scores, axis=1)
val_mean = np.mean(val_scores, axis=1)
val_std = np.std(val_scores, axis=1)

print("Learning curve completed!")

# Plot Learning Curve
fig, ax = plt.subplots(figsize=(10, 7))
ax.plot(train_sizes, train_mean, 'o-', color='r', label='Training Score', linewidth=2, markersize=6)
ax.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.1, color='r')

ax.plot(train_sizes, val_mean, 'o-', color='g', label='Validation Score', linewidth=2, markersize=6)
ax.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.1, color='g')

ax.set_xlabel('Training Set Size', fontsize=12)
ax.set_ylabel('Accuracy Score', fontsize=12)
ax.set_title('Learning Curve - Logistic Regression', fontsize=14, fontweight='bold')
ax.legend(loc='best', fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Learning curve plotted successfully!")

## 5b. Decision Tree Baseline 

This is a interpretable baseline model. Its ROC/AUC evaluation uses held-out validation predictions so the metric is not inflated by training-set memorization.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import learning_curve
from sklearn.metrics import roc_curve, roc_auc_score, accuracy_score, confusion_matrix
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Load the datasets
print("Loading datasets...")
review_train_x = pd.read_csv(csv_path(TRAIN_X_FILE))
review_test_x = pd.read_csv(csv_path(TEST_X_FILE))
review_train_y = pd.read_csv(csv_path(TRAIN_Y_FILE))
business_data = pd.read_csv(csv_path(BUSINESS_CLEAN_FILE, [BUSINESS_FILE]))

print(f"review_train_x shape: {review_train_x.shape}")
print(f"review_test_x shape: {review_test_x.shape}")
print(f"review_train_y shape: {review_train_y.shape}")
print(f"business_data shape: {business_data.shape}")
print("\nDatasets loaded successfully!")

In [ ]:
# Step 1: Add label column to distinguish train and test data
print("Adding data source labels...")
review_train_x['data_source'] = 'train'
review_test_x['data_source'] = 'test'

print(f"review_train_x with label - shape: {review_train_x.shape}")
print(f"review_test_x with label - shape: {review_test_x.shape}")

# Step 2: Concatenate train and test data vertically
print("\nConcatenating train and test data...")
combined_reviews = pd.concat([review_train_x, review_test_x], axis=0, ignore_index=True)
del review_train_x, review_test_x  # Free memory
print(f"Combined reviews shape: {combined_reviews.shape}")
print(f"Data source distribution:\n{combined_reviews['data_source'].value_counts()}")

# Step 3: Select only numeric columns from business_data to reduce memory
print("\nPreparing business features...")
numeric_cols = ['business_id'] + business_data.select_dtypes(include=[np.number]).columns.tolist()
business_data_numeric = business_data[numeric_cols].copy()
print(f"Business data numeric columns selected: {len(numeric_cols)}")

# Merge combined data with business_data on business_id
print("\nMerging with business_data (numeric columns only)...")
merged_data = combined_reviews.merge(business_data_numeric, on='business_id', how='inner')
del combined_reviews, business_data_numeric  # Free memory
print(f"Merged data shape after business join: {merged_data.shape}")
print(f"Data source distribution after merge:\n{merged_data['data_source'].value_counts()}")

In [ ]:
# Step 4: Resplit data back into train and test, and drop the label column
print("Splitting data back into train and test sets...")
train_data = merged_data[merged_data['data_source'] == 'train'].copy()
test_data = merged_data[merged_data['data_source'] == 'test'].copy()

# Drop the data_source column
train_data = train_data.drop('data_source', axis=1)
test_data = test_data.drop('data_source', axis=1)

print(f"Train data shape: {train_data.shape}")
print(f"Test data shape: {test_data.shape}")

# Step 5: Select features from business_data
print("\nPreparing features...")
business_features = business_data.select_dtypes(include=[np.number]).columns.tolist()
if 'is_open' in business_features:
    business_features.remove('is_open')

# Select features that exist in train/test data
feature_cols = [col for col in business_features if col in train_data.columns]
print(f"Number of features: {len(feature_cols)}")
print(f"Feature columns: {feature_cols}")

# Extract features
X_train_full = train_data[feature_cols].fillna(0)
X_test_full = test_data[feature_cols].fillna(0)

print(f"\nX_train_full shape: {X_train_full.shape}")
print(f"X_test_full shape: {X_test_full.shape}")

In [ ]:
# Step 6: Align target variable safely
print("Preparing target variable with review_id alignment when available...")

if "review_id" in train_data.columns and "review_id" in review_train_y.columns:
    target_cols = ["review_id", "top_useful"]
    train_targets = review_train_y[target_cols].copy()
    before_rows = len(train_data)
    train_data = train_data.merge(train_targets, on="review_id", how="left", validate="one_to_one")
    assert len(train_data) == before_rows, "Target merge changed train_data row count."
    assert train_data["top_useful"].notna().all(), "Some merged training rows are missing top_useful labels."
    y_train_full = train_data["top_useful"].astype(int).values
    X_train_full = train_data[feature_cols].fillna(0)
else:
    assert len(review_train_y) == len(X_train_full), (
        "review_train_y and X_train_full lengths differ. Avoid slicing because it can silently misalign labels."
    )
    y_train_full = review_train_y["top_useful"].values

print(f"y_train_full shape: {y_train_full.shape}")
print("Target variable distribution for training data:")
print(f"Class 0: {(y_train_full == 0).sum()} ({(y_train_full == 0).sum() / len(y_train_full) * 100:.2f}%)")
print(f"Class 1: {(y_train_full == 1).sum()} ({(y_train_full == 1).sum() / len(y_train_full) * 100:.2f}%)")

print(f"\nX_train_full: {X_train_full.shape}")
print(f"X_test_full: {X_test_full.shape}")
print(f"y_train_full: {y_train_full.shape}")
assert len(X_train_full) == len(y_train_full), "Features and target are not aligned."
print("\nData preparation complete!")


In [ ]:
# Step 7: Decision trees do not require feature scaling
print("Preparing decision tree inputs...")
X_train_tree = X_train_full
X_test_tree = X_test_full

print(f"X_train_tree shape: {X_train_tree.shape}")
print(f"X_test_tree shape: {X_test_tree.shape}")
print("\nDecision tree input preparation complete!")

In [ ]:
# Step 8: Train Decision Tree with a held-out validation split
print("Training Decision Tree Model with validation split...")
print("This may take a few minutes...\n")

X_train_tree_split, X_val_tree, y_train_tree_split, y_val_tree = train_test_split(
    X_train_tree,
    y_train_full,
    test_size=0.20,
    random_state=42,
    stratify=y_train_full
)

dt_model_val = DecisionTreeClassifier(random_state=42, max_depth=None, min_samples_split=10)
dt_model_val.fit(X_train_tree_split, y_train_tree_split)

y_val_tree_pred_proba = dt_model_val.predict_proba(X_val_tree)[:, 1]
y_val_tree_pred = dt_model_val.predict(X_val_tree)

train_score = dt_model_val.score(X_train_tree_split, y_train_tree_split)
val_score = dt_model_val.score(X_val_tree, y_val_tree)

# Final model for test predictions is trained on all available training rows.
dt_model = DecisionTreeClassifier(random_state=42, max_depth=None, min_samples_split=10)
dt_model.fit(X_train_tree, y_train_full)
y_test_pred_proba = dt_model.predict_proba(X_test_tree)[:, 1]
y_test_pred = dt_model.predict(X_test_tree)

print("\n" + "=" * 60)
print("DECISION TREE MODEL PERFORMANCE")
print("=" * 60)
print(f"Train Accuracy on split-train: {train_score:.4f}")
print(f"Validation Accuracy: {val_score:.4f}")
print(f"Validation prediction shape: {y_val_tree_pred_proba.shape}")

print("\n" + "=" * 60)
print("TEST SET PREDICTIONS")
print("=" * 60)
print("Prediction distribution:")
print(f"Class 0 predictions: {(y_test_pred == 0).sum()} ({(y_test_pred == 0).sum() / len(y_test_pred) * 100:.2f}%)")
print(f"Class 1 predictions: {(y_test_pred == 1).sum()} ({(y_test_pred == 1).sum() / len(y_test_pred) * 100:.2f}%)")
print("\nProbability distribution for Class 1:")
print(f"Mean probability: {y_test_pred_proba.mean():.4f}")
print(f"Std probability: {y_test_pred_proba.std():.4f}")
print("=" * 60)


In [ ]:
# Step 9: Calculate ROC curve and TPR at 10% FPR on held-out validation data
print("Computing ROC Curve on held-out validation data...")

fpr, tpr, thresholds = roc_curve(y_val_tree, y_val_tree_pred_proba)
roc_auc = roc_auc_score(y_val_tree, y_val_tree_pred_proba)

# Find the best TPR among thresholds satisfying FPR <= 10%.
fpr_target = 0.10
valid_idx = np.where(fpr <= fpr_target)[0]
if len(valid_idx) == 0:
    idx = int(np.argmin(fpr))
else:
    idx = valid_idx[np.argmax(tpr[valid_idx])]

tpr_at_fpr = tpr[idx]
actual_fpr = fpr[idx]
threshold_at_fpr = thresholds[idx]

print("\n" + "=" * 60)
print("ROC CURVE ANALYSIS - TPR AT FPR <= 10% (Validation Data)")
print("=" * 60)
print(f"Target FPR: {fpr_target:.4f}")
print(f"Actual FPR: {actual_fpr:.4f}")
print(f"TPR at FPR <= 10%: {tpr_at_fpr:.4f}")
print(f"Threshold: {threshold_at_fpr:.4f}")
print(f"ROC-AUC Score: {roc_auc:.4f}")
print("=" * 60)


In [ ]:
# Step 10: Plot ROC Curve
print("Plotting ROC Curve...")
fig, ax = plt.subplots(figsize=(10, 7))
ax.plot(fpr, tpr, 'b-', label=f'ROC Curve (AUC={roc_auc:.4f})', linewidth=2)
ax.plot([0, 1], [0, 1], 'k--', label='Random Classifier', linewidth=1)

# Highlight the point at 10% FPR
ax.plot(actual_fpr, tpr_at_fpr, 'ro', markersize=10, label=f'Point at FPR <= 10% (TPR={tpr_at_fpr:.4f})')

ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curve - Decision Tree (Validation Data)', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1])
plt.tight_layout()
plt.show()

print("ROC Curve plotted successfully!")

In [ ]:
# Step 11: Generate Learning Curve
print("Generating Learning Curve (this may take a while for Decision Tree)...")
print("Note: Using sample sizes for efficiency on large dataset\n")

# Use a smaller subset for learning curve if dataset is too large
sample_size = min(50000, len(X_train_tree))
sample_idx = np.random.choice(len(X_train_tree), sample_size, replace=False)
X_sample = X_train_tree.iloc[sample_idx]
y_sample = y_train_full[sample_idx]

train_sizes, train_scores, val_scores = learning_curve(
    DecisionTreeClassifier(random_state=42, max_depth=None, min_samples_split=10),
    X_sample, y_sample,
    cv=5,
    n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 10),
    verbose=1,
    scoring='accuracy'
)

train_mean = np.mean(train_scores, axis=1)
train_std = np.std(train_scores, axis=1)
val_mean = np.mean(val_scores, axis=1)
val_std = np.std(val_scores, axis=1)

print("Learning curve completed!")

# Plot Learning Curve
fig, ax = plt.subplots(figsize=(10, 7))
ax.plot(train_sizes, train_mean, 'o-', color='r', label='Training Score', linewidth=2, markersize=6)
ax.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.1, color='r')

ax.plot(train_sizes, val_mean, 'o-', color='g', label='Validation Score', linewidth=2, markersize=6)
ax.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.1, color='g')

ax.set_xlabel('Training Set Size', fontsize=12)
ax.set_ylabel('Accuracy Score', fontsize=12)
ax.set_title('Learning Curve - Decision Tree', fontsize=14, fontweight='bold')
ax.legend(loc='best', fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Learning curve plotted successfully!")

## 6. Model 5: Logistic Regression Without Holiday Features

This ablation model removes holiday-derived predictors. It helps isolate whether holiday features contribute meaningful predictive signal beyond the core review, business, user, and tip features.


In [ ]:
# Yelp Top-Useful Prediction
# Baseline Logistic Regression Version
#
# Includes:
# 1. External data source: US holiday calendar
# 2. 10+ feature insight tables/charts
# 3. Baseline logistic regression classifier
# 4. ROC curve and ROC-AUC
# 5. TPR at FPR <= 10%
# 6. Final submission generated from the validation-selected threshold

import os
import gc
import ast
import json
import re
import time
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, accuracy_score

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression


In [ ]:
# 1. Settings

RANDOM_STATE = 42
N_JOBS = -1

# Data files are stored in the same project folder as this notebook.
# DATA_DIR is defined in the Project Setup section above
# OUTPUT_DIR is defined in the Project Setup section above

# Required operating point: maximize TPR while keeping FPR at or below 10%.
FPR_LIMIT = 0.10

# change user.useful to False because it is leakage
USE_USER_USEFUL_FEATURES = False

print("Settings loaded.")


In [ ]:

# 2. Load Data
# Direct pd.read_csv calls are used for Kaggle compatibility.

train_x = pd.read_csv(csv_path(TRAIN_X_FILE))
train_y = pd.read_csv(csv_path(TRAIN_Y_FILE))
test_x = pd.read_csv(csv_path(TEST_X_FILE))
business = pd.read_csv(csv_path(BUSINESS_CLEAN_FILE, [BUSINESS_FILE]))
user = pd.read_csv(csv_path(USER_CLEAN_FILE, [USER_FILE]))
tip = pd.read_csv(csv_path(TIP_CLEAN_FILE, [TIP_FILE]))

if train_y.shape[1] == 1:
    y = train_y.iloc[:, 0].astype(int).reset_index(drop=True)
else:
    y = train_y["top_useful"].astype(int).reset_index(drop=True)

# Find distribution and shapes
print("train_x:", train_x.shape)
print("train_y:", train_y.shape)
print("test_x:", test_x.shape)
print("business:", business.shape)
print("user:", user.shape)
print("tip:", tip.shape)

print("\nLabel distribution:")
print(y.value_counts())
print("Positive rate:", y.mean())

assert len(train_x) == len(y), "train_x and train_y row counts do not match."


## Define evaluation functions for model performance assessment

In [ ]:
# ============================================================
# 3. Evaluation Functions
# ============================================================

# Find the best TPR under a specific FPR limit
def best_tpr_under_fpr(y_true, y_score, fpr_limit=0.10):
    fpr, tpr, thresholds = roc_curve(y_true, y_score)
    valid_idx = np.where(fpr <= fpr_limit)[0]

    if len(valid_idx) == 0:
        return {
            "auc": roc_auc_score(y_true, y_score),
            "fpr": np.nan,
            "tpr": np.nan,
            "threshold": np.nan,
            "accuracy": np.nan,
            "tn": np.nan,
            "fp": np.nan,
            "fn": np.nan,
            "tp": np.nan
        }

    best_idx = valid_idx[np.argmax(tpr[valid_idx])]
    best_threshold = thresholds[best_idx]

    y_pred = (y_score >= best_threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    return {
        "auc": roc_auc_score(y_true, y_score),
        "fpr": fp / (fp + tn),
        "tpr": tp / (tp + fn),
        "threshold": best_threshold,
        "accuracy": accuracy_score(y_true, y_pred),
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp
    }


def print_metrics(name, metrics):
    print("\n" + "=" * 80)
    print(name)
    print("=" * 80)
    for k, v in metrics.items():
        print(f"{k}: {v}")


def get_model_score(model, X_data):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X_data)[:, 1]
    return model.decision_function(X_data)


def plot_roc_with_operating_point(y_true, y_score, metrics, output_dir, filename="baseline_logreg_roc_curve.png"):
    fpr, tpr, _ = roc_curve(y_true, y_score)
    auc = roc_auc_score(y_true, y_score)

    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, label=f"Baseline Logistic Regression AUC = {auc:.4f}")
    plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random classifier")
    plt.axvline(FPR_LIMIT, linestyle=":", color="red", label="10% FPR limit")
    plt.scatter(metrics["fpr"], metrics["tpr"], color="red", zorder=5, label=f"TPR = {metrics['tpr']:.4f} at FPR = {metrics['fpr']:.4f}")
    plt.title("ROC Curve: Baseline Logistic Regression")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.xlim(0, 1)
    plt.ylim(0, 1)
    plt.legend(loc="lower right")
    plt.tight_layout()

    fig_path = os.path.join(output_dir, filename)
    plt.savefig(fig_path, dpi=200)
    plt.show()

    print("Saved ROC curve:", fig_path)
    return fig_path


## Create review-level text, sentiment, date, rating, and holiday features
This section transforms raw review data into rich review-level predictors by engineering features from multiple dimensions of each review. It extracts textual structure features such as review length, word count, sentence count, punctuation usage, capitalization patterns, and digit frequency; builds simple sentiment indicators using predefined positive and negative keyword counts; captures temporal patterns from review dates including year, month, weekday, weekend behavior, and holiday proximity; and incorporates star rating behavior through positive, negative, and extreme rating flags. Together, these features convert unstructured review text and metadata into machine-learning-ready variables that capture writing style, sentiment tone, timing effects, and rating behavior.

In [ ]:
# ============================================================
# 5. Review-Level Features
# ============================================================

def add_review_features(df):
    df = df.copy()

    if "text" not in df.columns:
        df["text"] = ""

    df["text"] = df["text"].fillna("").astype(str)

    df["text_len"] = df["text"].str.len().astype("float32")
    df["word_count"] = df["text"].str.split().str.len().astype("float32")
    df["avg_word_len"] = df["text_len"] / (df["word_count"] + 1)

    df["sentence_count"] = df["text"].str.count(r"[.!?]+").astype("float32")
    df["exclaim_count"] = df["text"].str.count("!").astype("float32")
    df["question_count"] = df["text"].str.count(r"\?").astype("float32")
    df["comma_count"] = df["text"].str.count(",").astype("float32")
    df["period_count"] = df["text"].str.count(r"\.").astype("float32")
    df["digit_count"] = df["text"].str.count(r"\d").astype("float32")

    df["upper_count"] = df["text"].str.count(r"[A-Z]").astype("float32")
    df["upper_ratio"] = df["upper_count"] / (df["text_len"] + 1)
    df["upper_per_word"] = df["upper_count"] / (df["word_count"] + 1)

    df["log_text_len"] = np.log1p(df["text_len"].clip(lower=0))
    df["log_word_count"] = np.log1p(df["word_count"].clip(lower=0))

    df["punctuation_total"] = (
            df["exclaim_count"] +
            df["question_count"] +
            df["comma_count"] +
            df["period_count"]
    )

    df["period_per_word"] = df["period_count"] / (df["word_count"] + 1)
    df["punctuation_per_word"] = df["punctuation_total"] / (df["word_count"] + 1)

    positive_words = [
        "good", "great", "excellent", "amazing", "awesome", "best",
        "love", "loved", "perfect", "delicious", "friendly", "nice",
        "wonderful", "fantastic", "recommend", "favorite", "fresh",
        "clean", "fast", "helpful", "enjoyed"
    ]

    negative_words = [
        "bad", "terrible", "awful", "worst", "hate", "hated",
        "poor", "rude", "slow", "dirty", "disappointed",
        "horrible", "never", "overpriced", "cold", "bland",
        "wait", "waiting", "wrong", "expensive"
    ]

    text_lower = df["text"].str.lower()

    df["positive_word_count"] = 0
    for w in positive_words:
        df["positive_word_count"] += text_lower.str.count(r"\b" + w + r"\b")

    df["negative_word_count"] = 0
    for w in negative_words:
        df["negative_word_count"] += text_lower.str.count(r"\b" + w + r"\b")

    df["positive_word_count"] = df["positive_word_count"].astype("float32")
    df["negative_word_count"] = df["negative_word_count"].astype("float32")

    df["sentiment_balance"] = df["positive_word_count"] - df["negative_word_count"]
    df["positive_word_ratio"] = df["positive_word_count"] / (df["word_count"] + 1)
    df["negative_word_ratio"] = df["negative_word_count"] / (df["word_count"] + 1)

    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"], errors="coerce")
        df["review_year"] = df["date"].dt.year.astype("float32")
        df["review_month"] = df["date"].dt.month.astype("float32")
        df["review_dayofweek"] = df["date"].dt.dayofweek.astype("float32")
        df["review_hour"] = df["date"].dt.hour.astype("float32")
        df["is_weekend"] = df["review_dayofweek"].isin([5, 6]).astype("float32")
    else:
        df["review_year"] = np.nan
        df["review_month"] = np.nan
        df["review_dayofweek"] = np.nan
        df["review_hour"] = np.nan
        df["is_weekend"] = np.nan

    if "stars" in df.columns:
        df["review_stars"] = pd.to_numeric(df["stars"], errors="coerce").astype("float32")
    else:
        df["review_stars"] = np.nan

    df["is_extreme_star"] = ((df["review_stars"] <= 2) | (df["review_stars"] >= 5)).astype("float32")
    df["is_positive_star"] = (df["review_stars"] >= 4).astype("float32")
    df["is_negative_star"] = (df["review_stars"] <= 2).astype("float32")

    return df

## Clean and create business-level features from business metadata
This section processes the raw business dataset and transforms business information into structured numerical features for modeling. It cleans core variables such as ratings, review counts, location, and open status, parses business categories into keyword-based indicators, extracts and converts nested business attributes (such as price range, WiFi, alcohol, reservations, and delivery options), and derives operational features like days open and average daily hours from business hours data. The goal is to convert complex categorical and semi-structured business metadata into machine-learning-ready business-level predictors.

In [ ]:
# ============================================================
# 6. Business Features
# ============================================================

def parse_attr_dict(val):
    if isinstance(val, dict):
        return val

    if pd.isna(val):
        return {}

    s = str(val)

    try:
        return json.loads(
            s.replace("'", '"')
            .replace("True", "true")
            .replace("False", "false")
            .replace("None", "null")
        )
    except Exception:
        try:
            return ast.literal_eval(s)
        except Exception:
            return {}


def get_numeric_col(df, col, default=np.nan):
    if col in df.columns:
        return pd.to_numeric(df[col], errors="coerce")
    else:
        return pd.Series([default] * len(df), index=df.index)


def attr_to_number(attr_dict, key):
    val = attr_dict.get(key, np.nan)

    if isinstance(val, bool):
        return float(val)

    if pd.isna(val):
        return np.nan

    val_str = str(val).strip().strip("'\"").lower()

    if val_str in ["true", "1", "yes"]:
        return 1.0
    if val_str in ["false", "0", "no"]:
        return 0.0
    if val_str in ["none", "nan", "null", ""]:
        return np.nan

    try:
        return float(val_str)
    except Exception:
        return np.nan


def map_attr_value(val, mapping):
    if pd.isna(val):
        return np.nan

    val_str = str(val).strip().strip("'\"").lower()
    return mapping.get(val_str, np.nan)


def clean_business_features(business):
    b = business.copy()
    b = b.drop_duplicates(subset=["business_id"], keep="first")

    b["business_stars"] = get_numeric_col(b, "stars").astype("float32")
    b["business_review_count"] = get_numeric_col(b, "review_count").astype("float32")
    b["is_open"] = get_numeric_col(b, "is_open", default=0).fillna(0).astype("float32")
    b["latitude"] = get_numeric_col(b, "latitude").astype("float32")
    b["longitude"] = get_numeric_col(b, "longitude").astype("float32")

    b["log_business_review_count"] = np.log1p(b["business_review_count"].clip(lower=0))

    if "categories" in b.columns:
        b["categories"] = b["categories"].fillna("").astype(str).str.lower()
    else:
        b["categories"] = ""

    b["category_count"] = b["categories"].apply(
        lambda x: 0 if x == "" else len([c for c in x.split(",") if c.strip()])
    ).astype("float32")

    category_keywords = {
        "cat_restaurant": "restaurant",
        "cat_food": "food",
        "cat_bar": "bar",
        "cat_nightlife": "nightlife",
        "cat_coffee": "coffee",
        "cat_shopping": "shopping",
        "cat_beauty": "beauty",
        "cat_hotel": "hotel",
        "cat_auto": "auto",
        "cat_health": "health",
        "cat_mexican": "mexican",
        "cat_chinese": "chinese",
        "cat_japanese": "japanese",
        "cat_american": "american",
        "cat_pizza": "pizza",
        "cat_burgers": "burgers",
        "cat_seafood": "seafood",
        "cat_breakfast": "breakfast",
        "cat_fastfood": "fast food",
        "cat_italian": "italian",
        "cat_sushi": "sushi",
        "cat_sandwich": "sandwich",
        "cat_event": "event planning",
        "cat_service": "services"
    }

    for new_col, keyword in category_keywords.items():
        b[new_col] = b["categories"].str.contains(keyword, regex=False).astype("float32")

    if "attributes" in b.columns:
        attrs = b["attributes"].apply(parse_attr_dict)
    else:
        attrs = pd.Series([{} for _ in range(len(b))], index=b.index)

    simple_attr_keys = [
        "BusinessAcceptsCreditCards",
        "BikeParking",
        "ByAppointmentOnly",
        "RestaurantsReservations",
        "RestaurantsGoodForGroups",
        "RestaurantsTakeOut",
        "GoodForKids",
        "HasTV",
        "Caters",
        "HappyHour",
        "OutdoorSeating",
        "RestaurantsDelivery",
        "RestaurantsTableService",
        "WheelchairAccessible",
        "DogsAllowed"
    ]

    for key in simple_attr_keys:
        b["attr_" + key] = attrs.apply(lambda d: attr_to_number(d, key)).astype("float32")

    attire_map = {"casual": 1.0, "dressy": 2.0, "formal": 3.0}
    alcohol_map = {"none": 0.0, "beer_and_wine": 1.0, "full_bar": 2.0}
    noise_map = {"quiet": 1.0, "average": 2.0, "loud": 3.0, "very_loud": 4.0}
    wifi_map = {"no": 0.0, "free": 1.0, "paid": 2.0}

    price_raw = attrs.apply(lambda d: d.get("RestaurantsPriceRange2", np.nan))
    b["attr_RestaurantsPriceRange2"] = pd.to_numeric(price_raw, errors="coerce").astype("float32")

    b["attr_RestaurantsAttire"] = attrs.apply(
        lambda d: map_attr_value(d.get("RestaurantsAttire", np.nan), attire_map)
    ).astype("float32")

    b["attr_Alcohol"] = attrs.apply(
        lambda d: map_attr_value(d.get("Alcohol", np.nan), alcohol_map)
    ).astype("float32")

    b["attr_NoiseLevel"] = attrs.apply(
        lambda d: map_attr_value(d.get("NoiseLevel", np.nan), noise_map)
    ).astype("float32")

    b["attr_WiFi"] = attrs.apply(
        lambda d: map_attr_value(d.get("WiFi", np.nan), wifi_map)
    ).astype("float32")

    def parse_hours(val):
        if pd.isna(val):
            return 0, 0.0

        try:
            h = ast.literal_eval(str(val))
            if not isinstance(h, dict):
                return 0, 0.0

            days_open = len(h)
            total_hours = 0.0

            for _, timerange in h.items():
                parts = str(timerange).split("-")
                if len(parts) != 2:
                    continue

                s1 = parts[0].split(":")
                s2 = parts[1].split(":")

                start = float(s1[0]) + float(s1[1]) / 60
                end = float(s2[0]) + float(s2[1]) / 60

                diff = end - start
                if diff < 0:
                    diff += 24

                total_hours += diff

            avg_hours = total_hours / days_open if days_open > 0 else 0.0
            return days_open, avg_hours

        except Exception:
            return 0, 0.0

    if "hours" in b.columns:
        hours_parsed = b["hours"].apply(parse_hours)
        b["days_open"] = hours_parsed.apply(lambda x: x[0]).astype("float32")
        b["avg_hours_per_day"] = hours_parsed.apply(lambda x: x[1]).astype("float32")
    else:
        b["days_open"] = 0.0
        b["avg_hours_per_day"] = 0.0

    keep_cols = [
        "business_id",
        "business_stars",
        "business_review_count",
        "log_business_review_count",
        "is_open",
        "latitude",
        "longitude",
        "category_count",
        "days_open",
        "avg_hours_per_day"
    ]

    keep_cols += list(category_keywords.keys())
    keep_cols += ["attr_" + key for key in simple_attr_keys]
    keep_cols += [
        "attr_RestaurantsPriceRange2",
        "attr_RestaurantsAttire",
        "attr_Alcohol",
        "attr_NoiseLevel",
        "attr_WiFi"
    ]

    keep_cols = [c for c in keep_cols if c in b.columns]

    return b[keep_cols]

## Clean and create user-level features from user metadata
This section transforms raw user information into structured user-level predictors for modeling. It removes duplicate users, converts numerical user statistics such as review count, funny votes, cool votes, fans, and average stars into usable numeric features, and optionally includes user useful votes depending on the leakage setting. It also creates behavioral features such as elite-year count, friend count, and the year the user started using Yelp. 

In [ ]:
# ============================================================
# 7. User Features
# ============================================================

def clean_user_features(user):
    u = user.copy()
    u = u.drop_duplicates(subset=["user_id"], keep="first")

    numeric_cols = ["review_count", "funny", "cool", "fans", "average_stars"]

    if USE_USER_USEFUL_FEATURES:
        numeric_cols.append("useful")

    for col in numeric_cols:
        if col in u.columns:
            u["user_" + col] = pd.to_numeric(u[col], errors="coerce").astype("float32")
        else:
            u["user_" + col] = np.nan

    if "elite" in u.columns:
        u["elite"] = u["elite"].fillna("").astype(str)
        u["elite"] = u["elite"].str.replace("20,20", "2020", regex=False)

        u["user_elite_count"] = u["elite"].apply(
            lambda x: 0 if x.strip() == "" or x.lower() in ["nan", "none"]
            else len([i for i in x.split(",") if i.strip()])
        ).astype("float32")
    else:
        u["user_elite_count"] = 0.0

    if "friends" in u.columns:
        u["friends"] = u["friends"].fillna("").astype(str)
        u["user_friend_count"] = u["friends"].apply(
            lambda x: 0 if x.strip() == "" or x.lower() in ["none", "nan"]
            else len([i for i in x.split(",") if i.strip()])
        ).astype("float32")
    else:
        u["user_friend_count"] = 0.0

    if "yelping_since" in u.columns:
        u["yelping_since"] = pd.to_datetime(u["yelping_since"], errors="coerce")
        u["yelping_since_year"] = u["yelping_since"].dt.year.astype("float32")
    else:
        u["yelping_since_year"] = np.nan

    keep_cols = ["user_id"]
    keep_cols += ["user_" + col for col in numeric_cols]
    keep_cols += [
        "user_elite_count",
        "user_friend_count",
        "yelping_since_year"
    ]

    keep_cols = [c for c in keep_cols if c in u.columns]

    return u[keep_cols]

## Aggregate tip-level features for businesses and users
This section summarizes tip data into business-level and user-level aggregated features. It extracts basic text characteristics from tips, such as text length and word count, incorporates compliment counts, and then groups the data separately by business and by user. For each business and user, it calculates total tip volume, average tip length, average tip word count, and total compliments received.

In [ ]:
# ============================================================
# 8. Tip Aggregation
# ============================================================

def aggregate_tip_features(tip):
    t = tip.copy()

    if "text" not in t.columns:
        t["text"] = ""

    t["text"] = t["text"].fillna("").astype(str)
    t["tip_text_len"] = t["text"].str.len().astype("float32")
    t["tip_word_count"] = t["text"].str.split().str.len().astype("float32")

    if "compliment_count" in t.columns:
        t["compliment_count"] = pd.to_numeric(
            t["compliment_count"], errors="coerce"
        ).fillna(0).astype("float32")
    else:
        t["compliment_count"] = 0.0

    tip_business = t.groupby("business_id", as_index=False).agg(
        business_tip_count=("text", "count"),
        business_tip_avg_len=("tip_text_len", "mean"),
        business_tip_avg_words=("tip_word_count", "mean"),
        business_tip_compliments=("compliment_count", "sum")
    )

    tip_user = t.groupby("user_id", as_index=False).agg(
        user_tip_count=("text", "count"),
        user_tip_avg_len=("tip_text_len", "mean"),
        user_tip_avg_words=("tip_word_count", "mean"),
        user_tip_compliments=("compliment_count", "sum")
    )

    for df in [tip_business, tip_user]:
        for c in df.columns:
            if c not in ["business_id", "user_id"]:
                df[c] = pd.to_numeric(df[c], errors="coerce").astype("float32")

    return tip_business, tip_user

## Build final feature tables for training and testing
This section combines all engineered features into the final training and testing feature tables. It first creates cleaned business features, user features, and aggregated tip features, then applies review-level feature engineering to both train and test datasets. After that, it merges business-level, user-level, business-tip, and user-tip features into the review data using business_id and user_id. Finally, it prints the table dimensions and total processing time to confirm the feature-building process.

In [ ]:
# ============================================================
# 9. Build Feature Tables
# ============================================================

t0 = time.time()

business_clean = clean_business_features(business)
user_clean = clean_user_features(user)
tip_business, tip_user = aggregate_tip_features(tip)

print("business_clean:", business_clean.shape)
print("user_clean:", user_clean.shape)
print("tip_business:", tip_business.shape)
print("tip_user:", tip_user.shape)

train_fe = add_review_features(train_x)
test_fe = add_review_features(test_x)

print("train_fe before merge:", train_fe.shape)
print("test_fe before merge:", test_fe.shape)

train_fe = train_fe.merge(business_clean, on="business_id", how="left")
test_fe = test_fe.merge(business_clean, on="business_id", how="left")

train_fe = train_fe.merge(user_clean, on="user_id", how="left")
test_fe = test_fe.merge(user_clean, on="user_id", how="left")

train_fe = train_fe.merge(tip_business, on="business_id", how="left")
test_fe = test_fe.merge(tip_business, on="business_id", how="left")

train_fe = train_fe.merge(tip_user, on="user_id", how="left")
test_fe = test_fe.merge(tip_user, on="user_id", how="left")

print("train_fe after merge:", train_fe.shape)
print("test_fe after merge:", test_fe.shape)
print("Feature table time:", (time.time() - t0) / 60, "minutes")

## Create interaction, ratio, and log-transformed features
This section adds advanced engineered features to capture relationships between review, user, and business information. It calculates rating differences between the review rating, business average rating, and user average rating, applies log transformations to reduce skewness in count-based variables, and creates per-review ratios such as user feedback per review and fans per review. It also builds interaction features, such as review length combined with user activity, business popularity, tip counts, and rating sentiment. These features help the model capture more complex patterns that simple individual variables may miss.

In [ ]:
# ============================================================
# 10. Interaction, Ratio, Log Features
# ============================================================

for df in [train_fe, test_fe]:

    df["star_diff_from_business"] = df["review_stars"] - df["business_stars"]
    df["user_star_diff"] = df["review_stars"] - df["user_average_stars"]

    df["abs_star_diff_from_business"] = df["star_diff_from_business"].abs()
    df["abs_user_star_diff"] = df["user_star_diff"].abs()

    df["log_user_review_count"] = np.log1p(df["user_review_count"].fillna(0).clip(lower=0))
    df["log_user_fans"] = np.log1p(df["user_fans"].fillna(0).clip(lower=0))
    df["log_user_friend_count"] = np.log1p(df["user_friend_count"].fillna(0).clip(lower=0))
    df["log_user_elite_count"] = np.log1p(df["user_elite_count"].fillna(0).clip(lower=0))

    if USE_USER_USEFUL_FEATURES and "user_useful" in df.columns:
        df["log_user_useful"] = np.log1p(df["user_useful"].fillna(0).clip(lower=0))
    else:
        df["user_useful"] = 0.0
        df["log_user_useful"] = 0.0

    if "user_funny" not in df.columns:
        df["user_funny"] = 0.0

    if "user_cool" not in df.columns:
        df["user_cool"] = 0.0

    df["log_user_funny"] = np.log1p(df["user_funny"].fillna(0).clip(lower=0))
    df["log_user_cool"] = np.log1p(df["user_cool"].fillna(0).clip(lower=0))

    df["user_total_feedback"] = df["user_useful"] + df["user_funny"] + df["user_cool"]
    df["log_user_total_feedback"] = np.log1p(df["user_total_feedback"].fillna(0).clip(lower=0))

    df["user_feedback_per_review"] = df["user_total_feedback"] / (df["user_review_count"] + 1)
    df["user_useful_per_review"] = df["user_useful"] / (df["user_review_count"] + 1)
    df["user_funny_per_review"] = df["user_funny"] / (df["user_review_count"] + 1)
    df["user_cool_per_review"] = df["user_cool"] / (df["user_review_count"] + 1)
    df["user_fans_per_review"] = df["user_fans"] / (df["user_review_count"] + 1)

    df["user_useful_share"] = df["user_useful"] / (df["user_total_feedback"] + 1)
    df["user_funny_share"] = df["user_funny"] / (df["user_total_feedback"] + 1)
    df["user_cool_share"] = df["user_cool"] / (df["user_total_feedback"] + 1)

    df["review_len_x_user_review_count"] = df["log_text_len"] * df["log_user_review_count"]
    df["review_len_x_user_useful"] = df["log_text_len"] * df["user_useful_per_review"]
    df["log_text_x_log_user_useful"] = df["log_text_len"] * df["log_user_useful"]
    df["log_text_x_user_feedback_per_review"] = df["log_text_len"] * df["user_feedback_per_review"]
    df["log_word_x_log_user_useful"] = df["log_word_count"] * df["log_user_useful"]

    df["business_popularity_x_star_diff"] = df["log_business_review_count"] * df["star_diff_from_business"]
    df["business_popularity_x_text_len"] = df["log_business_review_count"] * df["log_text_len"]
    df["log_text_x_log_business_reviews"] = df["log_text_len"] * df["log_business_review_count"]
    df["log_word_x_log_business_reviews"] = df["log_word_count"] * df["log_business_review_count"]

    df["negative_long_review"] = df["is_negative_star"] * df["log_text_len"]
    df["positive_long_review"] = df["is_positive_star"] * df["log_text_len"]
    df["extreme_long_review"] = df["is_extreme_star"] * df["log_text_len"]

    df["log_business_tip_count"] = np.log1p(df["business_tip_count"].fillna(0).clip(lower=0))
    df["log_user_tip_count"] = np.log1p(df["user_tip_count"].fillna(0).clip(lower=0))

    df["business_tip_x_text_len"] = df["log_business_tip_count"] * df["log_text_len"]
    df["user_tip_x_text_len"] = df["log_user_tip_count"] * df["log_text_len"]

train_fe = train_fe.loc[:, ~train_fe.columns.duplicated()]
test_fe = test_fe.loc[:, ~test_fe.columns.duplicated()]

print("train_fe after interaction:", train_fe.shape)
print("test_fe after interaction:", test_fe.shape)

gc.collect()

## Create out-of-fold target encoding features
This section creates target encoding features for high-cardinality variables such as user_id, business_id, review year, and review stars. It uses out-of-fold encoding to reduce data leakage by calculating encoded values from training folds only, then applies smoothed full-data encoding to the test set. It also creates target-encoding interaction features with review length and user/business combinations. These features help the model capture historical usefulness patterns by user, business, rating, and time while controlling overfitting.

In [ ]:
# 11. Out-of-Fold Target Encoding (HIGHLY OPTIMIZED with Vectorization)

def add_oof_target_encoding(
        train_df,
        test_df,
        y,
        cols,
        n_splits=5,
        smoothing=30,
        random_state=42
):
    """
    Highly optimized out-of-fold target encoding using:
    - Integer-based hash keys (no string conversion)
    - Pure NumPy vectorized operations
    - Efficient groupby using pd.core.algorithms.factorize
    """
    y_arr = np.asarray(y, dtype="float32")
    global_mean = y_arr.mean()

    if isinstance(cols, str):
        cols = [cols]

    enc_name = "te_" + "_".join(cols)

    # Create hash-based integer keys using pandas factorize (MUCH faster than string conversion)
    train_key_tuple = train_df[cols].apply(lambda x: tuple(x), axis=1)
    test_key_tuple = test_df[cols].apply(lambda x: tuple(x), axis=1)
    
    key_train, key_train_levels = pd.factorize(train_key_tuple)
    key_test, _ = pd.factorize(test_key_tuple)
    
    oof_values = np.full(len(train_df), global_mean, dtype="float32")
    
    skf = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state
    )

    # Build OOF encodings using efficient numpy operations
    for tr_idx, va_idx in skf.split(train_df, y_arr):
        fold_key_train = key_train[tr_idx]
        fold_y_train = y_arr[tr_idx]
        
        # Use pandas groupby for efficiency, then convert to dict
        fold_df = pd.DataFrame({"key": fold_key_train, "target": fold_y_train})
        stats = fold_df.groupby("key")["target"].agg(["mean", "size"])
        
        # Compute smoothed values
        smoothed = (stats["mean"] * stats["size"] + global_mean * smoothing) / (stats["size"] + smoothing)
        stats_dict = smoothed.to_dict()
        
        # Vectorized assignment: use np.vectorize or direct indexing
        val_keys = key_train[va_idx]
        for unique_key in np.unique(val_keys):
            if unique_key in stats_dict:
                mask = val_keys == unique_key
                oof_values[va_idx[mask]] = stats_dict[unique_key]

    # Compute full dataset encoding
    full_df = pd.DataFrame({"key": key_train, "target": y_arr})
    full_stats = full_df.groupby("key")["target"].agg(["mean", "size"])
    full_smoothed = (full_stats["mean"] * full_stats["size"] + global_mean * smoothing) / (full_stats["size"] + smoothing)
    full_stats_dict = full_smoothed.to_dict()
    
    # Assign results (no copy needed, work in-place)
    train_df[enc_name] = oof_values
    
    # Vectorized test encoding using numpy vectorize for speed
    test_encoding = np.array([full_stats_dict.get(k, global_mean) for k in key_test], dtype="float32")
    test_df[enc_name] = test_encoding

    print("Added target encoding:", enc_name)

    return train_df, test_df


y_series = pd.Series(y).reset_index(drop=True).astype(int)
train_fe = train_fe.reset_index(drop=True)
test_fe = test_fe.reset_index(drop=True)

single_te_cols = [
    "user_id",
    "business_id",
    "review_year",
    "review_stars"
]

for col in single_te_cols:
    if col in train_fe.columns and col in test_fe.columns:
        train_fe, test_fe = add_oof_target_encoding(
            train_fe,
            test_fe,
            y_series,
            cols=[col],
            n_splits=5,
            smoothing=30,
            random_state=RANDOM_STATE
        )

combo_te_cols = [
    ["business_id", "review_stars"],
    ["user_id", "review_stars"],
    ["business_id", "review_year"],
    ["user_id", "review_year"],
    ["review_year", "review_stars"]
]

for cols in combo_te_cols:
    if all(c in train_fe.columns for c in cols) and all(c in test_fe.columns for c in cols):
        train_fe, test_fe = add_oof_target_encoding(
            train_fe,
            test_fe,
            y_series,
            cols=cols,
            n_splits=5,
            smoothing=50,
            random_state=RANDOM_STATE
        )

for df in [train_fe, test_fe]:

    if "te_user_id" in df.columns:
        df["te_user_x_log_text_len"] = df["te_user_id"] * df["log_text_len"]

    if "te_business_id" in df.columns:
        df["te_business_x_log_text_len"] = df["te_business_id"] * df["log_text_len"]

    if "te_user_id" in df.columns and "te_business_id" in df.columns:
        df["te_user_minus_business"] = df["te_user_id"] - df["te_business_id"]
        df["te_user_x_business"] = df["te_user_id"] * df["te_business_id"]

    if "te_business_id_review_stars" in df.columns:
        df["te_business_star_x_log_text"] = df["te_business_id_review_stars"] * df["log_text_len"]

    if "te_user_id_review_stars" in df.columns:
        df["te_user_star_x_log_text"] = df["te_user_id_review_stars"] * df["log_text_len"]

te_cols = [c for c in train_fe.columns if c.startswith("te_")]

print("Number of TE columns:", len(te_cols))
print("train_fe after TE:", train_fe.shape)
print("test_fe after TE:", test_fe.shape)

gc.collect()

## Select final numeric features for modeling

In [ ]:
# 12. Select Numeric Features

drop_cols = [
    "review_id",
    "user_id",
    "business_id",
    "text",
    "date",
    "stars"
]

feature_cols = [c for c in train_fe.columns if c not in drop_cols]

X = train_fe[feature_cols].select_dtypes(include=["number", "bool"]).copy()
X_test_final = test_fe[X.columns].copy()

X = X.replace([np.inf, -np.inf], np.nan)
X_test_final = X_test_final.replace([np.inf, -np.inf], np.nan)

X = X.astype("float32")
X_test_final = X_test_final.astype("float32")

print("Number of features:", X.shape[1])
print("X:", X.shape)
print("X_test_final:", X_test_final.shape)

feature_list_path = os.path.join(OUTPUT_DIR, "super_final_feature_list.csv")
pd.DataFrame({"feature": X.columns}).to_csv(feature_list_path, index=False) 
print("Saved feature list:", feature_list_path)

## Split data into Train and Validation sets

In [ ]:
# 13. Train / Validation Split

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("y_train positive rate:", y_train.mean())
print("y_val positive rate:", y_val.mean())

gc.collect()

In [ ]:
# ============================================================ 
# 14. Baseline Logistic Regression Model
# ============================================================

baseline_logreg_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        C=1.0,
        penalty="l2",
        class_weight="balanced",
        solver="saga",
        max_iter=1000,
        n_jobs=N_JOBS,
        random_state=RANDOM_STATE
    ))
])

print("Training baseline logistic regression...")
baseline_logreg_model.fit(X_train, y_train)

baseline_val_score = baseline_logreg_model.predict_proba(X_val)[:, 1]
baseline_test_score = baseline_logreg_model.predict_proba(X_test_final)[:, 1]

print("Baseline logistic regression training complete.")
print("Validation scores shape:", baseline_val_score.shape)
print("Test scores shape:", baseline_test_score.shape)

gc.collect()


## Evaluate ROC, AUC, and TPR at 10% FPR


In [ ]:
# ============================================================
# 15. Baseline ROC Curve and TPR at 10% FPR
# ============================================================

baseline_metrics = best_tpr_under_fpr(
    y_val,
    baseline_val_score,
    fpr_limit=FPR_LIMIT
)

print_metrics("BASELINE LOGISTIC REGRESSION VALIDATION METRICS", baseline_metrics)

roc_curve_path = plot_roc_with_operating_point(
    y_true=y_val,
    y_score=baseline_val_score,
    metrics=baseline_metrics,
    output_dir=OUTPUT_DIR,
    filename="baseline_logreg_roc_curve.png"
)

baseline_metrics_df = pd.DataFrame([{
    "model": "Baseline Logistic Regression",
    "auc": baseline_metrics["auc"],
    "fpr_limit": FPR_LIMIT,
    "fpr_at_threshold": baseline_metrics["fpr"],
    "tpr_at_10pct_fpr": baseline_metrics["tpr"],
    "threshold": baseline_metrics["threshold"],
    "accuracy": baseline_metrics["accuracy"],
    "tn": baseline_metrics["tn"],
    "fp": baseline_metrics["fp"],
    "fn": baseline_metrics["fn"],
    "tp": baseline_metrics["tp"],
    "roc_curve_path": roc_curve_path
}])

baseline_metrics_path = os.path.join(OUTPUT_DIR, "baseline_logreg_validation_metrics.csv")
baseline_metrics_df.to_csv(baseline_metrics_path, index=False)

print("Saved baseline metrics:", baseline_metrics_path)
print(baseline_metrics_df)


## Export baseline logistic regression submission


In [ ]:
# ============================================================
# 16. Export Baseline Logistic Regression Submission
# ============================================================

baseline_threshold = baseline_metrics["threshold"]
baseline_test_pred = (baseline_test_score >= baseline_threshold).astype(int)

baseline_submission = pd.DataFrame({
    "top_useful": baseline_test_pred
})

baseline_submission_path = os.path.join(OUTPUT_DIR, "group_7_yelp.csv")
baseline_submission.to_csv(
    baseline_submission_path,
    index=False,
    header=False
)

baseline_score_path = os.path.join(OUTPUT_DIR, "baseline_logreg_test_scores.csv")
pd.DataFrame({
    "baseline_logreg_score": baseline_test_score,
    "prediction": baseline_test_pred
}).to_csv(baseline_score_path, index=False)

baseline_summary = {
    "model": "Baseline Logistic Regression",
    "validation_auc": baseline_metrics["auc"],
    "validation_fpr": baseline_metrics["fpr"],
    "validation_tpr_at_10pct_fpr": baseline_metrics["tpr"],
    "threshold": baseline_threshold,
    "positive_predictions": int(baseline_test_pred.sum()),
    "positive_ratio": float(baseline_test_pred.mean())
}

baseline_summary_path = os.path.join(OUTPUT_DIR, "baseline_logreg_final_summary.csv")
pd.DataFrame([baseline_summary]).to_csv(baseline_summary_path, index=False)

print("Submission:", baseline_submission_path)
print("Scores:", baseline_score_path)
print("Summary:", baseline_summary_path)
print("Positive predictions:", baseline_test_pred.sum())
print("Positive ratio:", baseline_test_pred.mean())
print("Validation FPR:", baseline_metrics["fpr"])
print("Validation TPR at 10% FPR:", baseline_metrics["tpr"])


## 7. Model 6: Logistic Regression With Holiday Features

This logistic-regression ablation keeps the holiday/date features and can be compared with Model 5 to assess the value of the external holiday signal.


In [ ]:
# Yelp Top-Useful Prediction
# Baseline Logistic Regression Version
#
# Includes:
# 1. External data source: US holiday calendar
# 2. 10+ feature insight tables/charts
# 3. Baseline logistic regression classifier
# 4. ROC curve and ROC-AUC
# 5. TPR at FPR <= 10%
# 6. Final submission generated from the validation-selected threshold

import os
import gc
import ast
import json
import re
import time
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, accuracy_score

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

import holidays


## Import dataset and configure initial settings

In [ ]:
# 1. Settings

RANDOM_STATE = 42
N_JOBS = -1

# Data files are stored in the same project folder as this notebook.
# DATA_DIR is defined in the Project Setup section above
# OUTPUT_DIR is defined in the Project Setup section above

# Required operating point: maximize TPR while keeping FPR at or below 10%.
FPR_LIMIT = 0.10

# change user.useful to False because it is leakage
USE_USER_USEFUL_FEATURES = False

print("Settings loaded.")


## Load datasets and inspect data structure

In [ ]:

# 2. Load Data
# Direct pd.read_csv calls are used for Kaggle compatibility.

train_x = pd.read_csv(csv_path(TRAIN_X_FILE))
train_y = pd.read_csv(csv_path(TRAIN_Y_FILE))
test_x = pd.read_csv(csv_path(TEST_X_FILE))
business = pd.read_csv(csv_path(BUSINESS_CLEAN_FILE, [BUSINESS_FILE]))
user = pd.read_csv(csv_path(USER_CLEAN_FILE, [USER_FILE]))
tip = pd.read_csv(csv_path(TIP_CLEAN_FILE, [TIP_FILE]))

if train_y.shape[1] == 1:
    y = train_y.iloc[:, 0].astype(int).reset_index(drop=True)
else:
    y = train_y["top_useful"].astype(int).reset_index(drop=True)

# Find distribution and shapes
print("train_x:", train_x.shape)
print("train_y:", train_y.shape)
print("test_x:", test_x.shape)
print("business:", business.shape)
print("user:", user.shape)
print("tip:", tip.shape)

print("\nLabel distribution:")
print(y.value_counts())
print("Positive rate:", y.mean())

assert len(train_x) == len(y), "train_x and train_y row counts do not match."


## Define evaluation functions for model performance assessment

In [ ]:
# ============================================================
# 3. Evaluation Functions
# ============================================================

# Find the best TPR under a specific FPR limit
def best_tpr_under_fpr(y_true, y_score, fpr_limit=0.10):
    fpr, tpr, thresholds = roc_curve(y_true, y_score)
    valid_idx = np.where(fpr <= fpr_limit)[0]

    if len(valid_idx) == 0:
        return {
            "auc": roc_auc_score(y_true, y_score),
            "fpr": np.nan,
            "tpr": np.nan,
            "threshold": np.nan,
            "accuracy": np.nan,
            "tn": np.nan,
            "fp": np.nan,
            "fn": np.nan,
            "tp": np.nan
        }

    best_idx = valid_idx[np.argmax(tpr[valid_idx])]
    best_threshold = thresholds[best_idx]

    y_pred = (y_score >= best_threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    return {
        "auc": roc_auc_score(y_true, y_score),
        "fpr": fp / (fp + tn),
        "tpr": tp / (tp + fn),
        "threshold": best_threshold,
        "accuracy": accuracy_score(y_true, y_pred),
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp
    }


def print_metrics(name, metrics):
    print("\n" + "=" * 80)
    print(name)
    print("=" * 80)
    for k, v in metrics.items():
        print(f"{k}: {v}")


def get_model_score(model, X_data):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X_data)[:, 1]
    return model.decision_function(X_data)


def plot_roc_with_operating_point(y_true, y_score, metrics, output_dir, filename="baseline_logreg_roc_curve.png"):
    fpr, tpr, _ = roc_curve(y_true, y_score)
    auc = roc_auc_score(y_true, y_score)

    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, label=f"Baseline Logistic Regression AUC = {auc:.4f}")
    plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random classifier")
    plt.axvline(FPR_LIMIT, linestyle=":", color="red", label="10% FPR limit")
    plt.scatter(metrics["fpr"], metrics["tpr"], color="red", zorder=5, label=f"TPR = {metrics['tpr']:.4f} at FPR = {metrics['fpr']:.4f}")
    plt.title("ROC Curve: Baseline Logistic Regression")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.xlim(0, 1)
    plt.ylim(0, 1)
    plt.legend(loc="lower right")
    plt.tight_layout()

    fig_path = os.path.join(output_dir, filename)
    plt.savefig(fig_path, dpi=200)
    plt.show()

    print("Saved ROC curve:", fig_path)
    return fig_path


## Create external holiday-based features from review dates
This section adds U.S. holiday-related variables based on each review date. It identifies whether a review was written on a U.S. holiday, calculates how many days the review is from the nearest holiday, and measures the days since the previous holiday and until the next holiday. It also creates holiday-window indicators, such as whether the review was written within 3 or 7 days of a holiday. 

In [ ]:
# ============================================================
# 4. External Holiday Features
# ============================================================

def add_us_holiday_features(df):
    df = df.copy()

    if "date" not in df.columns:
        df["is_us_holiday"] = 0.0
        df["days_to_nearest_holiday"] = 999.0
        df["days_since_previous_holiday"] = 999.0
        df["days_until_next_holiday"] = 999.0
        df["is_holiday_window_3"] = 0.0
        df["is_holiday_window_7"] = 0.0
        df["is_before_holiday_3"] = 0.0
        df["is_after_holiday_3"] = 0.0
        return df

    date_series = pd.to_datetime(df["date"], errors="coerce")
    years = date_series.dt.year.dropna().astype(int)

    if len(years) == 0:
        min_year, max_year = 2004, 2023
    else:
        min_year = int(years.min()) - 1
        max_year = int(years.max()) + 1

    us_holidays = holidays.US(years=range(min_year, max_year + 1))
    holiday_dates = sorted(list(us_holidays.keys()))

    holiday_ord = np.array(
        pd.to_datetime(holiday_dates).values.astype("datetime64[D]").astype("int64")
    )

    date_day = date_series.dt.floor("D")
    valid_mask = date_series.notna().values
    date_ord_all = date_day.values.astype("datetime64[D]").astype("int64")

    is_holiday = np.zeros(len(df), dtype="float32")
    nearest_days = np.full(len(df), 999.0, dtype="float32")
    since_prev = np.full(len(df), 999.0, dtype="float32")
    until_next = np.full(len(df), 999.0, dtype="float32")

    valid_dates = date_ord_all[valid_mask]

    idx = np.searchsorted(holiday_ord, valid_dates)

    has_prev = idx > 0
    has_next = idx < len(holiday_ord)

    prev_days = np.full(len(valid_dates), 999.0)
    next_days = np.full(len(valid_dates), 999.0)

    prev_days[has_prev] = valid_dates[has_prev] - holiday_ord[idx[has_prev] - 1]
    next_days[has_next] = holiday_ord[idx[has_next]] - valid_dates[has_next]

    exact_holiday = np.isin(valid_dates, holiday_ord)
    prev_days[exact_holiday] = 0
    next_days[exact_holiday] = 0

    nearest = np.minimum(prev_days, next_days)

    is_holiday[valid_mask] = exact_holiday.astype("float32")
    nearest_days[valid_mask] = nearest.astype("float32")
    since_prev[valid_mask] = prev_days.astype("float32")
    until_next[valid_mask] = next_days.astype("float32")

    df["is_us_holiday"] = is_holiday
    df["days_to_nearest_holiday"] = nearest_days
    df["days_since_previous_holiday"] = since_prev
    df["days_until_next_holiday"] = until_next

    df["is_holiday_window_3"] = (df["days_to_nearest_holiday"] <= 3).astype("float32")
    df["is_holiday_window_7"] = (df["days_to_nearest_holiday"] <= 7).astype("float32")
    df["is_before_holiday_3"] = (df["days_until_next_holiday"] <= 3).astype("float32")
    df["is_after_holiday_3"] = (df["days_since_previous_holiday"] <= 3).astype("float32")

    return df

## Create review-level text, sentiment, date, rating, and holiday features
This section transforms raw review data into rich review-level predictors by engineering features from multiple dimensions of each review. It extracts textual structure features such as review length, word count, sentence count, punctuation usage, capitalization patterns, and digit frequency; builds simple sentiment indicators using predefined positive and negative keyword counts; captures temporal patterns from review dates including year, month, weekday, weekend behavior, and holiday proximity; and incorporates star rating behavior through positive, negative, and extreme rating flags. Together, these features convert unstructured review text and metadata into machine-learning-ready variables that capture writing style, sentiment tone, timing effects, and rating behavior.

In [ ]:
# ============================================================
# 5. Review-Level Features
# ============================================================

def add_review_features(df):
    df = df.copy()

    if "text" not in df.columns:
        df["text"] = ""

    df["text"] = df["text"].fillna("").astype(str)

    df["text_len"] = df["text"].str.len().astype("float32")
    df["word_count"] = df["text"].str.split().str.len().astype("float32")
    df["avg_word_len"] = df["text_len"] / (df["word_count"] + 1)

    df["sentence_count"] = df["text"].str.count(r"[.!?]+").astype("float32")
    df["exclaim_count"] = df["text"].str.count("!").astype("float32")
    df["question_count"] = df["text"].str.count(r"\?").astype("float32")
    df["comma_count"] = df["text"].str.count(",").astype("float32")
    df["period_count"] = df["text"].str.count(r"\.").astype("float32")
    df["digit_count"] = df["text"].str.count(r"\d").astype("float32")

    df["upper_count"] = df["text"].str.count(r"[A-Z]").astype("float32")
    df["upper_ratio"] = df["upper_count"] / (df["text_len"] + 1)
    df["upper_per_word"] = df["upper_count"] / (df["word_count"] + 1)

    df["log_text_len"] = np.log1p(df["text_len"].clip(lower=0))
    df["log_word_count"] = np.log1p(df["word_count"].clip(lower=0))

    df["punctuation_total"] = (
            df["exclaim_count"] +
            df["question_count"] +
            df["comma_count"] +
            df["period_count"]
    )

    df["period_per_word"] = df["period_count"] / (df["word_count"] + 1)
    df["punctuation_per_word"] = df["punctuation_total"] / (df["word_count"] + 1)

    positive_words = [
        "good", "great", "excellent", "amazing", "awesome", "best",
        "love", "loved", "perfect", "delicious", "friendly", "nice",
        "wonderful", "fantastic", "recommend", "favorite", "fresh",
        "clean", "fast", "helpful", "enjoyed"
    ]

    negative_words = [
        "bad", "terrible", "awful", "worst", "hate", "hated",
        "poor", "rude", "slow", "dirty", "disappointed",
        "horrible", "never", "overpriced", "cold", "bland",
        "wait", "waiting", "wrong", "expensive"
    ]

    text_lower = df["text"].str.lower()

    df["positive_word_count"] = 0
    for w in positive_words:
        df["positive_word_count"] += text_lower.str.count(r"\b" + w + r"\b")

    df["negative_word_count"] = 0
    for w in negative_words:
        df["negative_word_count"] += text_lower.str.count(r"\b" + w + r"\b")

    df["positive_word_count"] = df["positive_word_count"].astype("float32")
    df["negative_word_count"] = df["negative_word_count"].astype("float32")

    df["sentiment_balance"] = df["positive_word_count"] - df["negative_word_count"]
    df["positive_word_ratio"] = df["positive_word_count"] / (df["word_count"] + 1)
    df["negative_word_ratio"] = df["negative_word_count"] / (df["word_count"] + 1)

    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"], errors="coerce")
        df["review_year"] = df["date"].dt.year.astype("float32")
        df["review_month"] = df["date"].dt.month.astype("float32")
        df["review_dayofweek"] = df["date"].dt.dayofweek.astype("float32")
        df["review_hour"] = df["date"].dt.hour.astype("float32")
        df["is_weekend"] = df["review_dayofweek"].isin([5, 6]).astype("float32")
    else:
        df["review_year"] = np.nan
        df["review_month"] = np.nan
        df["review_dayofweek"] = np.nan
        df["review_hour"] = np.nan
        df["is_weekend"] = np.nan

    if "stars" in df.columns:
        df["review_stars"] = pd.to_numeric(df["stars"], errors="coerce").astype("float32")
    else:
        df["review_stars"] = np.nan

    df["is_extreme_star"] = ((df["review_stars"] <= 2) | (df["review_stars"] >= 5)).astype("float32")
    df["is_positive_star"] = (df["review_stars"] >= 4).astype("float32")
    df["is_negative_star"] = (df["review_stars"] <= 2).astype("float32")

    df = add_us_holiday_features(df)

    return df

## Clean and create business-level features from business metadata
This section processes the raw business dataset and transforms business information into structured numerical features for modeling. It cleans core variables such as ratings, review counts, location, and open status, parses business categories into keyword-based indicators, extracts and converts nested business attributes (such as price range, WiFi, alcohol, reservations, and delivery options), and derives operational features like days open and average daily hours from business hours data. The goal is to convert complex categorical and semi-structured business metadata into machine-learning-ready business-level predictors.

In [ ]:
# ============================================================
# 6. Business Features
# ============================================================

def parse_attr_dict(val):
    if isinstance(val, dict):
        return val

    if pd.isna(val):
        return {}

    s = str(val)

    try:
        return json.loads(
            s.replace("'", '"')
            .replace("True", "true")
            .replace("False", "false")
            .replace("None", "null")
        )
    except Exception:
        try:
            return ast.literal_eval(s)
        except Exception:
            return {}


def get_numeric_col(df, col, default=np.nan):
    if col in df.columns:
        return pd.to_numeric(df[col], errors="coerce")
    else:
        return pd.Series([default] * len(df), index=df.index)


def attr_to_number(attr_dict, key):
    val = attr_dict.get(key, np.nan)

    if isinstance(val, bool):
        return float(val)

    if pd.isna(val):
        return np.nan

    val_str = str(val).strip().strip("'\"").lower()

    if val_str in ["true", "1", "yes"]:
        return 1.0
    if val_str in ["false", "0", "no"]:
        return 0.0
    if val_str in ["none", "nan", "null", ""]:
        return np.nan

    try:
        return float(val_str)
    except Exception:
        return np.nan


def map_attr_value(val, mapping):
    if pd.isna(val):
        return np.nan

    val_str = str(val).strip().strip("'\"").lower()
    return mapping.get(val_str, np.nan)


def clean_business_features(business):
    b = business.copy()
    b = b.drop_duplicates(subset=["business_id"], keep="first")

    b["business_stars"] = get_numeric_col(b, "stars").astype("float32")
    b["business_review_count"] = get_numeric_col(b, "review_count").astype("float32")
    b["is_open"] = get_numeric_col(b, "is_open", default=0).fillna(0).astype("float32")
    b["latitude"] = get_numeric_col(b, "latitude").astype("float32")
    b["longitude"] = get_numeric_col(b, "longitude").astype("float32")

    b["log_business_review_count"] = np.log1p(b["business_review_count"].clip(lower=0))

    if "categories" in b.columns:
        b["categories"] = b["categories"].fillna("").astype(str).str.lower()
    else:
        b["categories"] = ""

    b["category_count"] = b["categories"].apply(
        lambda x: 0 if x == "" else len([c for c in x.split(",") if c.strip()])
    ).astype("float32")

    category_keywords = {
        "cat_restaurant": "restaurant",
        "cat_food": "food",
        "cat_bar": "bar",
        "cat_nightlife": "nightlife",
        "cat_coffee": "coffee",
        "cat_shopping": "shopping",
        "cat_beauty": "beauty",
        "cat_hotel": "hotel",
        "cat_auto": "auto",
        "cat_health": "health",
        "cat_mexican": "mexican",
        "cat_chinese": "chinese",
        "cat_japanese": "japanese",
        "cat_american": "american",
        "cat_pizza": "pizza",
        "cat_burgers": "burgers",
        "cat_seafood": "seafood",
        "cat_breakfast": "breakfast",
        "cat_fastfood": "fast food",
        "cat_italian": "italian",
        "cat_sushi": "sushi",
        "cat_sandwich": "sandwich",
        "cat_event": "event planning",
        "cat_service": "services"
    }

    for new_col, keyword in category_keywords.items():
        b[new_col] = b["categories"].str.contains(keyword, regex=False).astype("float32")

    if "attributes" in b.columns:
        attrs = b["attributes"].apply(parse_attr_dict)
    else:
        attrs = pd.Series([{} for _ in range(len(b))], index=b.index)

    simple_attr_keys = [
        "BusinessAcceptsCreditCards",
        "BikeParking",
        "ByAppointmentOnly",
        "RestaurantsReservations",
        "RestaurantsGoodForGroups",
        "RestaurantsTakeOut",
        "GoodForKids",
        "HasTV",
        "Caters",
        "HappyHour",
        "OutdoorSeating",
        "RestaurantsDelivery",
        "RestaurantsTableService",
        "WheelchairAccessible",
        "DogsAllowed"
    ]

    for key in simple_attr_keys:
        b["attr_" + key] = attrs.apply(lambda d: attr_to_number(d, key)).astype("float32")

    attire_map = {"casual": 1.0, "dressy": 2.0, "formal": 3.0}
    alcohol_map = {"none": 0.0, "beer_and_wine": 1.0, "full_bar": 2.0}
    noise_map = {"quiet": 1.0, "average": 2.0, "loud": 3.0, "very_loud": 4.0}
    wifi_map = {"no": 0.0, "free": 1.0, "paid": 2.0}

    price_raw = attrs.apply(lambda d: d.get("RestaurantsPriceRange2", np.nan))
    b["attr_RestaurantsPriceRange2"] = pd.to_numeric(price_raw, errors="coerce").astype("float32")

    b["attr_RestaurantsAttire"] = attrs.apply(
        lambda d: map_attr_value(d.get("RestaurantsAttire", np.nan), attire_map)
    ).astype("float32")

    b["attr_Alcohol"] = attrs.apply(
        lambda d: map_attr_value(d.get("Alcohol", np.nan), alcohol_map)
    ).astype("float32")

    b["attr_NoiseLevel"] = attrs.apply(
        lambda d: map_attr_value(d.get("NoiseLevel", np.nan), noise_map)
    ).astype("float32")

    b["attr_WiFi"] = attrs.apply(
        lambda d: map_attr_value(d.get("WiFi", np.nan), wifi_map)
    ).astype("float32")

    def parse_hours(val):
        if pd.isna(val):
            return 0, 0.0

        try:
            h = ast.literal_eval(str(val))
            if not isinstance(h, dict):
                return 0, 0.0

            days_open = len(h)
            total_hours = 0.0

            for _, timerange in h.items():
                parts = str(timerange).split("-")
                if len(parts) != 2:
                    continue

                s1 = parts[0].split(":")
                s2 = parts[1].split(":")

                start = float(s1[0]) + float(s1[1]) / 60
                end = float(s2[0]) + float(s2[1]) / 60

                diff = end - start
                if diff < 0:
                    diff += 24

                total_hours += diff

            avg_hours = total_hours / days_open if days_open > 0 else 0.0
            return days_open, avg_hours

        except Exception:
            return 0, 0.0

    if "hours" in b.columns:
        hours_parsed = b["hours"].apply(parse_hours)
        b["days_open"] = hours_parsed.apply(lambda x: x[0]).astype("float32")
        b["avg_hours_per_day"] = hours_parsed.apply(lambda x: x[1]).astype("float32")
    else:
        b["days_open"] = 0.0
        b["avg_hours_per_day"] = 0.0

    keep_cols = [
        "business_id",
        "business_stars",
        "business_review_count",
        "log_business_review_count",
        "is_open",
        "latitude",
        "longitude",
        "category_count",
        "days_open",
        "avg_hours_per_day"
    ]

    keep_cols += list(category_keywords.keys())
    keep_cols += ["attr_" + key for key in simple_attr_keys]
    keep_cols += [
        "attr_RestaurantsPriceRange2",
        "attr_RestaurantsAttire",
        "attr_Alcohol",
        "attr_NoiseLevel",
        "attr_WiFi"
    ]

    keep_cols = [c for c in keep_cols if c in b.columns]

    return b[keep_cols]

## Clean and create user-level features from user metadata
This section transforms raw user information into structured user-level predictors for modeling. It removes duplicate users, converts numerical user statistics such as review count, funny votes, cool votes, fans, and average stars into usable numeric features, and optionally includes user useful votes depending on the leakage setting. It also creates behavioral features such as elite-year count, friend count, and the year the user started using Yelp. 

In [ ]:
# ============================================================
# 7. User Features
# ============================================================

def clean_user_features(user):
    u = user.copy()
    u = u.drop_duplicates(subset=["user_id"], keep="first")

    numeric_cols = ["review_count", "funny", "cool", "fans", "average_stars"]

    if USE_USER_USEFUL_FEATURES:
        numeric_cols.append("useful")

    for col in numeric_cols:
        if col in u.columns:
            u["user_" + col] = pd.to_numeric(u[col], errors="coerce").astype("float32")
        else:
            u["user_" + col] = np.nan

    if "elite" in u.columns:
        u["elite"] = u["elite"].fillna("").astype(str)
        u["elite"] = u["elite"].str.replace("20,20", "2020", regex=False)

        u["user_elite_count"] = u["elite"].apply(
            lambda x: 0 if x.strip() == "" or x.lower() in ["nan", "none"]
            else len([i for i in x.split(",") if i.strip()])
        ).astype("float32")
    else:
        u["user_elite_count"] = 0.0

    if "friends" in u.columns:
        u["friends"] = u["friends"].fillna("").astype(str)
        u["user_friend_count"] = u["friends"].apply(
            lambda x: 0 if x.strip() == "" or x.lower() in ["none", "nan"]
            else len([i for i in x.split(",") if i.strip()])
        ).astype("float32")
    else:
        u["user_friend_count"] = 0.0

    if "yelping_since" in u.columns:
        u["yelping_since"] = pd.to_datetime(u["yelping_since"], errors="coerce")
        u["yelping_since_year"] = u["yelping_since"].dt.year.astype("float32")
    else:
        u["yelping_since_year"] = np.nan

    keep_cols = ["user_id"]
    keep_cols += ["user_" + col for col in numeric_cols]
    keep_cols += [
        "user_elite_count",
        "user_friend_count",
        "yelping_since_year"
    ]

    keep_cols = [c for c in keep_cols if c in u.columns]

    return u[keep_cols]

## Aggregate tip-level features for businesses and users
This section summarizes tip data into business-level and user-level aggregated features. It extracts basic text characteristics from tips, such as text length and word count, incorporates compliment counts, and then groups the data separately by business and by user. For each business and user, it calculates total tip volume, average tip length, average tip word count, and total compliments received.

In [ ]:
# ============================================================
# 8. Tip Aggregation
# ============================================================

def aggregate_tip_features(tip):
    t = tip.copy()

    if "text" not in t.columns:
        t["text"] = ""

    t["text"] = t["text"].fillna("").astype(str)
    t["tip_text_len"] = t["text"].str.len().astype("float32")
    t["tip_word_count"] = t["text"].str.split().str.len().astype("float32")

    if "compliment_count" in t.columns:
        t["compliment_count"] = pd.to_numeric(
            t["compliment_count"], errors="coerce"
        ).fillna(0).astype("float32")
    else:
        t["compliment_count"] = 0.0

    tip_business = t.groupby("business_id", as_index=False).agg(
        business_tip_count=("text", "count"),
        business_tip_avg_len=("tip_text_len", "mean"),
        business_tip_avg_words=("tip_word_count", "mean"),
        business_tip_compliments=("compliment_count", "sum")
    )

    tip_user = t.groupby("user_id", as_index=False).agg(
        user_tip_count=("text", "count"),
        user_tip_avg_len=("tip_text_len", "mean"),
        user_tip_avg_words=("tip_word_count", "mean"),
        user_tip_compliments=("compliment_count", "sum")
    )

    for df in [tip_business, tip_user]:
        for c in df.columns:
            if c not in ["business_id", "user_id"]:
                df[c] = pd.to_numeric(df[c], errors="coerce").astype("float32")

    return tip_business, tip_user

## Build final feature tables for training and testing
This section combines all engineered features into the final training and testing feature tables. It first creates cleaned business features, user features, and aggregated tip features, then applies review-level feature engineering to both train and test datasets. After that, it merges business-level, user-level, business-tip, and user-tip features into the review data using business_id and user_id. Finally, it prints the table dimensions and total processing time to confirm the feature-building process.

In [ ]:
# ============================================================
# 9. Build Feature Tables
# ============================================================

t0 = time.time()

business_clean = clean_business_features(business)
user_clean = clean_user_features(user)
tip_business, tip_user = aggregate_tip_features(tip)

print("business_clean:", business_clean.shape)
print("user_clean:", user_clean.shape)
print("tip_business:", tip_business.shape)
print("tip_user:", tip_user.shape)

train_fe = add_review_features(train_x)
test_fe = add_review_features(test_x)

print("train_fe before merge:", train_fe.shape)
print("test_fe before merge:", test_fe.shape)

train_fe = train_fe.merge(business_clean, on="business_id", how="left")
test_fe = test_fe.merge(business_clean, on="business_id", how="left")

train_fe = train_fe.merge(user_clean, on="user_id", how="left")
test_fe = test_fe.merge(user_clean, on="user_id", how="left")

train_fe = train_fe.merge(tip_business, on="business_id", how="left")
test_fe = test_fe.merge(tip_business, on="business_id", how="left")

train_fe = train_fe.merge(tip_user, on="user_id", how="left")
test_fe = test_fe.merge(tip_user, on="user_id", how="left")

print("train_fe after merge:", train_fe.shape)
print("test_fe after merge:", test_fe.shape)
print("Feature table time:", (time.time() - t0) / 60, "minutes")

## Create interaction, ratio, and log-transformed features
This section adds advanced engineered features to capture relationships between review, user, and business information. It calculates rating differences between the review rating, business average rating, and user average rating, applies log transformations to reduce skewness in count-based variables, and creates per-review ratios such as user feedback per review and fans per review. It also builds interaction features, such as review length combined with user activity, business popularity, tip counts, and rating sentiment. These features help the model capture more complex patterns that simple individual variables may miss.

In [ ]:
# ============================================================
# 10. Interaction, Ratio, Log Features
# ============================================================

for df in [train_fe, test_fe]:

    df["star_diff_from_business"] = df["review_stars"] - df["business_stars"]
    df["user_star_diff"] = df["review_stars"] - df["user_average_stars"]

    df["abs_star_diff_from_business"] = df["star_diff_from_business"].abs()
    df["abs_user_star_diff"] = df["user_star_diff"].abs()

    df["log_user_review_count"] = np.log1p(df["user_review_count"].fillna(0).clip(lower=0))
    df["log_user_fans"] = np.log1p(df["user_fans"].fillna(0).clip(lower=0))
    df["log_user_friend_count"] = np.log1p(df["user_friend_count"].fillna(0).clip(lower=0))
    df["log_user_elite_count"] = np.log1p(df["user_elite_count"].fillna(0).clip(lower=0))

    if USE_USER_USEFUL_FEATURES and "user_useful" in df.columns:
        df["log_user_useful"] = np.log1p(df["user_useful"].fillna(0).clip(lower=0))
    else:
        df["user_useful"] = 0.0
        df["log_user_useful"] = 0.0

    if "user_funny" not in df.columns:
        df["user_funny"] = 0.0

    if "user_cool" not in df.columns:
        df["user_cool"] = 0.0

    df["log_user_funny"] = np.log1p(df["user_funny"].fillna(0).clip(lower=0))
    df["log_user_cool"] = np.log1p(df["user_cool"].fillna(0).clip(lower=0))

    df["user_total_feedback"] = df["user_useful"] + df["user_funny"] + df["user_cool"]
    df["log_user_total_feedback"] = np.log1p(df["user_total_feedback"].fillna(0).clip(lower=0))

    df["user_feedback_per_review"] = df["user_total_feedback"] / (df["user_review_count"] + 1)
    df["user_useful_per_review"] = df["user_useful"] / (df["user_review_count"] + 1)
    df["user_funny_per_review"] = df["user_funny"] / (df["user_review_count"] + 1)
    df["user_cool_per_review"] = df["user_cool"] / (df["user_review_count"] + 1)
    df["user_fans_per_review"] = df["user_fans"] / (df["user_review_count"] + 1)

    df["user_useful_share"] = df["user_useful"] / (df["user_total_feedback"] + 1)
    df["user_funny_share"] = df["user_funny"] / (df["user_total_feedback"] + 1)
    df["user_cool_share"] = df["user_cool"] / (df["user_total_feedback"] + 1)

    df["review_len_x_user_review_count"] = df["log_text_len"] * df["log_user_review_count"]
    df["review_len_x_user_useful"] = df["log_text_len"] * df["user_useful_per_review"]
    df["log_text_x_log_user_useful"] = df["log_text_len"] * df["log_user_useful"]
    df["log_text_x_user_feedback_per_review"] = df["log_text_len"] * df["user_feedback_per_review"]
    df["log_word_x_log_user_useful"] = df["log_word_count"] * df["log_user_useful"]

    df["business_popularity_x_star_diff"] = df["log_business_review_count"] * df["star_diff_from_business"]
    df["business_popularity_x_text_len"] = df["log_business_review_count"] * df["log_text_len"]
    df["log_text_x_log_business_reviews"] = df["log_text_len"] * df["log_business_review_count"]
    df["log_word_x_log_business_reviews"] = df["log_word_count"] * df["log_business_review_count"]

    df["negative_long_review"] = df["is_negative_star"] * df["log_text_len"]
    df["positive_long_review"] = df["is_positive_star"] * df["log_text_len"]
    df["extreme_long_review"] = df["is_extreme_star"] * df["log_text_len"]

    df["log_business_tip_count"] = np.log1p(df["business_tip_count"].fillna(0).clip(lower=0))
    df["log_user_tip_count"] = np.log1p(df["user_tip_count"].fillna(0).clip(lower=0))

    df["business_tip_x_text_len"] = df["log_business_tip_count"] * df["log_text_len"]
    df["user_tip_x_text_len"] = df["log_user_tip_count"] * df["log_text_len"]

train_fe = train_fe.loc[:, ~train_fe.columns.duplicated()]
test_fe = test_fe.loc[:, ~test_fe.columns.duplicated()]

print("train_fe after interaction:", train_fe.shape)
print("test_fe after interaction:", test_fe.shape) 

gc.collect()

## Create out-of-fold target encoding features
This section creates target encoding features for high-cardinality variables such as user_id, business_id, review year, and review stars. It uses out-of-fold encoding to reduce data leakage by calculating encoded values from training folds only, then applies smoothed full-data encoding to the test set. It also creates target-encoding interaction features with review length and user/business combinations. These features help the model capture historical usefulness patterns by user, business, rating, and time while controlling overfitting.

In [ ]:
# 11. Out-of-Fold Target Encoding (HIGHLY OPTIMIZED with Vectorization)

def add_oof_target_encoding(
        train_df,
        test_df,
        y,
        cols,
        n_splits=5,
        smoothing=30,
        random_state=42
):
    """
    Highly optimized out-of-fold target encoding using:
    - Integer-based hash keys (no string conversion)
    - Pure NumPy vectorized operations
    - Efficient groupby using pd.core.algorithms.factorize
    """
    y_arr = np.asarray(y, dtype="float32")
    global_mean = y_arr.mean() 

    if isinstance(cols, str):
        cols = [cols]

    enc_name = "te_" + "_".join(cols)

    # Create hash-based integer keys using pandas factorize (MUCH faster than string conversion)
    train_key_tuple = train_df[cols].apply(lambda x: tuple(x), axis=1)
    test_key_tuple = test_df[cols].apply(lambda x: tuple(x), axis=1)
    
    key_train, key_train_levels = pd.factorize(train_key_tuple)
    key_test, _ = pd.factorize(test_key_tuple)
    
    oof_values = np.full(len(train_df), global_mean, dtype="float32")
    
    skf = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state
    )

    # Build OOF encodings using efficient numpy operations
    for tr_idx, va_idx in skf.split(train_df, y_arr):
        fold_key_train = key_train[tr_idx]
        fold_y_train = y_arr[tr_idx]
        
        # Use pandas groupby for efficiency, then convert to dict
        fold_df = pd.DataFrame({"key": fold_key_train, "target": fold_y_train})
        stats = fold_df.groupby("key")["target"].agg(["mean", "size"])
        
        # Compute smoothed values
        smoothed = (stats["mean"] * stats["size"] + global_mean * smoothing) / (stats["size"] + smoothing)
        stats_dict = smoothed.to_dict()
        
        # Vectorized assignment: use np.vectorize or direct indexing
        val_keys = key_train[va_idx]
        for unique_key in np.unique(val_keys):
            if unique_key in stats_dict:
                mask = val_keys == unique_key
                oof_values[va_idx[mask]] = stats_dict[unique_key]

    # Compute full dataset encoding
    full_df = pd.DataFrame({"key": key_train, "target": y_arr})
    full_stats = full_df.groupby("key")["target"].agg(["mean", "size"])
    full_smoothed = (full_stats["mean"] * full_stats["size"] + global_mean * smoothing) / (full_stats["size"] + smoothing)
    full_stats_dict = full_smoothed.to_dict()
    
    # Assign results (no copy needed, work in-place)
    train_df[enc_name] = oof_values
    
    # Vectorized test encoding using numpy vectorize for speed
    test_encoding = np.array([full_stats_dict.get(k, global_mean) for k in key_test], dtype="float32")
    test_df[enc_name] = test_encoding

    print("Added target encoding:", enc_name)

    return train_df, test_df


y_series = pd.Series(y).reset_index(drop=True).astype(int)
train_fe = train_fe.reset_index(drop=True)
test_fe = test_fe.reset_index(drop=True)

single_te_cols = [
    "user_id",
    "business_id",
    "review_year",
    "review_stars"
]

for col in single_te_cols:
    if col in train_fe.columns and col in test_fe.columns:
        train_fe, test_fe = add_oof_target_encoding(
            train_fe,
            test_fe,
            y_series,
            cols=[col],
            n_splits=5,
            smoothing=30,
            random_state=RANDOM_STATE
        )

combo_te_cols = [
    ["business_id", "review_stars"],
    ["user_id", "review_stars"],
    ["business_id", "review_year"],
    ["user_id", "review_year"],
    ["review_year", "review_stars"]
]

for cols in combo_te_cols:
    if all(c in train_fe.columns for c in cols) and all(c in test_fe.columns for c in cols):
        train_fe, test_fe = add_oof_target_encoding(
            train_fe,
            test_fe,
            y_series,
            cols=cols,
            n_splits=5,
            smoothing=50,
            random_state=RANDOM_STATE
        )

for df in [train_fe, test_fe]:

    if "te_user_id" in df.columns:
        df["te_user_x_log_text_len"] = df["te_user_id"] * df["log_text_len"]

    if "te_business_id" in df.columns:
        df["te_business_x_log_text_len"] = df["te_business_id"] * df["log_text_len"]

    if "te_user_id" in df.columns and "te_business_id" in df.columns:
        df["te_user_minus_business"] = df["te_user_id"] - df["te_business_id"]
        df["te_user_x_business"] = df["te_user_id"] * df["te_business_id"]

    if "te_business_id_review_stars" in df.columns:
        df["te_business_star_x_log_text"] = df["te_business_id_review_stars"] * df["log_text_len"]

    if "te_user_id_review_stars" in df.columns:
        df["te_user_star_x_log_text"] = df["te_user_id_review_stars"] * df["log_text_len"]

te_cols = [c for c in train_fe.columns if c.startswith("te_")]

print("Number of TE columns:", len(te_cols))
print("train_fe after TE:", train_fe.shape)
print("test_fe after TE:", test_fe.shape)

gc.collect()

## Select final numeric features for modeling

In [ ]:
# 12. Select Numeric Features

drop_cols = [
    "review_id",
    "user_id",
    "business_id",
    "text",
    "date",
    "stars"
]

feature_cols = [c for c in train_fe.columns if c not in drop_cols]

X = train_fe[feature_cols].select_dtypes(include=["number", "bool"]).copy()
X_test_final = test_fe[X.columns].copy()

X = X.replace([np.inf, -np.inf], np.nan)
X_test_final = X_test_final.replace([np.inf, -np.inf], np.nan)

X = X.astype("float32")
X_test_final = X_test_final.astype("float32")

print("Number of features:", X.shape[1])
print("X:", X.shape)
print("X_test_final:", X_test_final.shape)

feature_list_path = os.path.join(OUTPUT_DIR, "super_final_feature_list.csv")
pd.DataFrame({"feature": X.columns}).to_csv(feature_list_path, index=False)
print("Saved feature list:", feature_list_path)

## Split data into Train and Validation sets

In [ ]:
# 13. Train / Validation Split

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("y_train positive rate:", y_train.mean())
print("y_val positive rate:", y_val.mean())

gc.collect()

In [ ]:
# ============================================================ 
# 14. Baseline Logistic Regression Model
# ============================================================

baseline_logreg_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        C=1.0,
        penalty="l2",
        class_weight="balanced",
        solver="saga",
        max_iter=1000,
        n_jobs=N_JOBS,
        random_state=RANDOM_STATE
    ))
])

print("Training baseline logistic regression...")
baseline_logreg_model.fit(X_train, y_train)

baseline_val_score = baseline_logreg_model.predict_proba(X_val)[:, 1]
baseline_test_score = baseline_logreg_model.predict_proba(X_test_final)[:, 1]

print("Baseline logistic regression training complete.")
print("Validation scores shape:", baseline_val_score.shape)
print("Test scores shape:", baseline_test_score.shape)

gc.collect()


## Evaluate ROC, AUC, and TPR at 10% FPR


In [ ]:
# ============================================================
# 15. Baseline ROC Curve and TPR at 10% FPR
# ============================================================

baseline_metrics = best_tpr_under_fpr(
    y_val,
    baseline_val_score,
    fpr_limit=FPR_LIMIT
)

print_metrics("BASELINE LOGISTIC REGRESSION VALIDATION METRICS", baseline_metrics)

roc_curve_path = plot_roc_with_operating_point(
    y_true=y_val,
    y_score=baseline_val_score,
    metrics=baseline_metrics,
    output_dir=OUTPUT_DIR,
    filename="baseline_logreg_roc_curve.png"
)

baseline_metrics_df = pd.DataFrame([{
    "model": "Baseline Logistic Regression",
    "auc": baseline_metrics["auc"],
    "fpr_limit": FPR_LIMIT,
    "fpr_at_threshold": baseline_metrics["fpr"],
    "tpr_at_10pct_fpr": baseline_metrics["tpr"],
    "threshold": baseline_metrics["threshold"],
    "accuracy": baseline_metrics["accuracy"],
    "tn": baseline_metrics["tn"],
    "fp": baseline_metrics["fp"],
    "fn": baseline_metrics["fn"],
    "tp": baseline_metrics["tp"],
    "roc_curve_path": roc_curve_path
}])

baseline_metrics_path = os.path.join(OUTPUT_DIR, "baseline_logreg_validation_metrics.csv")
baseline_metrics_df.to_csv(baseline_metrics_path, index=False)

print("Saved baseline metrics:", baseline_metrics_path)
print(baseline_metrics_df)


## Export baseline logistic regression submission


In [ ]:
# ============================================================
# 16. Export Baseline Logistic Regression Submission
# ============================================================

baseline_threshold = baseline_metrics["threshold"]
baseline_test_pred = (baseline_test_score >= baseline_threshold).astype(int)

baseline_submission = pd.DataFrame({
    "top_useful": baseline_test_pred
})

baseline_submission_path = os.path.join(OUTPUT_DIR, "group_7_yelp.csv")
baseline_submission.to_csv(
    baseline_submission_path,
    index=False,
    header=False
)

baseline_score_path = os.path.join(OUTPUT_DIR, "baseline_logreg_test_scores.csv")
pd.DataFrame({
    "baseline_logreg_score": baseline_test_score,
    "prediction": baseline_test_pred
}).to_csv(baseline_score_path, index=False)

baseline_summary = {
    "model": "Baseline Logistic Regression",
    "validation_auc": baseline_metrics["auc"],
    "validation_fpr": baseline_metrics["fpr"],
    "validation_tpr_at_10pct_fpr": baseline_metrics["tpr"],
    "threshold": baseline_threshold,
    "positive_predictions": int(baseline_test_pred.sum()),
    "positive_ratio": float(baseline_test_pred.mean())
}

baseline_summary_path = os.path.join(OUTPUT_DIR, "baseline_logreg_final_summary.csv")
pd.DataFrame([baseline_summary]).to_csv(baseline_summary_path, index=False)

print("Submission:", baseline_submission_path)
print("Scores:", baseline_score_path)
print("Summary:", baseline_summary_path)
print("Positive predictions:", baseline_test_pred.sum())
print("Positive ratio:", baseline_test_pred.mean())
print("Validation FPR:", baseline_metrics["fpr"])
print("Validation TPR at 10% FPR:", baseline_metrics["tpr"])


## 8. Feature Engineering Shared by Random Forest and LightGBM

Random Forest and the standalone LightGBM comparison use a separate feature-engineering path. This avoids accidentally changing the final blend feature path while still keeping these comparison models runnable.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve
)
from sklearn.model_selection import learning_curve, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder


In [ ]:
business = pd.read_csv(csv_path(BUSINESS_CLEAN_FILE, [BUSINESS_FILE]))
user = pd.read_csv(csv_path(USER_CLEAN_FILE, [USER_FILE]))
tip = pd.read_csv(csv_path(TIP_CLEAN_FILE, [TIP_FILE]))

In [ ]:
# Loading train and test datasets:
test_x = pd.read_csv(csv_path(TEST_X_FILE))
train_x = pd.read_csv(csv_path(TRAIN_X_FILE))
train_y = pd.read_csv(csv_path(TRAIN_Y_FILE))

In [ ]:
train_x.info()

In [ ]:
train_y.info()

In [ ]:
test_x.info()

In [ ]:
user = user.rename(columns={
    "review_count": "user_review_count",
    "fans": "user_fans",
    "average_stars": "user_avg_stars",
    "friends_count": "user_friends_count",
    "is_elite": "user_is_elite",
    "elite_years": "user_elite_years"
})


business = business.rename(columns={
    "review_count": "business_review_count",
    "stars": "business_stars",
    "is_open": "business_is_open"
})

tip = tip.rename(columns={
    "compliment_count": "tip_compliment",
    "text": "tip_text",
    "date": "tip_date"
})

In [ ]:
null_columns = ['BusinessAcceptsCreditCards','BikeParking','ByAppointmentOnly','RestaurantsReservations','HasTV','Caters',
'RestaurantsTakeOut',
'HappyHour',
'OutdoorSeating',
'RestaurantsDelivery',
'BusinessAcceptsBitcoin',
'WheelchairAccessible',
'DogsAllowed',
'Corkage',
'DriveThru',
'CoatCheck',
'BYOB',
'BusinessParking',
'GoodForMeal','Ambience','Music','HairSpecializesIn','BestNights','RestaurantsAttire','Alcohol','AgesAllowed','BYOBCorkage','Smoking','WiFi','NoiseLevel']

In [ ]:
business['RestaurantsAttire'] = business['RestaurantsAttire'].replace(False, 'Unknown')
business['Alcohol'] = business['Alcohol'].replace(False, 'Unknown')
business['AgesAllowed'] = business['AgesAllowed'].replace(False, 'Unknown')
business['BYOBCorkage'] = business['BYOBCorkage'].replace(False, 'No')
business['Smoking'] = business['Smoking'].replace(False,'No')
business['WiFi']= business['WiFi'].replace(False, 'No')
business['WiFi'] =business['WiFi'].replace('Paid','paid')
business['NoiseLevel'] = business['NoiseLevel'].replace(False,'Unknown')

In [ ]:
bool_cols = business.select_dtypes(include=['bool']).columns

business[bool_cols] = business[bool_cols].astype('int64')

In [ ]:
object_cols= business.select_dtypes(include=['object']).columns

In [ ]:
category_cols = ['RestaurantsAttire', 'Alcohol', 'NoiseLevel','WiFi', 'Smoking', 'BYOBCorkage', 'AgesAllowed']


business[category_cols] = business[category_cols].astype('category')

In [ ]:
user['elite_year_count'] = user['elite'].apply(lambda x: len(str(x).split(',')) if str(x).strip() not in ('', 'nan', 'None', 'NaT') else 0)

In [ ]:
user['elite_year_count'].value_counts()

In [ ]:
bins = [0, 1, 3, 6, 10, float('inf')]
labels = ['0', '1-2', '3-5', '6-9', '10+']

user['elite_year_bin'] = pd.cut(user['elite_year_count'], bins=bins, labels=labels, right=False)

In [ ]:
user['elite_year_bin'].value_counts()

In [ ]:
user['yelping_since'] = pd.to_datetime(user['yelping_since']).dt.date

In [ ]:
bins = [0, 25, 50, 100, 200, 500]
labels = ['very_short', 'short', 'medium', 'long', 'very_long']

tip['text_len_bin'] = pd.cut(tip['tip_text'].str.len(), bins=bins, labels=labels,include_lowest=True)

tip['text_len_bin'] = tip['text_len_bin'].cat.add_categories('unknown').fillna('unknown')

In [ ]:
tip['text_len_bin'].value_counts()

In [ ]:
tip['tip_date'] = pd.to_datetime(tip['tip_date']).dt.date

In [ ]:
tip.head()

In [ ]:
train = pd.concat([train_x,train_y],axis=1)

In [ ]:
train = train.merge(user, on="user_id", how="left")
test_x = test_x.merge(user, on="user_id", how="left")

train = train.merge(business, on="business_id", how="left")
test_x = test_x.merge(business, on="business_id", how="left")

In [ ]:
tip_biz = tip.groupby('business_id').agg(
    tip_count           = ('tip_text', 'count'),      
    tip_avg_compliments = ('tip_compliment', 'mean')
).reset_index()

print(f"tip_biz shape: {tip_biz.shape}")  # should be <= 300,845 unique businesses
print(tip_biz.head())

In [ ]:
train = train.merge(tip_biz, on='business_id', how='left')
test_x  = test_x.merge(tip_biz,  on='business_id', how='left')

# Fill NaNs (businesses with no tips)
train[['tip_count', 'tip_avg_compliments']] = train[['tip_count', 'tip_avg_compliments']].fillna(0)
test_x[['tip_count', 'tip_avg_compliments']]  = test_x[['tip_count', 'tip_avg_compliments']].fillna(0)

# Verify
print(train[['tip_count', 'tip_avg_compliments']].isna().sum())
print(test_x[['tip_count', 'tip_avg_compliments']].isna().sum())

In [ ]:
bins = [0, 100, 250, 500, 1000, float('inf')]
labels = ['very_short', 'short', 'medium', 'long', 'very_long']

train['text_len_bin'] = pd.cut(train['text'].str.len(), bins=bins, labels=labels, include_lowest=True)

In [ ]:
train[train['user_review_count'].isna()]

Need to make sure these two rows have all the data replaced with 0 or unknown (cant drop it because it will cause an imbalance between train_x and train_y)

In [ ]:
print(train.isnull().sum()[train.isnull().sum()>0])

In [ ]:
train = train.drop(columns = ['elite','friends','attributes','hours','categories','useful'], axis=1, errors='ignore')

### TEST MODIFICATION ###

In [ ]:
bins = [0, 100, 250, 500, 1000, float('inf')]
labels = ['very_short', 'short', 'medium', 'long', 'very_long']

test_x['text_len_bin'] = pd.cut(test_x['text'].str.len(), bins=bins, labels=labels, include_lowest=True)

test_x['text_len_bin'] = test_x['text_len_bin'].cat.add_categories('unknown').fillna('unknown')

In [ ]:
test_x = test_x.drop(columns = ['elite','friends','attributes','hours','useful','categories'], axis=1, errors='ignore')

### FINAL CHECK ###

In [ ]:
train.head()

In [ ]:
test_x.head()

### Adding Time dimensionality to the Random Forest ###


date feature engineering

In [ ]:

train['date'] = pd.to_datetime(train['date'])
test_x['date'] = pd.to_datetime(test_x['date'])

reference_date = pd.Timestamp('2022-12-31')

train['days_since_review'] = (reference_date - train['date']).dt.days
test_x['days_since_review'] = (reference_date - test_x['date']).dt.days

# Year and month
train['review_year'] = train['date'].dt.year
train['review_month'] = train['date'].dt.month

test_x['review_year'] = test_x['date'].dt.year
test_x['review_month'] = test_x['date'].dt.month

In [ ]:
train['days_since_review'].value_counts()

In [ ]:
train['review_month'].value_counts()

In [ ]:
train['review_year'].value_counts()

#### Yelping Since Feature Engineering


In [ ]:
# Convert to datetime
train['yelping_since'] = pd.to_datetime(train['yelping_since'])
test_x['yelping_since'] = pd.to_datetime(test_x['yelping_since'])

reference_date = pd.Timestamp('2022-12-31')  # or use train['date'].max()

# How long the user has been on Yelp (in days)
train['user_tenure_days'] = (reference_date - train['yelping_since']).dt.days
test_x['user_tenure_days'] = (reference_date - test_x['yelping_since']).dt.days

# Year they joined — captures Yelp's early adopter vs late joiner dynamic
train['yelping_since_year'] = train['yelping_since'].dt.year
test_x['yelping_since_year'] = test_x['yelping_since'].dt.year

# Drop the raw column
train = train.drop(columns=['yelping_since'])
test_x = test_x.drop(columns=['yelping_since'])

In [ ]:
train['user_tenure_days'].value_counts()

In [ ]:
### Random Forest Classifier ###

In [ ]:
# Set plotting defaults
plt.rc('axes', titlesize=18, labelsize=16)
plt.rc('xtick', labelsize=13)
plt.rc('ytick', labelsize=13)
plt.rc('legend', fontsize=12, title_fontsize=12)

In [ ]:
# Create train/validation split

drop_cols = [
    'review_id', 'user_id', 'business_id', 'text', 'address',
    'city', 'state', 'postal_code', 'latitude', 'longitude',
    'date', 'top_useful'
]

y = train['top_useful'].astype(int)
X = train.drop(columns=drop_cols, errors='ignore')

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"X_train: {X_train.shape}")
print(f"X_val:   {X_val.shape}")
print("\nTraining class balance:")
print(y_train.value_counts(normalize=True))
print("\nValidation class balance:")
print(y_val.value_counts(normalize=True))

In [ ]:
# Build preprocessing pipeline
# - Numeric columns: median imputation
# - Categorical columns: most-frequent imputation + one-hot encoding
# Keeping preprocessing inside a Pipeline prevents train/validation leakage.

numeric_cols = X_train.select_dtypes(include=['number', 'bool']).columns.tolist()
categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=True))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ],
    remainder='drop'
)

print(f"Numeric columns: {len(numeric_cols)}")
print(f"Categorical columns: {len(categorical_cols)}")
print("Categorical columns:", categorical_cols)

## 9. Model 3: Random Forest

This section trains a Random Forest comparison model using the tree-based feature matrix. It includes validation performance, ROC analysis, selected-threshold evaluation, test prediction, and feature importance output.


In [ ]:
# Basic Random Forest model
# This replaces RandomizedSearchCV with one clear baseline Random Forest.

rf_model = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('model', RandomForestClassifier(
        n_estimators=100,
        max_depth=None,
        min_samples_split=10,
        min_samples_leaf=1,
        max_features='sqrt',
        class_weight='balanced',
        n_jobs=-1,
        random_state=42
    ))
])

rf_model.fit(X_train, y_train)

print("Random Forest model trained successfully.")

### Validation Performance ###

In [ ]:
# Basic validation metrics using the default 0.50 probability threshold

y_val_pred = rf_model.predict(X_val)
y_val_proba = rf_model.predict_proba(X_val)[:, 1]

accuracy = accuracy_score(y_val, y_val_pred)
auc = roc_auc_score(y_val, y_val_proba)

tn, fp, fn, tp = confusion_matrix(y_val, y_val_pred).ravel()
tpr = tp / (tp + fn)
fpr = fp / (fp + tn)

print(f"Validation Accuracy: {accuracy:.4f}")
print(f"Validation AUC:      {auc:.4f}")
print(f"TPR / Recall:        {tpr:.4f}")
print(f"FPR:                 {fpr:.4f}")

print("\nClassification Report:")
print(classification_report(y_val, y_val_pred))

### Learning Curve ###

In [ ]:
# Learning curve
# To keep runtime reasonable on large data, this uses a stratified sample capped at 100,000 rows.
# Increase max_lc_rows if your machine can handle more.

max_lc_rows = 100_000

if len(X_train) > max_lc_rows:
    _, X_lc, _, y_lc = train_test_split(
        X_train,
        y_train,
        test_size=max_lc_rows,
        random_state=42,
        stratify=y_train
    )
else:
    X_lc = X_train
    y_lc = y_train

train_sizes, train_scores, val_scores = learning_curve(
    estimator=rf_model,
    X=X_lc,
    y=y_lc,
    train_sizes=np.linspace(0.1, 1.0, 5),
    cv=3,
    scoring='roc_auc',
    n_jobs=-1,
    shuffle=True,
    random_state=42
)

train_mean = train_scores.mean(axis=1)
train_std = train_scores.std(axis=1)
val_mean = val_scores.mean(axis=1)
val_std = val_scores.std(axis=1)

plt.figure(figsize=(8, 6))
plt.plot(train_sizes, train_mean, marker='o', label='Training ROC AUC')
plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.2)

plt.plot(train_sizes, val_mean, marker='o', label='Cross-validation ROC AUC')
plt.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.2)

plt.xlabel('Training Set Size')
plt.ylabel('ROC AUC')
plt.title('Learning Curve - Random Forest')
plt.legend()
plt.grid(True)
plt.show()

learning_curve_results = pd.DataFrame({
    'train_size': train_sizes,
    'train_auc_mean': train_mean,
    'train_auc_std': train_std,
    'cv_auc_mean': val_mean,
    'cv_auc_std': val_std
})

learning_curve_results

### ROC Curve and Threshold Selection ###

In [ ]:
# ROC curve on the validation set

fpr_values, tpr_values, thresholds = roc_curve(y_val, y_val_proba)
auc = roc_auc_score(y_val, y_val_proba)

print(f"Validation ROC AUC: {auc:.4f}")

# Optional threshold rule:
# Choose the threshold with the highest TPR while keeping FPR below 0.10.
target_fpr = 0.10
valid_mask = fpr_values <= target_fpr

if valid_mask.any():
    best_idx_within_mask = np.argmax(tpr_values[valid_mask])
    best_threshold = thresholds[valid_mask][best_idx_within_mask]
    best_tpr = tpr_values[valid_mask][best_idx_within_mask]
    best_fpr = fpr_values[valid_mask][best_idx_within_mask]
else:
    # Fallback: You should rarely hit this, but it prevents the notebook from breaking.
    best_idx = np.argmax(tpr_values - fpr_values)
    best_threshold = thresholds[best_idx]
    best_tpr = tpr_values[best_idx]
    best_fpr = fpr_values[best_idx]

print(f"Chosen threshold: {best_threshold:.4f}")
print(f"TPR at threshold: {best_tpr:.4f}")
print(f"FPR at threshold: {best_fpr:.4f}")

plt.figure(figsize=(8, 6))
plt.plot(fpr_values, tpr_values, label=f'ROC Curve (AUC = {auc:.3f})')
plt.plot([0, 1], [0, 1], linestyle='--', label='Random Classifier')
plt.axvline(x=target_fpr, linestyle='--', label=f'FPR = {target_fpr:.2f} limit')
plt.scatter([best_fpr], [best_tpr], s=80, label=f'Threshold = {best_threshold:.3f}')

plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Random Forest')
plt.legend()
plt.grid(True)
plt.show()

### Evaluation at Chosen Threshold ###

In [ ]:
# Apply the selected threshold and evaluate again

y_val_pred_threshold = (y_val_proba >= best_threshold).astype(int)

tn, fp, fn, tp = confusion_matrix(y_val, y_val_pred_threshold).ravel()
tpr_final = tp / (tp + fn)
fpr_final = fp / (fp + tn)
acc_final = accuracy_score(y_val, y_val_pred_threshold)

print(f"=== Validation Metrics at threshold={best_threshold:.4f} ===")
print(f"Accuracy:      {acc_final:.4f}")
print(f"TPR / Recall:  {tpr_final:.4f}")
print(f"FPR:           {fpr_final:.4f}")

print("\nClassification Report:")
print(classification_report(y_val, y_val_pred_threshold))

### Prediction on test_x ###

In [ ]:
# Prepare test data with the same columns used during training

test_drop_cols = [
    'review_id', 'user_id', 'business_id', 'text', 'date', 'address',
    'city', 'state', 'postal_code', 'latitude', 'longitude', 'top_useful'
]

X_test = test_x.drop(columns=test_drop_cols, errors='ignore')

# Match validation/training feature columns exactly.
# Any missing columns are added as NaN and handled by the preprocessing pipeline.
X_test = X_test.reindex(columns=X.columns, fill_value=np.nan)

print(f"X_test: {X_test.shape}")

In [ ]:
# Predict probabilities and apply the selected threshold

test_proba = rf_model.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= best_threshold).astype(int)

output = pd.DataFrame({
    'predicted_top_useful': test_pred,
    'probability': test_proba
})

output.to_csv(os.path.join(OUTPUT_DIR, 'test_predictions.csv'), index=False)

print(output['predicted_top_useful'].value_counts())
print("\nPredictions saved to test_predictions.csv")

### Feature Importance ###

In [ ]:
# Top Random Forest feature importances

feature_names = rf_model.named_steps['preprocess'].get_feature_names_out()
importances = pd.Series(
    rf_model.named_steps['model'].feature_importances_,
    index=feature_names
).sort_values(ascending=False)

importances.head(20)

## 9b. Model 7: LightGBM

This section was completed from the standalone LightGBM notebook. It trains LightGBM with the original intended parameters, uses out-of-fold validation predictions, and reports the best TPR subject to FPR ≤ 0.10.


In [ ]:

# ============================================================
# Model 7: LightGBM completed from standalone light-gbm.ipynb
# ============================================================
# This block reloads and rebuilds the LightGBM-specific feature table so it does
# not depend on variables created by the Random Forest section and does not

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, roc_curve
import lightgbm as lgb

# -----------------------------
# Load LightGBM source datasets
# -----------------------------
lgbm_business = pd.read_csv(csv_path(BUSINESS_CLEAN_FILE, [BUSINESS_FILE]))
lgbm_user = pd.read_csv(csv_path(USER_CLEAN_FILE, [USER_FILE]))
lgbm_tip = pd.read_csv(csv_path(TIP_CLEAN_FILE, [TIP_FILE]))
lgbm_test_x_raw = pd.read_csv(csv_path(TEST_X_FILE))
lgbm_train_x_raw = pd.read_csv(csv_path(TRAIN_X_FILE))
lgbm_train_y_raw = pd.read_csv(csv_path(TRAIN_Y_FILE))

print("LightGBM raw shapes:")
print("  train_x:", lgbm_train_x_raw.shape)
print("  train_y:", lgbm_train_y_raw.shape)
print("  test_x :", lgbm_test_x_raw.shape)
print("  business:", lgbm_business.shape)
print("  user:", lgbm_user.shape)
print("  tip:", lgbm_tip.shape)

# -----------------------------
# Column renaming from standalone LightGBM notebook
# -----------------------------
lgbm_user = lgbm_user.rename(columns={
    "review_count": "user_review_count",
    "fans": "user_fans",
    "average_stars": "user_avg_stars",
    "friends_count": "user_friends_count",
    "useful": "user_useful",
    "funny": "user_funny",
    "cool": "user_cool"
})

lgbm_business = lgbm_business.rename(columns={
    "review_count": "business_review_count",
    "stars": "business_stars",
    "is_open": "business_is_open"
})

lgbm_tip = lgbm_tip.rename(columns={
    "compliment_count": "tip_compliment",
    "text": "tip_text",
    "date": "tip_date"
})

# -----------------------------
# Business feature engineering
# -----------------------------
lgbm_null_columns = [
    'BusinessAcceptsCreditCards','BikeParking','ByAppointmentOnly',
    'RestaurantsReservations','HasTV','Caters','RestaurantsTakeOut','HappyHour',
    'OutdoorSeating','BusinessAcceptsBitcoin','DogsAllowed','GoodForKids','CoatCheck',
    'RestaurantsGoodForGroups','WheelchairAccessible','RestaurantsDelivery','GoodForDancing',
    'RestaurantsTableService','GoodForMeal_dessert','GoodForMeal_latenight','GoodForMeal_lunch',
    'GoodForMeal_dinner','GoodForMeal_brunch','GoodForMeal_breakfast','Music_dj','Music_background_music',
    'Music_no_music','Music_karaoke','Music_live','Music_video','Music_jukebox','BestNights_monday',
    'BestNights_tuesday','BestNights_friday','BestNights_wednesday','BestNights_thursday','BestNights_sunday',
    'BestNights_saturday','Ambience_romantic','Ambience_intimate','Ambience_classy','Ambience_hipster',
    'Ambience_divey','Ambience_touristy','Ambience_trendy','Ambience_upscale','Ambience_casual'
]
lgbm_existing_null_columns = [c for c in lgbm_null_columns if c in lgbm_business.columns]
lgbm_business[lgbm_existing_null_columns] = lgbm_business[lgbm_existing_null_columns].fillna('Unknown')

if 'WiFi' in lgbm_business.columns:
    lgbm_business['WiFi'] = lgbm_business['WiFi'].replace('Paid', 'paid')

lgbm_bool_cols = lgbm_business.select_dtypes(include=['bool']).columns
lgbm_business[lgbm_bool_cols] = lgbm_business[lgbm_bool_cols].astype('int64')

lgbm_cat_cols = [
    'BusinessAcceptsCreditCards', 'BikeParking', 'ByAppointmentOnly',
    'RestaurantsReservations', 'RestaurantsAttire', 'Alcohol', 'HasTV', 'Caters',
    'RestaurantsTakeOut', 'HappyHour', 'OutdoorSeating', 'BusinessAcceptsBitcoin',
    'DogsAllowed', 'GoodForKids', 'CoatCheck', 'RestaurantsGoodForGroups',
    'WheelchairAccessible', 'RestaurantsDelivery', 'NoiseLevel', 'WiFi', 'Smoking',
    'GoodForDancing', 'RestaurantsTableService', 'BYOBCorkage', 'AgesAllowed'
]
lgbm_existing_cat_cols = [c for c in lgbm_cat_cols if c in lgbm_business.columns]
lgbm_business[lgbm_existing_cat_cols] = lgbm_business[lgbm_existing_cat_cols].astype('category')

# -----------------------------
# User feature engineering
# -----------------------------
if 'elite' in lgbm_user.columns:
    lgbm_user['elite_year_count'] = lgbm_user['elite'].apply(
        lambda x: len(str(x).split(',')) if str(x).strip() not in ('', 'nan', 'None', 'NaT') else 0
    )
else:
    lgbm_user['elite_year_count'] = 0

lgbm_elite_bins = [0, 1, 3, 6, 10, float('inf')]
lgbm_elite_labels = ['Not Elite', '1-2 Years', '3-5 Years', '6-9 Years', '10+ Years']
lgbm_user['elite_year_bin'] = pd.cut(
    lgbm_user['elite_year_count'],
    bins=lgbm_elite_bins,
    labels=lgbm_elite_labels,
    include_lowest=True
)

# -----------------------------
# Merge train/test together first, matching the standalone notebook order
# -----------------------------
lgbm_train_x_raw = lgbm_train_x_raw.copy()
lgbm_test_x_raw = lgbm_test_x_raw.copy()
lgbm_train_x_raw["is_train"] = 1
lgbm_test_x_raw["is_train"] = 0

lgbm_combined = pd.concat([lgbm_train_x_raw, lgbm_test_x_raw], axis=0).reset_index(drop=True)
print("LightGBM combined before merges:", lgbm_combined.shape)

lgbm_combined = lgbm_combined.merge(lgbm_user, on="user_id", how="left")
lgbm_combined = lgbm_combined.merge(lgbm_business, on="business_id", how="left")
print("LightGBM combined after user/business merges:", lgbm_combined.shape)

# -----------------------------
# Tip aggregation from standalone LightGBM notebook
# -----------------------------
if {'business_id', 'tip_text', 'tip_compliment'}.issubset(lgbm_tip.columns):
    lgbm_tip_aggregate = lgbm_tip.groupby('business_id').agg(
        tip_count=('tip_text', 'count'),
        tip_avg_compliments=('tip_compliment', 'mean')
    ).reset_index()
else:
    raise KeyError("LightGBM tip data must contain business_id, tip_text, and tip_compliment after renaming.")

print("LightGBM tip_aggregate shape:", lgbm_tip_aggregate.shape)
lgbm_combined = lgbm_combined.merge(lgbm_tip_aggregate, on='business_id', how='left')
lgbm_combined[['tip_count', 'tip_avg_compliments']] = lgbm_combined[['tip_count', 'tip_avg_compliments']].fillna(0)
print("LightGBM combined after tip merge:", lgbm_combined.shape)

# -----------------------------
# Review text length and time features
# -----------------------------
lgbm_text_len_bins = [0, 100, 250, 500, 1000, float('inf')]
lgbm_text_len_labels = ['very_short', 'short', 'medium', 'long', 'very_long']
lgbm_combined['text_len_bin'] = pd.cut(
    lgbm_combined['text'].str.len(),
    bins=lgbm_text_len_bins,
    labels=lgbm_text_len_labels,
    include_lowest=True
)

lgbm_combined['date'] = pd.to_datetime(lgbm_combined['date'])
lgbm_combined['yelping_since'] = pd.to_datetime(lgbm_combined['yelping_since'])
lgbm_combined['review_year'] = lgbm_combined['date'].dt.year
lgbm_combined['review_month'] = lgbm_combined['date'].dt.month
lgbm_combined['review_dayofweek'] = lgbm_combined['date'].dt.dayofweek
lgbm_combined['review_is_weekend'] = lgbm_combined['review_dayofweek'].isin([5, 6]).astype(int)

lgbm_reference_date = pd.Timestamp('2022-12-31')
lgbm_combined['days_since_review'] = (lgbm_reference_date - lgbm_combined['date']).dt.days
lgbm_combined['user_tenure_days'] = (lgbm_combined['date'] - lgbm_combined['yelping_since']).dt.days

# -----------------------------
# Split back into LightGBM train/test
# -----------------------------
lgbm_train_x_fe = lgbm_combined[lgbm_combined['is_train'] == 1].drop(columns=['is_train']).reset_index(drop=True)
lgbm_test_x_fe = lgbm_combined[lgbm_combined['is_train'] == 0].drop(columns=['is_train']).reset_index(drop=True)

if lgbm_train_y_raw.shape[1] == 1:
    y_lgbm = lgbm_train_y_raw.iloc[:, 0].astype(int).reset_index(drop=True).values
else:
    y_lgbm = lgbm_train_y_raw['top_useful'].astype(int).reset_index(drop=True).values

lgbm_train_full = pd.concat([lgbm_train_x_fe, pd.Series(y_lgbm, name='top_useful')], axis=1)

lgbm_cols_to_drop = [
    'review_id', 'user_id', 'business_id', 'text', 'date', 'yelping_since', 'elite', 'categories'
]
lgbm_existing_drop_train = [c for c in lgbm_cols_to_drop + ['top_useful'] if c in lgbm_train_full.columns]
lgbm_existing_drop_test = [c for c in lgbm_cols_to_drop if c in lgbm_test_x_fe.columns]

X_train_lgbm_full = lgbm_train_full.drop(columns=lgbm_existing_drop_train)
X_test_lgbm_full = lgbm_test_x_fe.drop(columns=lgbm_existing_drop_test)

# Match standalone notebook categorical handling.
for col in ['city', 'state']:
    if col in X_train_lgbm_full.columns:
        X_train_lgbm_full[col] = X_train_lgbm_full[col].astype('category')
    if col in X_test_lgbm_full.columns:
        X_test_lgbm_full[col] = X_test_lgbm_full[col].astype('category')

X_train_lgbm_full = X_train_lgbm_full.select_dtypes(include=[np.number, 'bool', 'category'])
X_test_lgbm_full = X_test_lgbm_full.select_dtypes(include=[np.number, 'bool', 'category'])

# Ensure test columns exactly match train columns, preserving train feature order.
for col in X_train_lgbm_full.columns:
    if col not in X_test_lgbm_full.columns:
        X_test_lgbm_full[col] = np.nan
X_test_lgbm_full = X_test_lgbm_full[X_train_lgbm_full.columns]

# Downcast numeric columns as in the standalone LightGBM notebook.
for col in X_train_lgbm_full.select_dtypes(include=['float64']).columns:
    X_train_lgbm_full[col] = X_train_lgbm_full[col].astype('float32')
    X_test_lgbm_full[col] = X_test_lgbm_full[col].astype('float32')
for col in X_train_lgbm_full.select_dtypes(include=['int64']).columns:
    X_train_lgbm_full[col] = X_train_lgbm_full[col].astype('int32')
    X_test_lgbm_full[col] = X_test_lgbm_full[col].astype('int32')

print("LightGBM modeling shapes:")
print("  X_train_lgbm_full:", X_train_lgbm_full.shape)
print("  X_test_lgbm_full :", X_test_lgbm_full.shape)
print("  y_lgbm           :", y_lgbm.shape)
print("  LightGBM feature count:", X_train_lgbm_full.shape[1])
print("  Missing values in train:", int(X_train_lgbm_full.isna().sum().sum()))
print("  Missing values in test :", int(X_test_lgbm_full.isna().sum().sum()))

assert len(X_train_lgbm_full) == len(y_lgbm), "LightGBM X/y row counts do not match."
assert list(X_train_lgbm_full.columns) == list(X_test_lgbm_full.columns), "LightGBM train/test feature columns are not aligned."

# -----------------------------
# 5-fold LightGBM CV from standalone notebook, corrected to run
# -----------------------------
n_splits_lgbm = 5
skf_lgbm = StratifiedKFold(n_splits=n_splits_lgbm, shuffle=True, random_state=RANDOM_STATE)

lgbm_oof_preds = np.zeros(len(X_train_lgbm_full), dtype='float32')
lgbm_test_preds = np.zeros(len(X_test_lgbm_full), dtype='float32')
lgbm_tpr_at_10_fpr_scores = []

lgbm_params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'n_jobs': -1,
    'random_state': RANDOM_STATE,
    'verbose': -1
}

print(f"Starting {n_splits_lgbm}-Fold LightGBM Cross-Validation...")

for fold_lgbm, (train_idx_lgbm, val_idx_lgbm) in enumerate(skf_lgbm.split(X_train_lgbm_full, y_lgbm), start=1):
    X_tr_lgbm = X_train_lgbm_full.iloc[train_idx_lgbm]
    X_val_lgbm = X_train_lgbm_full.iloc[val_idx_lgbm]
    y_tr_lgbm = y_lgbm[train_idx_lgbm]
    y_val_lgbm = y_lgbm[val_idx_lgbm]

    dtrain_lgbm = lgb.Dataset(X_tr_lgbm, label=y_tr_lgbm, categorical_feature='auto')
    dval_lgbm = lgb.Dataset(X_val_lgbm, label=y_val_lgbm, reference=dtrain_lgbm, categorical_feature='auto')

    lgbm_model = lgb.train(
        lgbm_params,
        dtrain_lgbm,
        num_boost_round=1000,
        valid_sets=[dtrain_lgbm, dval_lgbm],
        valid_names=['train', 'valid'],
        callbacks=[
            lgb.early_stopping(stopping_rounds=50),
            lgb.log_evaluation(period=100)
        ]
    )

    lgbm_val_probs = lgbm_model.predict(X_val_lgbm)
    lgbm_oof_preds[val_idx_lgbm] = lgbm_val_probs.astype('float32')
    lgbm_test_preds += lgbm_model.predict(X_test_lgbm_full).astype('float32') / n_splits_lgbm

    # Use FPR <= 0.10, matching the project metric. This corrects the standalone typo.
    lgbm_fold_fpr, lgbm_fold_tpr, lgbm_fold_thresholds = roc_curve(y_val_lgbm, lgbm_val_probs)
    lgbm_valid_idx = np.where(lgbm_fold_fpr <= FPR_LIMIT)[0]
    if len(lgbm_valid_idx) > 0:
        lgbm_best_fold_idx = lgbm_valid_idx[np.argmax(lgbm_fold_tpr[lgbm_valid_idx])]
        lgbm_tpr_at_10 = lgbm_fold_tpr[lgbm_best_fold_idx]
        lgbm_fpr_at_10 = lgbm_fold_fpr[lgbm_best_fold_idx]
    else:
        lgbm_tpr_at_10 = np.nan
        lgbm_fpr_at_10 = np.nan

    lgbm_tpr_at_10_fpr_scores.append(lgbm_tpr_at_10)
    print(
        f"Fold {fold_lgbm} | AUC: {roc_auc_score(y_val_lgbm, lgbm_val_probs):.4f} "
        f"| TPR@FPR<=10%: {lgbm_tpr_at_10:.4f} | FPR: {lgbm_fpr_at_10:.4f}"
    )

# -----------------------------
# OOF threshold and submission
# -----------------------------
lgbm_fpr, lgbm_tpr, lgbm_thresholds = roc_curve(y_lgbm, lgbm_oof_preds)
lgbm_valid_oof_idx = np.where(lgbm_fpr <= FPR_LIMIT)[0]

if len(lgbm_valid_oof_idx) > 0:
    lgbm_best_idx = lgbm_valid_oof_idx[np.argmax(lgbm_tpr[lgbm_valid_oof_idx])]
else:
    # Fallback should rarely happen; keeps the section runnable if predictions are degenerate.
    lgbm_best_idx = int(np.argmin(np.abs(lgbm_fpr - FPR_LIMIT)))

lgbm_best_threshold = lgbm_thresholds[lgbm_best_idx]
lgbm_best_fpr = lgbm_fpr[lgbm_best_idx]
lgbm_best_tpr = lgbm_tpr[lgbm_best_idx]

print("\n--- Model 7 LightGBM Final Results ---")
print(f"Mean fold TPR at FPR <= 10%: {np.nanmean(lgbm_tpr_at_10_fpr_scores):.4f} (+/- {np.nanstd(lgbm_tpr_at_10_fpr_scores):.4f})")
print(f"OOF threshold: {lgbm_best_threshold:.6f}")
print(f"OOF FPR: {lgbm_best_fpr:.6f}")
print(f"OOF TPR: {lgbm_best_tpr:.6f}")
print("OOF prediction shape:", lgbm_oof_preds.shape)
print("Test prediction shape:", lgbm_test_preds.shape)

lgbm_final_test_binary = (lgbm_test_preds >= lgbm_best_threshold).astype(int)
print("LightGBM test prediction distribution:")
print(pd.Series(lgbm_final_test_binary).value_counts())

lgbm_submission = pd.DataFrame(lgbm_final_test_binary)
lgbm_submission_path = os.path.join(OUTPUT_DIR, 'submission_predictions_lightgbm.csv')
lgbm_submission.to_csv(lgbm_submission_path, index=False, header=False)
print("LightGBM submission file created:", lgbm_submission_path)


## 11. Final Submission Output

This section consolidates the final output files. The final submission is produced by Model 1; comparison-model outputs are retained for diagnostics and reporting only.


In [ ]:

if "super_submission_path" in globals():
    print("Final blend submission saved to:", super_submission_path)
else:
    print("Final blend submission has not been created yet. Run Model 1 through the export cell first.")


## 12. Sanity Checks and Summary

The final cells print the key metrics and checks needed before submission: final blended TPR/FPR, threshold, feature count, prediction alignment, and output-file confirmation.


In [ ]:

# Final notebook summary

print("=" * 80)
print("FINAL BLENDED MODEL SUMMARY")
print("=" * 80)
print("Final blended model TPR:", super_metrics.get("tpr", None))
print("Final blended model FPR:", super_metrics.get("fpr", None))
print("Final threshold:", best_threshold)
print("Number of features used in final model:", X.shape[1] if "X" in globals() else "Not available")
print("Final submission path:", super_submission_path if "super_submission_path" in globals() else "Not available")
print("=" * 80)
